# 0211 — ETL Big Bang: Building `pienza.db` From Scratch

This notebook is the genesis of the downstream research pipeline. It takes a collection of disparate raw sheets and isolated data assets, consolidates them, and builds the central SQLite database (`pienza.db`) that powers every subsequent model and the live Streamlit Observatory.

It merges data from three primary pipelines:

- **Engine 1:** Trip events logged in real-time by the custom GTS-4 webapp.
- **Engine 2:** The approx 4,700 OCR offers extracted via OCR.
- **Platform Official Exports:** Lifetime trip history and activity earnings pulled directly from the source platform.

After unifying these datasets, the notebook executes a strict set of data-integrity corrections, materializes the `engineered_features` and `silver_palette` data layers, and generates the consolidated analytical views required by downstream dependencies. Because the entire project's architecture rests on this foundation, this is the largest and highest-stakes notebook in the repository.

<div style="border-left: 4px solid #21918c; padding: 4px 20px; background: rgba(33,145,140,0.05); border-radius: 0 6px 6px 0;">

_Engine 2's OCR data was primarily cleaned upstream. However, this pipeline deliberately ingests both the raw `raw_offers_ocr` and the polished `offers` exports side by side. This dual-ingestion preserves the original, unprocessed OCR state to guarantee strict data lineage and auditability._

</div>

```mermaid
flowchart TD
    A["GCS: raw offers + OCR"] --> B["Phase 1\noffers ETL"]
    B --> C["Phase 2\ntrip_events ETL\n(GTS-4, linking cascade)"]
    C --> D["Phase 3\nlifetime_trips + activity_earnings"]
    D --> E["Phase 4\nManual data corrections\n(patches to specific trips)"]
    E --> F["Phase 5\nengineered_features +\nconsolidated analytical views"]
    F --> G["Phase 6\nsilver_palette + final ML views"]
    G --> H["Closing checkpoint"]

    classDef default stroke:#21918c,stroke-width:2px;
    linkStyle default stroke:#21918c,stroke-width:2px;
```

### Final schema (ERD)

The diagram above shows the process; this shows the result — the full `pienza.db` schema this notebook builds, table by table.

![Pienza ERD](../observatory/assets/Pienza_ERD.png)

*Note: `silver_palette` was built after this ERD was drawn, so it's the one table in the final `pienza.db` that doesn't appear above.*

## Phase 1 — Offers ETL

Sets up GCS/database connections, then stages, cleans, and loads the raw OCR offer export into the `offers` table (with its lookup dimension tables), applies a boolean-type patch, and builds the first reconciled analytical view.

#### 1.1 — Setup and configuration

Connects to the GCS bucket and configures the database path and dependencies.

In [54]:
import os
import re
import sqlite3
import time
import warnings
import io             # <--- [REQUIRED] For GCS blob string buffering (BytesIO)

import numpy as np
import pandas as pd
from tqdm import tqdm
from IPython.display import display  # <--- [REQUIRED] For displaying DataFrames in Codespaces

from google.cloud import storage     # <--- [REQUIRED] For GCS 'storage.Client'
import gspread                       # For GSheets integration
from google.oauth2 import service_account           # For Auth via JSON key
from googleapiclient.discovery import build         # For Google Drive API (Schema retrieval)

from sqlalchemy import create_engine, text, inspect # For database management and views

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('future.no_silent_downcasting', True) # Clean transformation logic

print("Master dependencies loaded. GCS Data Lake environment initialized.")

SERVICE_ACCOUNT_FILE = '/workspaces/pienza/secrets/service-account.json'
GCS_BUCKET_NAME = "pienza_big_bang"
DATE_PREFIX = "260509"  # source-file date prefix used across this pipeline
project_root = '/workspaces/pienza/data/big_bang'
os.makedirs(project_root, exist_ok=True)

db_file_path = os.path.join(project_root, 'pienza.db')

creds = service_account.Credentials.from_service_account_file(SERVICE_ACCOUNT_FILE)
storage_client = storage.Client(credentials=creds, project=creds.project_id)
bucket = storage_client.bucket(GCS_BUCKET_NAME)

print("Libraries imported and GCS connection established.")
print(f"The database will be built from scratch at: {db_file_path}")

Master dependencies loaded. GCS Data Lake environment initialized.
Libraries imported and GCS connection established.
The database will be built from scratch at: /workspaces/pienza/data/big_bang/pienza.db


/tmp/ipykernel_16744/1535768430.py:21: Pandas4Warning: 'future.no_silent_downcasting' is deprecated, please refrain from using it.
  pd.set_option('future.no_silent_downcasting', True) # Clean transformation logic


#### 1.2 — Build and populate lookup tables

Creates the SQLite schema and the dimension/lookup tables (product categories, actions, reasons, etc.).

In [55]:
print(f"--- PHASE 1 STARTED: Building the '{db_file_path}' ---")

if os.path.exists(db_file_path):
    os.remove(db_file_path)
    print(f"Removed existing database file.")

try:
    sql_blob_name = f"{DATE_PREFIX}_schema_v251122.sql"
    blob = bucket.blob(sql_blob_name)
    schema_script = blob.download_as_text()

    conn = sqlite3.connect(db_file_path)
    conn.executescript(schema_script)
    conn.commit()
    print(f"Schema created successfully from GCS: {sql_blob_name}")
except Exception as e:
    print(f"ERROR during schema creation from GCS: {e}")
    assert False, "Failed to build schema from Data Lake."
finally:
    if 'conn' in locals(): conn.close()

print(f"\n--- Populating all lookup tables... ---")

# Same data to guarantee parity
product_category_data = [(1, 'uberx'), (2, 'comfort'), (3, 'business_comfort'), (4, 'black'), (5, 'uber_planet'), (6, 'uber_pet'), (7, 'envíos_uber')]
offer_action_data = [(1, 'accepted'), (2, 'reject')]
reason_primary_data = [(1, 'dropoff_non_operational'), (2, 'dropoff_proxy'), (3, 'low_profitability'), (4, 'long_pickup_time'), (5, 'dropoff_strategic_mismatch'), (6, 'expected_value_gamble'), (7, 'system_logic_failure')]
heuristic_flag_data = [(1, 'deadhead_risk'), (2, 'long_ride_traffic_risk'), (3, 'obj_end_session'), (4, 'market_anomaly'), (5, 'friday_traffic_risk'), (6, 'dropoff_uncertain'), (7, 'trap'), (8, 'protest_anomaly')]
post_offer_status_data = [(1, 'continue_game'), (2, 'disconnect')]
driver_state_at_request_data = [(1, 'looking_for_rides'), (2, 'trip_in_progress')]
outcome_data = [(1, 'completed'), (2, 'rider_canceled'), (3, 'driver_canceled'), (4, 'unfulfilled'), (5, 'system_failure')]
interpolation_quality_data = [(1, 'unanchored'), (2, 'interpolated'), (3, 'extrapolated_end'), (4, 'extrapolated_start'), (5, 'interpolated_stationary')]
record_status_data = [(1, 'valid'), (2, 'invalid_non_offer')]

try:
    conn = sqlite3.connect(db_file_path)
    cur = conn.cursor()
    cur.executemany("INSERT INTO product_category (product_category_id, category_name) VALUES (?, ?)", product_category_data)
    cur.executemany("INSERT INTO offer_action (offer_action_id, offer_action_description) VALUES (?, ?)", offer_action_data)
    cur.executemany("INSERT INTO reason_primary (reason_primary_id, reason_primary_description) VALUES (?, ?)", reason_primary_data)
    cur.executemany("INSERT INTO heuristic_flag (heuristic_flag_id, heuristic_flag_description) VALUES (?, ?)", heuristic_flag_data)
    cur.executemany("INSERT INTO post_offer_status (post_offer_status_id, post_offer_status_description) VALUES (?, ?)", post_offer_status_data)
    cur.executemany("INSERT INTO driver_state_at_request (driver_state_at_request_id, driver_state_at_request_description) VALUES (?, ?)", driver_state_at_request_data)
    cur.executemany("INSERT INTO outcome (outcome_id, outcome_description) VALUES (?, ?)", outcome_data)
    cur.executemany("INSERT INTO interpolation_quality (interpolation_quality_id, interpolation_quality_description) VALUES (?, ?)", interpolation_quality_data)
    cur.executemany("INSERT INTO record_status (record_status_id, record_status_description) VALUES (?, ?)", record_status_data)
    conn.commit()
    print("Success: All lookup tables have been populated.")
except sqlite3.Error as e:
    print(f"\nSQL error: {e}")
    conn.rollback()
finally:
    if 'conn' in locals(): conn.close()

print("\n--- PHASE 1 COMPLETE ---")

--- PHASE 1 STARTED: Building the '/workspaces/pienza/data/big_bang/pienza.db' ---
Removed existing database file.
Schema created successfully from GCS: 260509_schema_v251122.sql

--- Populating all lookup tables... ---
Success: All lookup tables have been populated.

--- PHASE 1 COMPLETE ---


#### 1.3 — Stage raw OCR data

Downloads and stages the raw OCR offer export from GCS.

In [56]:
print(f"--- PHASE 2 STARTED: Staging Raw OCR data from GCS Data Lake ---")

GCS_FILE_NAME = f"{DATE_PREFIX}_raw_offers_raw_requests_messy.csv"
staged_ocr_file = os.path.join(project_root, "staged_raw_ocr.parquet")

try:
    blob = bucket.blob(GCS_FILE_NAME)
    content = blob.download_as_bytes()
    
    # na_filter=False is what guarantees the 4765 records.
    # Prevents Pandas from interpreting the text "NULL" as a null value (NaN),
    # allowing those 7 records to survive, matching what happens in GSheets.
    df_raw = pd.read_csv(io.BytesIO(content), header=None, dtype=str, na_filter=False)
    print(f"Successfully downloaded '{GCS_FILE_NAME}' from bucket.")

    data_rows = df_raw.iloc[1:].copy()
    df = data_rows.iloc[:, 0:11].copy()

    df.columns = [
        'ocr_id', 'image_filename', 'time_taken', 'ride_type', 'upfront_fare',
        'pickup_details', 'pickup_address', 'trip_details', 'dropoff_address',
        'rider_rating', 'special_note'
    ]
    print("Data extracted and sliced to columns A-K using canonical headers.")

    # Since there are no NaNs (thanks to na_filter=False), dropna will not remove the "NULL" strings.
    df.dropna(subset=['ocr_id', 'image_filename'], how='any', inplace=True)
    df = df[(df['ocr_id'] != '') & (df['image_filename'] != '')].copy()
    
    print(f"Extracted and cleaned {len(df)} valid rows from GCS.")

    df_staged = df
    print("Skipping transformation. Staging raw text data to preserve source integrity.")

    df_staged.to_parquet(staged_ocr_file, index=False)
    print(f"SUCCESS! Staged {len(df_staged)} raw records to: {staged_ocr_file}")

except Exception as e:
    print(f"An error occurred during OCR data staging: {e}")

print("\n--- PHASE 2 COMPLETE (B) ---")

--- PHASE 2 STARTED: Staging Raw OCR data from GCS Data Lake ---
Successfully downloaded '260509_raw_offers_raw_requests_messy.csv' from bucket.
Data extracted and sliced to columns A-K using canonical headers.
Extracted and cleaned 4765 valid rows from GCS.
Skipping transformation. Staging raw text data to preserve source integrity.
SUCCESS! Staged 4765 raw records to: /workspaces/pienza/data/big_bang/staged_raw_ocr.parquet

--- PHASE 2 COMPLETE (B) ---


#### 1.4 — Load staged OCR data

Loads the staged OCR records into the database.

In [57]:
print("--- PHASE 3 STARTED: Loading staged OCR data into 'raw_offers_ocr' table ---")

# The path of the file generated in cell B4
source_parquet_file = os.path.join(project_root, "staged_raw_ocr.parquet")
table_name = 'raw_offers_ocr'

try:
    df_staged = pd.read_parquet(source_parquet_file)
    print(f"Extracted {len(df_staged)} raw records from staged Parquet file.")

    print("Transforming string 'NULL's into true NULL values...")
    # Same null-handling rigor as in notebook A
    df_to_load = df_staged.replace('NULL', np.nan)
    print("Nullification complete.")

    with sqlite3.connect(db_file_path) as conn:
        df_to_load.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Success: Loaded {len(df_to_load)} cleaned records into '{table_name}'.")

    with sqlite3.connect(db_file_path) as conn:
        count_df = pd.read_sql_query(f"SELECT COUNT(*) as count FROM {table_name}", conn)
        total_rows = count_df['count'][0]

    if total_rows == len(df_to_load):
        print(f"Validation successful: DB table contains {total_rows} records.")
    else:
        print(f"VALIDATION FAILED: Row count mismatch.")

except Exception as e:
    print(f"An error occurred during the loading process: {e}")

print("\n--- PHASE 3 COMPLETE (B) ---")

--- PHASE 3 STARTED: Loading staged OCR data into 'raw_offers_ocr' table ---
Extracted 4765 raw records from staged Parquet file.
Transforming string 'NULL's into true NULL values...
Nullification complete.
Success: Loaded 4765 cleaned records into 'raw_offers_ocr'.
Validation successful: DB table contains 4765 records.

--- PHASE 3 COMPLETE (B) ---


#### 1.5 — ETL for the offers table

Cleans, transforms, and loads the polished `offers` table and its many-to-many join tables.

In [58]:
print("--- PHASE 4 STARTED: Polishing and Loading data for the 'offers' table (B) ---")

print("\n--- [E] EXTRACTING all necessary data sources from GCS Bucket ---")
try:
    GCS_FILE_NAME = f"{DATE_PREFIX}_raw_offers_diamond_offers.csv"
    blob = bucket.blob(GCS_FILE_NAME)
    content = blob.download_as_bytes()
    
    # na_filter=False for parity with GSheets
    df_polished_raw = pd.read_csv(io.BytesIO(content), na_filter=False, dtype=str)
    print(f"Extracted {len(df_polished_raw)} polished rows from GCS CSV.")

    with sqlite3.connect(db_file_path) as conn:
        df_ocr_link = pd.read_sql_query("SELECT ocr_id FROM raw_offers_ocr", conn)
    print(f"Extracted {len(df_ocr_link)} rows from 'raw_offers_ocr' for linking.")

    with sqlite3.connect(db_file_path) as conn:
        lookup_table_names = [
            "product_category", "offer_action", "reason_primary", "heuristic_flag",
            "post_offer_status", "driver_state_at_request", "outcome",
            "interpolation_quality", "record_status"
        ]
        lookup_dfs = {}
        for table in tqdm(lookup_table_names, desc="Extracting lookups"):
            lookup_dfs[table] = pd.read_sql_query(f"SELECT * FROM {table}", conn)
    print(f"Extracted all {len(lookup_dfs)} lookup tables.")

except Exception as e:
    print(f"ERROR during extraction: {e}")
    assert False, "Extraction failed, halting execution."

# (THE TRANSFORMATION LOGIC IS IDENTICAL TO VERSION A)
print("\n--- [T] TRANSFORMING and enriching 'offer' data ---")

if 'ocr_fk' in df_polished_raw.columns:
    df_polished_raw.drop(columns=['ocr_fk'], inplace=True)
df_polished_raw['join_key'] = pd.to_numeric(df_polished_raw['offer_id'].str.replace('OF', ''), errors='coerce')
df_ocr_link['join_key'] = pd.to_numeric(df_ocr_link['ocr_id'].str.replace('OCR_', ''), errors='coerce')
df_ocr_link.rename(columns={'ocr_id': 'ocr_fk'}, inplace=True)
df_transformed = pd.merge(df_polished_raw, df_ocr_link[['join_key', 'ocr_fk']], on='join_key', how='left')
df_transformed.drop(columns=['join_key'], inplace=True)
print("ocr_fk link established.")

df_transformed.replace({'': None, 'N/A': None, 'nan': None, 'NULL': None}, inplace=True)

numeric_cols = [
    'upfront_fare', 'time_to_pickup_sec', 'dist_to_pickup_km', 'est_trip_time_sec',
    'est_trip_dist_km', 'pickup_lat', 'pickup_lon', 'dropoff_lat', 'dropoff_lon',
    'surge_amount', 'turbo_plus_amount', 'reservation_amount', 'priority_amount',
    'rider_star_rating', 'rider_trip_count', 'time_in_session_sec', 'session_progress_ratio',
    'inferred_agent_lat', 'inferred_agent_lon', 'inferred_agent_bearing', 'inferred_agent_speed_mps'
]
for col in numeric_cols:
    if col in df_transformed.columns:
        df_transformed[col] = pd.to_numeric(df_transformed[col], errors='coerce')
        df_transformed[col] = df_transformed[col].astype('float64')

df_transformed['offer_timestamp'] = pd.to_datetime(
    df_transformed['offer_timestamp'], format='%Y:%m:%d %H:%M:%S', errors='coerce'
).dt.strftime('%Y-%m-%d %H:%M:%S')

def three_state_bool_converter(value):
    if isinstance(value, str):
        val_lower = value.lower().strip()
        if val_lower == 'true': return True
        elif val_lower == 'false': return False
        else: return None
    elif isinstance(value, bool): return value
    else: return None

boolean_cols = [col for col in df_transformed.columns if col.startswith('is_')]
for col in boolean_cols:
    if col in df_transformed.columns:
        df_transformed[col] = df_transformed[col].apply(three_state_bool_converter)
        df_transformed[col] = df_transformed[col].astype('boolean')

merge_map_1_to_1 = {
    'product_category': ('product_category_fk', 'category_name'),
    'offer_action': ('offer_action_fk', 'offer_action_description'),
    'reason_primary': ('reason_primary_fk', 'reason_primary_description'),
    'post_offer_status': ('post_offer_status_fk', 'post_offer_status_description'),
    'driver_state_at_request': ('driver_state_at_request_fk', 'driver_state_at_request_description'),
    'outcome': ('outcome_fk', 'outcome_description'),
    'interpolation_quality': ('interpolation_quality_fk', 'interpolation_quality_description'),
    'record_status': ('record_status_fk', 'record_status_description')
}
for lookup_table, (source_fk_col, desc_col) in merge_map_1_to_1.items():
    if source_fk_col in df_transformed.columns:
        df_transformed[source_fk_col] = df_transformed[source_fk_col].astype(str).str.strip().str.lower().str.replace(' ', '_')
        if source_fk_col == 'product_category_fk':
            df_transformed[source_fk_col] = df_transformed[source_fk_col].str.replace('artículo', 'envíos_uber', regex=False)
        df_lookup = lookup_dfs[lookup_table]
        id_col = f"{lookup_table}_id"
        df_transformed = pd.merge(df_transformed, df_lookup[[id_col, desc_col]], left_on=source_fk_col, right_on=desc_col, how='left')
        df_transformed.drop(columns=[source_fk_col, desc_col], inplace=True)
        df_transformed.rename(columns={id_col: source_fk_col}, inplace=True)

df_heuristic_lookup = lookup_dfs['heuristic_flag']
df_flags_raw = df_transformed[['offer_id', 'heuristic_flag']].dropna(subset=['heuristic_flag'])
df_flags_raw['flag_list'] = df_flags_raw['heuristic_flag'].str.split(', ')
df_exploded = df_flags_raw.explode('flag_list')
df_exploded['flag_list'] = df_exploded['flag_list'].str.strip()
df_join_table_prep = pd.merge(df_exploded, df_heuristic_lookup, left_on='flag_list', right_on='heuristic_flag_description', how='inner')
df_heuristic_flag_offers_to_load = df_join_table_prep[['offer_id', 'heuristic_flag_id']].copy()
df_heuristic_flag_offers_to_load.rename(columns={'offer_id': 'offers_offer_id', 'heuristic_flag_id': 'heuristic_flag_heuristic_flag_id'}, inplace=True)

final_offer_columns = [
    'offer_id', 'session_fk', 'ocr_fk', 'image_content_hash', 'offer_timestamp', 'upfront_fare',
    'time_to_pickup_sec', 'dist_to_pickup_km', 'est_trip_time_sec', 'est_trip_dist_km',
    'pickup_address', 'dropoff_address', 'pickup_lat', 'pickup_lon', 'dropoff_lat', 'dropoff_lon',
    'is_surge', 'surge_amount', 'is_turbo_plus', 'turbo_plus_amount', 'is_reservation',
    'reservation_amount', 'is_priority', 'priority_amount', 'is_exclusive', 'is_vip',
    'is_identity_verified', 'is_long_trip', 'is_multiple_destinations', 'is_teens',
    'rider_star_rating', 'rider_trip_count', 'time_in_session_sec', 'session_progress_ratio',
    'inferred_agent_lat', 'inferred_agent_lon', 'inferred_agent_bearing',
    'inferred_agent_speed_mps', 'is_imputed', 'special_note_raw', 'comment_1', 'comment_2',
    'product_category_fk', 'offer_action_fk', 'reason_primary_fk', 'post_offer_status_fk',
    'driver_state_at_request_fk', 'outcome_fk', 'interpolation_quality_fk', 'record_status_fk'
]
existing_final_columns = [col for col in final_offer_columns if col in df_transformed.columns]
df_offers_to_load = df_transformed[existing_final_columns]

with sqlite3.connect(db_file_path) as conn:
    df_offers_to_load.to_sql("offers", conn, if_exists='replace', index=False)
    df_heuristic_flag_offers_to_load.to_sql("heuristic_flag_offers", conn, if_exists='replace', index=False)

print(f"\nPHASE 4 COMPLETE (B): Main 'offers' and M:M join tables loaded into the database.")

--- PHASE 4 STARTED: Polishing and Loading data for the 'offers' table (B) ---

--- [E] EXTRACTING all necessary data sources from GCS Bucket ---
Extracted 4765 polished rows from GCS CSV.
Extracted 4765 rows from 'raw_offers_ocr' for linking.


Extracting lookups: 100%|██████████| 9/9 [00:00<00:00, 1680.19it/s]

Extracted all 9 lookup tables.

--- [T] TRANSFORMING and enriching 'offer' data ---
ocr_fk link established.



PHASE 4 COMPLETE (B): Main 'offers' and M:M join tables loaded into the database.


#### 1.6 — Boolean-type patch

Normalizes three-state boolean columns to a consistent type.

In [59]:
print("--- Starting the 'Three-State Boolean' Post-Processing Patch (B) ---")

try:
    print("Extracting source data to identify 'Unknown' booleans from GCS...")
    GCS_FILE_NAME = f"{DATE_PREFIX}_raw_offers_diamond_offers.csv"
    blob = bucket.blob(GCS_FILE_NAME)
    content = blob.download_as_bytes()
    
    # IMPORTANT: na_filter=False so empty values read as "" instead of NaN
    # This ensures the filter mask behaves the same as in notebook A
    df_source = pd.read_csv(io.BytesIO(content), na_filter=False, dtype=str)
    df_source.columns = df_source.columns.str.lower().str.replace(' ', '_')
    
    boolean_cols = [col for col in df_source.columns if col.startswith('is_')]

    # Identify which IDs need to be nulled out
    ids_to_null = {col: [] for col in boolean_cols}
    for col in boolean_cols:
        # Same mask logic as in notebook A
        mask = ~df_source[col].isin(['TRUE', 'FALSE', 'True', 'False', True, False])
        ids_to_null[col] = df_source[mask]['offer_id'].tolist()

    print("Successfully identified offers with 'Unknown' boolean states from GCS.")

    print(f"\nConnecting to {os.path.basename(db_file_path)} for surgical UPDATEs...")
    total_updates = 0
    with sqlite3.connect(db_file_path) as conn:
        cursor = conn.cursor()
        for col, ids in ids_to_null.items():
            if ids:
                placeholders = ', '.join(['?'] * len(ids))
                update_sql = f"UPDATE offers SET {col} = NULL WHERE offer_id IN ({placeholders});"
                cursor.execute(update_sql, ids)
                total_updates += cursor.rowcount
                print(f"  - Set {cursor.rowcount} records to NULL for column '{col}'.")

    print(f"\nPatch complete. A total of {total_updates} fields were updated to NULL.")
    print("The 'offers' table now correctly represents the three-state boolean logic.")

except Exception as e:
    print(f"ERROR during post-processing patch: {e}")

print("\n--- PATCH COMPLETE (B) ---")

--- Starting the 'Three-State Boolean' Post-Processing Patch (B) ---
Extracting source data to identify 'Unknown' booleans from GCS...
Successfully identified offers with 'Unknown' boolean states from GCS.

Connecting to pienza.db for surgical UPDATEs...

Patch complete. A total of 0 fields were updated to NULL.
The 'offers' table now correctly represents the three-state boolean logic.

--- PATCH COMPLETE (B) ---


#### 1.7 — Master analytical view

Creates `v_reconciled_offer`, joining `offers` to its lookup tables.

In [60]:
print(f"--- Creating the master analytical view in B DB: {os.path.basename(db_file_path)} ---")

# This SQL is an exact mirror of notebook A's
view_sql = """
CREATE VIEW v_reconciled_offer AS
SELECT
    o.*,
    pc.category_name,
    oa.offer_action_description AS offer_action,
    rp.reason_primary_description AS reason_primary,
    pos.post_offer_status_description AS post_offer_status,
    ds.driver_state_at_request_description AS driver_state_at_request,
    ot.outcome_description AS outcome,
    iq.interpolation_quality_description AS interpolation_quality,
    rs.record_status_description AS record_status
FROM
    offers o
LEFT JOIN product_category pc        ON o.product_category_fk = pc.product_category_id
LEFT JOIN offer_action oa             ON o.offer_action_fk = oa.offer_action_id
LEFT JOIN reason_primary rp           ON o.reason_primary_fk = rp.reason_primary_id
LEFT JOIN post_offer_status pos       ON o.post_offer_status_fk = pos.post_offer_status_id
LEFT JOIN driver_state_at_request ds  ON o.driver_state_at_request_fk = ds.driver_state_at_request_id
LEFT JOIN outcome ot                  ON o.outcome_fk = ot.outcome_id
LEFT JOIN interpolation_quality iq    ON o.interpolation_quality_fk = iq.interpolation_quality_id
LEFT JOIN record_status rs            ON o.record_status_fk = rs.record_status_id;
"""

try:
    with sqlite3.connect(db_file_path) as conn:
        conn.execute("DROP VIEW IF EXISTS v_reconciled_offer;")
        conn.execute(view_sql)
    print("Success: 'v_reconciled_offer' VIEW created successfully in GCS Data Lake.")

except Exception as e:
    print(f"ERROR creating view in B DB: {e}")

--- Creating the master analytical view in B DB: pienza.db ---
Success: 'v_reconciled_offer' VIEW created successfully in GCS Data Lake.


#### 1.8 — Average fare by day of week

Sanity-check query on the reconciled view.

In [61]:
print("--- Calculating average fare (upfront_fare) by day of week (Data Lake B) ---")

query_sql = """
SELECT
    CASE strftime('%w', offer_timestamp)
        WHEN '0' THEN '0 - Sunday'
        WHEN '1' THEN '1 - Monday'
        WHEN '2' THEN '2 - Tuesday'
        WHEN '3' THEN '3 - Wednesday'
        WHEN '4' THEN '4 - Thursday'
        WHEN '5' THEN '5 - Friday'
        WHEN '6' THEN '6 - Saturday'
    END AS day_of_week,
    AVG(upfront_fare) AS average_fare
FROM
    v_reconciled_offer
WHERE
    upfront_fare IS NOT NULL
GROUP BY
    day_of_week
ORDER BY
    strftime('%w', offer_timestamp);
"""

try:
    with sqlite3.connect(db_file_path) as conn:
        df_avg_fare = pd.read_sql_query(query_sql, conn)

    print("SUCCESS: Query executed successfully in Data Lake B.")
    print("Displaying average fare by day of week:")
    
    display(df_avg_fare.style.format({'average_fare': '{:,.2f}'}))

except Exception as e:
    print(f"ERROR: Query failed in B. Reason: {e}")

--- Calculating average fare (upfront_fare) by day of week (Data Lake B) ---
SUCCESS: Query executed successfully in Data Lake B.
Displaying average fare by day of week:


,day_of_week,average_fare
0,0 - Sunday,106.12
1,1 - Monday,121.21
2,2 - Tuesday,124.36
3,3 - Wednesday,133.91
4,4 - Thursday,138.77
5,5 - Friday,140.64
6,6 - Saturday,124.83


#### 1.9 — Basic hypothesis query

Compares average fare for accepted vs. rejected offers.

In [62]:
# Path configuration for environment B
if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/content/drive/My Drive/_Pienza'
    db_file_path = os.path.join(project_root, 'Assets/Database/pienza.db')
    print("INFO: 'db_file_path' no encontrado. Usando ruta por defecto.")

print("\n--- Calculating average fare for Accepted vs. Rejected offers (Data Lake B) ---")

query_sql = """
SELECT
    offer_action,
    AVG(upfront_fare) AS average_fare,
    COUNT(*) AS total_offers
FROM
    v_reconciled_offer
WHERE
    offer_action IN ('accepted', 'reject')
    AND upfront_fare IS NOT NULL
GROUP BY
    offer_action
ORDER BY
    average_fare DESC;
"""

try:
    with sqlite3.connect(db_file_path) as conn:
        df_hypothesis = pd.read_sql_query(query_sql, conn)

    print("SUCCESS: The central hypothesis query executed successfully in Data Lake B.")
    print("Displaying results:")

    display(df_hypothesis.style.format({
        'average_fare': 'MXN {:,.2f}',
        'total_offers': '{:,}'
    }).hide(axis="index"))

except Exception as e:
    print(f"ERROR: Query failed in B. Reason: {e}")


--- Calculating average fare for Accepted vs. Rejected offers (Data Lake B) ---
SUCCESS: The central hypothesis query executed successfully in Data Lake B.
Displaying results:


offer_action,average_fare,total_offers
accepted,MXN 147.59,346
reject,MXN 129.28,"4,417"


## Phase 2 — trip_events ETL (GTS-4)

Stages and cleans the GTS-4 trip event export, normalizes event types, then runs the linking cascade that reconciles each event back to its originating offer (exact match, fuzzy fare match, then orphan-offer match). Includes several iterations of the linking/reconciliation logic (`B28`-`B32`) kept for traceability, plus the final cascade and its post-ETL patches.

#### 2.1 — Stage and clean GTS-4 data

Downloads and cleans the raw GTS-4 trip event export.

In [63]:
print("--- PHASE 5 STARTED: Staging and Cleaning GTS-4 Trip Events (Data Lake B) ---")

pd.set_option('future.no_silent_downcasting', True)

try:
    GCS_FILE_NAME = f"{DATE_PREFIX}_gts-4_trip_events.csv"
    blob = bucket.blob(GCS_FILE_NAME)
    content = blob.download_as_bytes()
    
    # Use na_filter=False to prevent "NULL" strings from being converted to NaN prematurely
    df_raw_gts = pd.read_csv(io.BytesIO(content), dtype=str, na_filter=False)
    print(f"Downloaded {GCS_FILE_NAME} from GCS. Found {len(df_raw_gts)} raw rows.")

    if 'event_id_legacy' in df_raw_gts.columns:
        df_raw_gts.rename(columns={'event_id_legacy': 'trip_id_legacy'}, inplace=True)

    df_raw_gts.reset_index(inplace=True)
    df_raw_gts.rename(columns={'index': 'event_id'}, inplace=True)
    df_raw_gts['event_id'] = df_raw_gts['event_id'] + 1 

    df_staged = df_raw_gts.copy()

    df_staged['event_timestamp'] = pd.to_datetime(df_staged['event_timestamp'], errors='coerce')
    df_staged.dropna(subset=['event_timestamp'], inplace=True)

    numeric_cols = ['event_lat', 'event_lon', 'upfront_fare', 'realized_fare']
    for col in numeric_cols:
        if col in df_staged.columns:
            df_staged[col] = pd.to_numeric(df_staged[col], errors='coerce')

    boolean_map = {'TRUE': True, 'FALSE': False, 'True': True, 'False': False, True: True, False: False}
    if 'is_imputed' in df_staged.columns:
        df_staged['is_imputed'] = df_staged['is_imputed'].map(boolean_map).fillna(False)

    staged_file = os.path.join(project_root, "staged_gts4.parquet")
    df_staged.to_parquet(staged_file, index=False)

    print(f"SUCCESS: Staged {len(df_staged)} cleaned records to: {staged_file}")
    print("\n--- Final Health Check (Data Lake B) ---")
    df_staged.info()

except Exception as e:
    print(f"ERROR during GTS-4 staging in Data Lake: {e}")

print("\n--- PHASE 5 COMPLETE (B) ---")

--- PHASE 5 STARTED: Staging and Cleaning GTS-4 Trip Events (Data Lake B) ---


/tmp/ipykernel_16744/3265052567.py:3: Pandas4Warning: 'future.no_silent_downcasting' is deprecated, please refrain from using it.
  pd.set_option('future.no_silent_downcasting', True)


Downloaded 260509_gts-4_trip_events.csv from GCS. Found 1031 raw rows.
SUCCESS: Staged 1031 cleaned records to: /workspaces/pienza/data/big_bang/staged_gts4.parquet

--- Final Health Check (Data Lake B) ---
<class 'pandas.DataFrame'>
RangeIndex: 1031 entries, 0 to 1030
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   event_id           1031 non-null   int64         
 1   trip_id_legacy     1031 non-null   str           
 2   event_timestamp    1031 non-null   datetime64[us]
 3   event_types_id_fk  1031 non-null   str           
 4   event_lat          966 non-null    float64       
 5   event_lon          966 non-null    float64       
 6   event_address      1031 non-null   str           
 7   upfront_fare       259 non-null    float64       
 8   realized_fare      249 non-null    float64       
 9   is_imputed         1031 non-null   object        
 10  comment            1031 non-null

#### 2.2 — Transform and normalize event_types

Builds the `event_types` lookup table.

In [64]:
print("\n--- PHASE 2: TRANSFORMING & NORMALIZING event_types (Data Lake B) ---")

unique_event_descriptions = df_staged['event_types_id_fk'].unique()

df_event_types = pd.DataFrame({'description': unique_event_descriptions})

# 3. Engineer the 'event_code' from the description
df_event_types['event_code'] = df_event_types['description'].str.split(':').str[0]

event_order = ['T0', 'T1', 'T2', 'T3', 'T4']
df_event_types['event_code'] = pd.Categorical(df_event_types['event_code'], categories=event_order, ordered=True)
df_event_types = df_event_types.sort_values('event_code')

df_event_types.reset_index(drop=True, inplace=True)
df_event_types.reset_index(inplace=True)
df_event_types.rename(columns={'index': 'event_type_id'}, inplace=True)
df_event_types['event_type_id'] = df_event_types['event_type_id'] + 1 

final_columns = ['event_type_id', 'event_code', 'description']
df_event_types = df_event_types[final_columns]

try:
    with sqlite3.connect(db_file_path) as conn:
        df_event_types.to_sql('event_types', conn, if_exists='replace', index=False)
    print("SUCCESS: 'event_types' table has been created and populated in Data Lake DB.")
    print(f"Total event types loaded: {len(df_event_types)}")
except Exception as e:
    print(f"ERROR: Failed to load 'event_types' table in B. Reason: {e}")

# Display the final lookup table for verification
print("\n--- Final event_types Lookup Table (Data Lake B) ---")
print(df_event_types)


--- PHASE 2: TRANSFORMING & NORMALIZING event_types (Data Lake B) ---
SUCCESS: 'event_types' table has been created and populated in Data Lake DB.
Total event types loaded: 5

--- Final event_types Lookup Table (Data Lake B) ---
   event_type_id event_code                           description
0              1         T0                 T0: Looking for rides
1              2         T1  T1: Ride Accepted, Driving to Pickup
2              3         T2             T2: Waiting for passenger
3              4         T3                      T3: Ride Started
4              5         T4                    T4: Ride completed


#### 2.3 — Enrich and load trip_events (first pass)

Initial ETL pass for `trip_events`, later superseded by the final cascade below.

In [65]:
print("\n--- PHASE 3: ENRICHING & LOADING trip_events (Data Lake B) ---")

df_trip_events_prep = pd.merge(
    df_staged,
    df_event_types[['description', 'event_type_id']], 
    left_on='event_types_id_fk', 
    right_on='description',
    how='left'
)

# 2. Architectural Correction: Drop redundant text and rename numeric ID
df_trip_events_prep.drop(columns=['event_types_id_fk', 'description'], inplace=True)
df_trip_events_prep.rename(columns={'event_type_id': 'event_types_id_fk'}, inplace=True)

final_columns = [
    'event_id',
    'event_timestamp',
    'event_lat',
    'event_lon',
    'event_address',
    'upfront_fare',
    'realized_fare',
    'source',
    'is_imputed',
    'comment',
    'trip_id_legacy',
    'event_types_id_fk'
]

existing_final_columns = [col for col in final_columns if col in df_trip_events_prep.columns]
df_trip_events = df_trip_events_prep[existing_final_columns]

# 4. Load the final, enriched DataFrame into the Data Lake SQLite database
try:
    with sqlite3.connect(db_file_path) as conn:
        df_trip_events.to_sql('trip_events', conn, if_exists='replace', index=False)
    print("SUCCESS: 'trip_events' table has been created and populated in Data Lake B.")
    print(f"Total events loaded: {len(df_trip_events)}")
except Exception as e:
    print(f"ERROR: Failed to load 'trip_events' table in B. Reason: {e}")

# Mask lat/lon, event_address, and fare fields for display only;
# df_trip_events itself (already written to the DB above) is left untouched.
def _obf_coord(v):
    if v is None or pd.isna(v):
        return v
    digits_seen = 0
    out = []
    for ch in str(v):
        if ch.isdigit():
            digits_seen += 1
            out.append(ch if digits_seen <= 4 else '#')
        else:
            out.append(ch)
    return ''.join(out)

def _obf_address(v):
    if v is None or str(v) in ('None', 'nan', ''):
        return v
    _mask_digits = lambda m: re.sub(r'\d', '#', m.group(0))
    return re.sub(r'\b(?!\d{5}\b)\d+[\w-]*\b', _mask_digits, str(v), count=1)

def _obf_fare(v):
    if v is None or pd.isna(v):
        return "-"
    int_part, dec_part = f"{float(v):.2f}".split('.')
    masked_int = int_part[:2] + '#' * max(0, len(int_part) - 2)
    return f"{masked_int}.{'#' * len(dec_part)}"

df_trip_events_display = df_trip_events.head().copy()
for col in ('event_lat', 'event_lon'):
    if col in df_trip_events_display.columns:
        df_trip_events_display[col] = df_trip_events_display[col].apply(_obf_coord)
if 'event_address' in df_trip_events_display.columns:
    df_trip_events_display['event_address'] = df_trip_events_display['event_address'].apply(_obf_address)
for col in ('upfront_fare', 'realized_fare'):
    if col in df_trip_events_display.columns:
        df_trip_events_display[col] = df_trip_events_display[col].apply(_obf_fare)

print("\n--- Verifying the first 5 rows of the loaded trip_events data (location/fare fields masked for display) ---")
display(df_trip_events_display)

print("\n--- Verifying the schema of the loaded trip_events data ---")
df_trip_events.info()


--- PHASE 3: ENRICHING & LOADING trip_events (Data Lake B) ---
SUCCESS: 'trip_events' table has been created and populated in Data Lake B.
Total events loaded: 1031

--- Verifying the first 5 rows of the loaded trip_events data (location/fare fields masked for display) ---


,event_id,event_timestamp,event_lat,event_lon,event_address,upfront_fare,realized_fare,source,is_imputed,comment,trip_id_legacy,event_types_id_fk
0,1,2025-08-22 06:48:00,NaN,NaN,N/A,13#.##,-,GTS-1,False,N/A,250822-01,2
1,2,2025-08-22 07:00:00,19.46#####,-99.16#####,"Colonia Ampliación Del Gas, Mexico City, Azcap...",-,-,GTS-1,False,N/A,250822-01,4
2,3,2025-08-22 07:22:05,19.43#####,-99.18#####,"Avenida Homero, Polanco #ª Sección, Mexico Cit...",-,11#.##,GTS-1,False,N/A,250822-01,5
3,4,2025-08-22 07:28:00,NaN,NaN,N/A,16#.##,-,GTS-1,False,N/A,250822-02,2
4,5,2025-08-22 07:38:00,19.45#####,-99.20#####,"##, Callejón San Joaquín, Argentina Antigua, M...",-,-,GTS-1,False,N/A,250822-02,4



--- Verifying the schema of the loaded trip_events data ---
<class 'pandas.DataFrame'>
RangeIndex: 1031 entries, 0 to 1030
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   event_id           1031 non-null   int64         
 1   event_timestamp    1031 non-null   datetime64[us]
 2   event_lat          966 non-null    float64       
 3   event_lon          966 non-null    float64       
 4   event_address      1031 non-null   str           
 5   upfront_fare       259 non-null    float64       
 6   realized_fare      249 non-null    float64       
 7   source             1031 non-null   str           
 8   is_imputed         1031 non-null   object        
 9   comment            1031 non-null   str           
 10  trip_id_legacy     1031 non-null   str           
 11  event_types_id_fk  1031 non-null   int64         
dtypes: datetime64[us](1), float64(4), int64(2), object(1), str(4)
memory

#### 2.4 — Trip financials query (v3.0)

Aggregated upfront vs. realized fare per trip, kept for traceability.

In [66]:
print("--- INITIALIZING ANALYTICAL QUERY: TRIP FINANCIALS ---")

if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/workspaces/pienza'
    db_file_path = os.path.join(project_root, 'data/big_bang/pienza.db')

if not os.path.exists(db_file_path):
    print("CRITICAL FAILURE: The database file was not found at the specified path.")
else:
    print(f"Connecting to database at: {db_file_path}")

    sql_query_v3 = """
    WITH trip_financials AS (
        SELECT
            trip_id_legacy,
            MAX(CASE WHEN et.event_code = 'T1' THEN te.upfront_fare ELSE NULL END) AS trip_upfront_fare,
            MAX(CASE WHEN et.event_code = 'T4' THEN te.realized_fare ELSE NULL END) AS trip_real_fare
        FROM
            trip_events te
        JOIN
            event_types et ON te.event_types_id_fk = et.event_type_id
        WHERE
            te.trip_id_legacy IS NOT NULL
        GROUP BY
            te.trip_id_legacy
    )
    SELECT 
        trip_id_legacy,
        trip_upfront_fare,
        trip_real_fare
    FROM 
        trip_financials;
    """

    try:
        with sqlite3.connect(db_file_path) as conn:
            df_results_v3 = pd.read_sql_query(sql_query_v3, conn)

        print("\nSUCCESS: CTE financial query executed in Data Lake B.")
        print("Displaying aggregated trip financials (fare values masked for display):")

        def _obf_fare(v):
            if v is None or pd.isna(v):
                return "-"
            int_part, dec_part = f"{float(v):.2f}".split('.')
            masked_int = int_part[:2] + '#' * max(0, len(int_part) - 2)
            return f"{masked_int}.{'#' * len(dec_part)}"

        df_display_v3 = df_results_v3.head(20).copy()
        for col in ['trip_upfront_fare', 'trip_real_fare']:
            df_display_v3[col] = df_display_v3[col].apply(_obf_fare)
        display(df_display_v3)

    except Exception as e:
        print(f"\nERROR: Query failed in B. Reason: {e}")

--- INITIALIZING ANALYTICAL QUERY: TRIP FINANCIALS ---
Connecting to database at: /workspaces/pienza/data/big_bang/pienza.db

SUCCESS: CTE financial query executed in Data Lake B.
Displaying aggregated trip financials (fare values masked for display):


,trip_id_legacy,trip_upfront_fare,trip_real_fare
0,250822-01,13#.##,11#.##
1,250822-02,16#.##,13#.##
2,250822-03,64.##,57.##
3,250822-04,14#.##,13#.##
4,250822-05,60.##,49.##
5,250822-06,10#.##,86.##
6,250822-07,17#.##,12#.##
7,250822-08,17#.##,13#.##
8,250822-09,89.##,76.##
9,250822-10,20#.##,15#.##


#### 2.5 — Trip summary query (v4.1)

Full trip timeline plus fares, kept for traceability.

In [67]:
project_root = '/workspaces/pienza'
db_file_path = os.path.join(project_root, 'data/big_bang/pienza.db')

query_the_money = """
WITH trip_summary_raw AS (
    SELECT
        trip_id_legacy,
        MAX(CASE WHEN et.event_code = 'T0' THEN te.event_timestamp ELSE NULL END) AS t0_timestamp,
        MAX(CASE WHEN et.event_code = 'T1' THEN te.event_timestamp ELSE NULL END) AS t1_timestamp,
        MAX(CASE WHEN et.event_code = 'T2' THEN te.event_timestamp ELSE NULL END) AS t2_timestamp,
        MAX(CASE WHEN et.event_code = 'T3' THEN te.event_timestamp ELSE NULL END) AS t3_timestamp,
        MAX(CASE WHEN et.event_code = 'T4' THEN te.event_timestamp ELSE NULL END) AS t4_timestamp,
        MAX(CASE WHEN et.event_code = 'T1' THEN te.upfront_fare ELSE NULL END) AS t1_upfront_fare,
        MAX(CASE WHEN et.event_code = 'T4' THEN te.realized_fare ELSE NULL END) AS t4_realized_fare
    FROM
        trip_events te
    JOIN
        event_types et ON te.event_types_id_fk = et.event_type_id
    GROUP BY
        te.trip_id_legacy
)
SELECT * FROM trip_summary_raw;
"""

print(f"--- INITIALIZING: Targeting Data Lake database at '{db_file_path}' ---")

if not os.path.exists(db_file_path):
    print("CRITICAL FAILURE: The database file does NOT exist.")
else:
    print("Pre-flight check PASSED. Database file found.")
    try:
        with sqlite3.connect(db_file_path) as conn:
            df_money_table = pd.read_sql_query(query_the_money, conn)

        print("SUCCESS: Query executed in Data Lake B.")

        timestamp_cols = ['t0_timestamp', 't1_timestamp', 't2_timestamp', 't3_timestamp', 't4_timestamp']
        for col in timestamp_cols:
            df_money_table[col] = pd.to_datetime(df_money_table[col], errors='coerce')

        def _obf_fare(v):
            if v is None or pd.isna(v):
                return "-"
            int_part, dec_part = f"{float(v):.2f}".split('.')
            masked_int = int_part[:2] + '#' * max(0, len(int_part) - 2)
            return f"{masked_int}.{'#' * len(dec_part)}"

        print("Displaying trip summary (fare values masked for display):")
        df_display_money = df_money_table.copy()
        for col in ['t1_upfront_fare', 't4_realized_fare']:
            df_display_money[col] = df_display_money[col].apply(_obf_fare)
        display(df_display_money)

    except Exception as e:
        print(f"ERROR: Query failed in Data Lake B. Reason: {e}")

--- INITIALIZING: Targeting Data Lake database at '/workspaces/pienza/data/big_bang/pienza.db' ---
Pre-flight check PASSED. Database file found.
SUCCESS: Query executed in Data Lake B.
Displaying trip summary (fare values masked for display):


,trip_id_legacy,t0_timestamp,t1_timestamp,t2_timestamp,t3_timestamp,t4_timestamp,t1_upfront_fare,t4_realized_fare
0,250822-01,NaT,2025-08-22 06:48:00,NaT,2025-08-22 07:00:00,2025-08-22 07:22:05,13#.##,11#.##
1,250822-02,NaT,2025-08-22 07:28:00,NaT,2025-08-22 07:38:00,2025-08-22 08:10:43,16#.##,13#.##
2,250822-03,NaT,2025-08-22 08:12:00,NaT,2025-08-22 08:17:00,2025-08-22 08:28:17,64.##,57.##
3,250822-04,NaT,2025-08-22 08:38:00,NaT,2025-08-22 08:49:00,2025-08-22 09:17:10,14#.##,13#.##
4,250822-05,NaT,2025-08-22 09:16:00,NaT,2025-08-22 09:20:00,2025-08-22 09:29:01,60.##,49.##
...,...,...,...,...,...,...,...,...
254,251001-03,NaT,2025-10-01 07:00:04,2025-10-01 07:10:17,2025-10-01 07:11:27,2025-10-01 07:43:01,18#.##,15#.##
255,251001-04,NaT,2025-10-01 07:43:53,2025-10-01 07:45:39,2025-10-01 07:47:21,2025-10-01 07:53:08,13#.##,12#.##
256,251001-05,2025-10-01 07:53:16,2025-10-01 07:57:49,2025-10-01 08:04:21,2025-10-01 08:04:43,2025-10-01 08:34:35,12#.##,10#.##
257,251001-06,2025-10-01 08:34:44,2025-10-01 08:36:35,NaT,2025-10-01 08:44:11,2025-10-01 09:13:29,16#.##,14#.##


#### 2.6 — Linking task draft (self-contained)

Early self-contained draft of the offer-to-event linking logic.

In [68]:
if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/workspaces/pienza'
    db_file_path = os.path.join(project_root, 'data/big_bang/pienza.db')

print("--- Architecting the 'v_trip_funnel_metrics' analytical view (Data Lake B) ---")

view_sql = """
CREATE VIEW v_trip_funnel_metrics AS
WITH PivotedEvents AS (
    SELECT
        trip_id_legacy,
        MAX(CASE WHEN event_types_id_fk = 1 THEN event_timestamp END) AS t0_looking,
        MAX(CASE WHEN event_types_id_fk = 2 THEN event_timestamp END) AS t1_accepted,
        MAX(CASE WHEN event_types_id_fk = 3 THEN event_timestamp END) AS t2_arrived,
        MAX(CASE WHEN event_types_id_fk = 4 THEN event_timestamp END) AS t3_started,
        MAX(CASE WHEN event_types_id_fk = 5 THEN event_timestamp END) AS t4_completed,
        MAX(CASE WHEN event_types_id_fk = 2 THEN upfront_fare END) AS upfront_fare,
        MAX(CASE WHEN event_types_id_fk = 5 THEN realized_fare END) AS realized_fare
    FROM
        trip_events
    GROUP BY
        trip_id_legacy
)
SELECT
    p.trip_id_legacy,
    p.upfront_fare,
    p.realized_fare,
    (julianday(p.t2_arrived) - julianday(p.t1_accepted)) * 86400.0 AS duration_to_pickup_sec,
    (julianday(p.t3_started) - julianday(p.t2_arrived)) * 86400.0 AS duration_waiting_sec,
    (julianday(p.t4_completed) - julianday(p.t3_started)) * 86400.0 AS duration_trip_sec,
    (julianday(p.t4_completed) - julianday(p.t1_accepted)) * 86400.0 AS duration_total_engagement_sec,
    p.t0_looking,
    p.t1_accepted,
    p.t2_arrived,
    p.t3_started,
    p.t4_completed
FROM
    PivotedEvents p;
"""

try:
    with sqlite3.connect(db_file_path) as conn:
        conn.execute("DROP VIEW IF EXISTS v_trip_funnel_metrics;")
        conn.execute(view_sql)
    print("SUCCESS: View 'v_trip_funnel_metrics' created successfully in Data Lake B.")

    print("\n--- Displaying LAST 20 rows of the new view for verification ---")
    with sqlite3.connect(db_file_path) as conn:
        verification_query = """
        SELECT *
        FROM v_trip_funnel_metrics
        ORDER BY t1_accepted DESC
        LIMIT 20;
        """
        df_verify = pd.read_sql_query(verification_query, conn)

        def _obf_fare(v):
            if v is None or pd.isna(v):
                return v
            int_part, dec_part = f"{float(v):.2f}".split('.')
            masked_int = int_part[:2] + '#' * max(0, len(int_part) - 2)
            return f"{masked_int}.{'#' * len(dec_part)}"

        df_verify_display = df_verify.copy()
        for col in ('upfront_fare', 'realized_fare'):
            if col in df_verify_display.columns:
                df_verify_display[col] = df_verify_display[col].apply(_obf_fare)
        display(df_verify_display)

except Exception as e:
    print(f"ERROR creating view in B: {e}")

--- Architecting the 'v_trip_funnel_metrics' analytical view (Data Lake B) ---
SUCCESS: View 'v_trip_funnel_metrics' created successfully in Data Lake B.

--- Displaying LAST 20 rows of the new view for verification ---


,trip_id_legacy,upfront_fare,realized_fare,duration_to_pickup_sec,duration_waiting_sec,duration_trip_sec,duration_total_engagement_sec,t0_looking,t1_accepted,t2_arrived,t3_started,t4_completed
0,251001-07,13#.##,12#.##,259.999998,48.000021,1992.000018,2300.000037,NaN,2025-10-01 09:13:54,2025-10-01 09:18:14,2025-10-01 09:19:02,2025-10-01 09:52:14
1,251001-06,16#.##,14#.##,NaN,NaN,1758.000000,2213.999981,2025-10-01 08:34:44,2025-10-01 08:36:35,NaN,2025-10-01 08:44:11,2025-10-01 09:13:29
2,251001-05,12#.##,10#.##,392.000006,22.000001,1791.999976,2205.999984,2025-10-01 07:53:16,2025-10-01 07:57:49,2025-10-01 08:04:21,2025-10-01 08:04:43,2025-10-01 08:34:35
3,251001-04,13#.##,12#.##,106.000029,101.999970,347.000009,555.000007,NaN,2025-10-01 07:43:53,2025-10-01 07:45:39,2025-10-01 07:47:21,2025-10-01 07:53:08
4,251001-03,18#.##,15#.##,613.000014,69.999982,1893.999986,2576.999983,NaN,2025-10-01 07:00:04,2025-10-01 07:10:17,2025-10-01 07:11:27,2025-10-01 07:43:01
5,251001-02,11#.##,97.##,395.999984,70.000023,1053.999996,1520.000003,2025-10-01 06:30:56,2025-10-01 06:33:52,2025-10-01 06:40:28,2025-10-01 06:41:38,2025-10-01 06:59:12
6,251001-01,15#.##,14#.##,390.000017,168.999968,1119.000006,1677.999991,2025-10-01 05:59:11,2025-10-01 06:02:43,2025-10-01 06:09:13,2025-10-01 06:12:02,2025-10-01 06:30:41
7,250930-10,19#.##,16#.##,NaN,NaN,1545.000029,1802.000003,2025-09-30 15:40:18,2025-09-30 15:44:44,NaN,2025-09-30 15:49:01,2025-09-30 16:14:46
8,250930-09,10#.##,84.##,NaN,NaN,1512.000006,1660.000008,NaN,2025-09-30 15:12:14,NaN,2025-09-30 15:14:42,2025-09-30 15:39:54
9,250930-08,10#.##,10#.##,501.000018,85.999976,818.000029,1405.000024,2025-09-30 14:41:41,2025-09-30 14:47:33,2025-09-30 14:55:54,2025-09-30 14:57:20,2025-09-30 15:10:58


#### 2.7 — Linking task, step 1/2 (v1.2)

Iteration of the linking logic, kept for traceability.

In [69]:
if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/workspaces/pienza'
    db_file_path = os.path.join(project_root, 'data/big_bang/pienza.db')

print("--- Architecting the foundational 'v_trip_funnel_wide' view (Data Lake B) ---")

view_sql = """
CREATE VIEW v_trip_funnel_wide AS
SELECT
    trip_id_legacy,
    MAX(CASE WHEN event_types_id_fk = 1 THEN event_timestamp END) AS t0_timestamp,
    MAX(CASE WHEN event_types_id_fk = 1 THEN event_address END)   AS t0_address,
    MAX(CASE WHEN event_types_id_fk = 2 THEN event_timestamp END) AS t1_timestamp,
    MAX(CASE WHEN event_types_id_fk = 2 THEN event_address END)   AS t1_address,
    MAX(CASE WHEN event_types_id_fk = 3 THEN event_timestamp END) AS t2_timestamp,
    MAX(CASE WHEN event_types_id_fk = 3 THEN event_address END)   AS t2_address,
    MAX(CASE WHEN event_types_id_fk = 4 THEN event_timestamp END) AS t3_timestamp,
    MAX(CASE WHEN event_types_id_fk = 4 THEN event_address END)   AS t3_address,
    MAX(CASE WHEN event_types_id_fk = 5 THEN event_timestamp END) AS t4_timestamp,
    MAX(CASE WHEN event_types_id_fk = 5 THEN event_address END)   AS t4_address,
    MAX(CASE WHEN event_types_id_fk = 2 THEN upfront_fare END)    AS upfront_fare,
    MAX(CASE WHEN event_types_id_fk = 5 THEN realized_fare END)   AS realized_fare
FROM
    trip_events
GROUP BY
    trip_id_legacy;
"""

try:
    with sqlite3.connect(db_file_path) as conn:
        conn.execute("DROP VIEW IF EXISTS v_trip_funnel_wide;")
        conn.execute(view_sql)
    print("SUCCESS: Foundational view 'v_trip_funnel_wide' created successfully in Data Lake B.")

    print("\n--- Displaying LAST 10 rows for verification (Most Recent Trips) ---")
    with sqlite3.connect(db_file_path) as conn:
        verification_query = """
        SELECT *
        FROM v_trip_funnel_wide
        ORDER BY t1_timestamp DESC
        LIMIT 10;
        """
        df_verify = pd.read_sql_query(verification_query, conn)

        def _obf_address(v):
            if v is None or str(v) in ('None', 'nan', ''):
                return v
            _mask_digits = lambda m: re.sub(r'\d', '#', m.group(0))
            return re.sub(r'\b(?!\d{5}\b)\d+[\w-]*\b', _mask_digits, str(v), count=1)

        def _obf_coord(v):
            if v is None or pd.isna(v):
                return v
            digits_seen = 0
            out = []
            for ch in str(v):
                if ch.isdigit():
                    digits_seen += 1
                    out.append(ch if digits_seen <= 4 else '#')
                else:
                    out.append(ch)
            return ''.join(out)

        def _obf_fare(v):
            if v is None or pd.isna(v):
                return v
            int_part, dec_part = f"{float(v):.2f}".split('.')
            masked_int = int_part[:2] + '#' * max(0, len(int_part) - 2)
            return f"{masked_int}.{'#' * len(dec_part)}"

        df_verify_display = df_verify.copy()
        for col in df_verify_display.columns:
            if 'address' in col:
                df_verify_display[col] = df_verify_display[col].apply(_obf_address)
            elif col.endswith('_lat') or col.endswith('_lon'):
                df_verify_display[col] = df_verify_display[col].apply(_obf_coord)
            elif 'fare' in col:
                df_verify_display[col] = df_verify_display[col].apply(_obf_fare)
        display(df_verify_display)

except Exception as e:
    print(f"ERROR creating view in B: {e}")

--- Architecting the foundational 'v_trip_funnel_wide' view (Data Lake B) ---
SUCCESS: Foundational view 'v_trip_funnel_wide' created successfully in Data Lake B.

--- Displaying LAST 10 rows for verification (Most Recent Trips) ---


,trip_id_legacy,t0_timestamp,t0_address,t1_timestamp,t1_address,t2_timestamp,t2_address,t3_timestamp,t3_address,t4_timestamp,t4_address,upfront_fare,realized_fare
0,251001-07,NaN,NaN,2025-10-01 09:13:54,"Marriott Mexico City Reforma Hotel, ###, Aveni...",2025-10-01 09:18:14,"Calle Nápoles, Zona Rosa, Mexico City, Cuauhté...",2025-10-01 09:19:02,"Calle Hamburgo, Zona Rosa, Mexico City, Cuauht...",2025-10-01 09:52:14,"Avenida Paseo de las Palmas, Lomas de Chapulte...",13#.##,12#.##
1,251001-06,2025-10-01 08:34:44,"###, Calle Montes Urales Norte, Lomas de Chapu...",2025-10-01 08:36:35,"###, Calle Montes Urales Norte, Lomas de Chapu...",NaN,NaN,2025-10-01 08:44:11,Tecamachalco 11650,2025-10-01 09:13:29,"Marriott Mexico City Reforma Hotel, ###, Aveni...",16#.##,14#.##
2,251001-05,2025-10-01 07:53:16,"Colonia La Puntada, Lomas de Vista Hermosa, Me...",2025-10-01 07:57:49,"Calle Bosque de los Tabachines, Bosques de las...",2025-10-01 08:04:21,"Privada Calle Ballonetas, Colonia Lomas del Ch...",2025-10-01 08:04:43,"Privada Calle Ballonetas, Colonia Lomas del Ch...",2025-10-01 08:34:35,"###, Calle Montes Urales Norte, Lomas de Chapu...",12#.##,10#.##
3,251001-04,NaN,NaN,2025-10-01 07:43:53,"##, Paseo de los Laureles, Bosques de las Loma...",2025-10-01 07:45:39,"Calle Bosque de los Tabachines, Bosques de las...",2025-10-01 07:47:21,"Calle Bosque de los Tabachines, Bosques de las...",2025-10-01 07:53:08,"Colonia La Puntada, Lomas de Vista Hermosa, Me...",13#.##,12#.##
4,251001-03,NaN,NaN,2025-10-01 07:00:04,"Avenida Ejército Nacional Mexicano, Granada, M...",2025-10-01 07:10:17,"Torre Río ###, 436, Avenida Río San Joaquín, N...",2025-10-01 07:11:27,"Torre Río ###, 436, Avenida Río San Joaquín, N...",2025-10-01 07:43:01,"Sambalca Enterprise, Calle Paseo de los Tamari...",18#.##,15#.##
5,251001-02,2025-10-01 06:30:56,"##, Calle Alicama, Lomas de Chapultepec 4ª Sec...",2025-10-01 06:33:52,"Avenida Lomas, Colonia Lomas de Virreyes, Loma...",2025-10-01 06:40:28,"Calle Poniente ##, Colonia Cove, Mexico City, ...",2025-10-01 06:41:38,"Calle Poniente ##, Colonia Cove, Mexico City, ...",2025-10-01 06:59:12,"Avenida Ejército Nacional Mexicano, Granada, M...",11#.##,97.##
6,251001-01,2025-10-01 05:59:11,"Calle Comte, Colonia Nueva Anzures, Anzures, M...",2025-10-01 06:02:43,"Avenida Río Mississippi, Cuauhtémoc, Mexico Ci...",2025-10-01 06:09:13,"Calle Eligio Ancona, Santa María la Ribera, Me...",2025-10-01 06:12:02,"###, Calle Nogal, Santa María la Ribera, Mexic...",2025-10-01 06:30:41,"Calle Pedregal, Lomas de Chapultepec #ª Secció...",15#.##,14#.##
7,250930-10,2025-09-30 15:40:18,"Corporativo Lomas Cantabria, ##, Cerrada de la...",2025-09-30 15:44:44,"Extra Periférico, ###, Boulevard Manuel Ávila ...",NaN,NaN,2025-09-30 15:49:01,FC Cuernavaca,2025-09-30 16:14:46,"###, Avenida Paseo de la Reforma, Little Seoul...",19#.##,16#.##
8,250930-09,NaN,NaN,2025-09-30 15:12:14,"Avenida Vasco de Quiroga, Santa Fe Cuajimalpa,...",NaN,NaN,2025-09-30 15:14:42,Geocoding failed,2025-09-30 15:39:54,"Corporativo Lomas Cantabria, ##, Cerrada de la...",10#.##,84.##
9,250930-08,2025-09-30 14:41:41,"Calle Bosque de Pirules, Colonia Bosques de Re...",2025-09-30 14:47:33,"Avenida Paseo de los Ahuehuetes Sur, Colonia B...",2025-09-30 14:55:54,"Roche, #, Calle Molino Bezares, Lomas de Bezar...",2025-09-30 14:57:20,"Avenida Constituyentes - Puente CONAFRUT, Boul...",2025-09-30 15:10:58,"Calle José Villagrán, Santa Fe Cuajimalpa, Mex...",10#.##,10#.##


#### 2.8 — Linking task, step 1/2 (v1.3)

Further iteration of the linking logic, kept for traceability.

In [70]:
if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/workspaces/pienza'
    db_file_path = os.path.join(project_root, 'data/big_bang/pienza.db')

print("--- Architecting the foundational 'v_trip_funnel_wide' view (Data Lake B - v1.3 Geo-Enriched) ---")

view_sql = """
CREATE VIEW v_trip_funnel_wide AS
SELECT
    trip_id_legacy,
    -- T0 Event Group (Looking)
    MAX(CASE WHEN event_types_id_fk = 1 THEN event_timestamp END) AS t0_timestamp,
    MAX(CASE WHEN event_types_id_fk = 1 THEN event_lat END)       AS t0_lat,
    MAX(CASE WHEN event_types_id_fk = 1 THEN event_lon END)       AS t0_lon,
    -- T1 Event Group (Accepted)
    MAX(CASE WHEN event_types_id_fk = 2 THEN event_timestamp END) AS t1_timestamp,
    MAX(CASE WHEN event_types_id_fk = 2 THEN event_lat END)       AS t1_lat,
    MAX(CASE WHEN event_types_id_fk = 2 THEN event_lon END)       AS t1_lon,
    -- T2 Event Group (Arrived)
    MAX(CASE WHEN event_types_id_fk = 3 THEN event_timestamp END) AS t2_timestamp,
    MAX(CASE WHEN event_types_id_fk = 3 THEN event_lat END)       AS t2_lat,
    MAX(CASE WHEN event_types_id_fk = 3 THEN event_lon END)       AS t2_lon,
    -- T3 Event Group (Started)
    MAX(CASE WHEN event_types_id_fk = 4 THEN event_timestamp END) AS t3_timestamp,
    MAX(CASE WHEN event_types_id_fk = 4 THEN event_lat END)       AS t3_lat,
    MAX(CASE WHEN event_types_id_fk = 4 THEN event_lon END)       AS t3_lon,
    -- T4 Event Group (Completed)
    MAX(CASE WHEN event_types_id_fk = 5 THEN event_timestamp END) AS t4_timestamp,
    MAX(CASE WHEN event_types_id_fk = 5 THEN event_lat END)       AS t4_lat,
    MAX(CASE WHEN event_types_id_fk = 5 THEN event_lon END)       AS t4_lon,
    -- Financials
    MAX(CASE WHEN event_types_id_fk = 2 THEN upfront_fare END)    AS upfront_fare,
    MAX(CASE WHEN event_types_id_fk = 5 THEN realized_fare END)   AS realized_fare
FROM
    trip_events
GROUP BY
    trip_id_legacy;
"""

try:
    with sqlite3.connect(db_file_path) as conn:
        conn.execute("DROP VIEW IF EXISTS v_trip_funnel_wide;")
        conn.execute(view_sql)
    print("SUCCESS: Foundational view 'v_trip_funnel_wide' (v1.3) created successfully in Data Lake B.")

    print("\n--- Displaying LAST 10 rows for verification (Geo-Enriched Sample) ---")
    with sqlite3.connect(db_file_path) as conn:
        verification_query = """
        SELECT *
        FROM v_trip_funnel_wide
        ORDER BY t1_timestamp DESC
        LIMIT 10;
        """
        df_verify = pd.read_sql_query(verification_query, conn)

        def _obf_address(v):
            if v is None or str(v) in ('None', 'nan', ''):
                return v
            _mask_digits = lambda m: re.sub(r'\d', '#', m.group(0))
            return re.sub(r'\b(?!\d{5}\b)\d+[\w-]*\b', _mask_digits, str(v), count=1)

        def _obf_coord(v):
            if v is None or pd.isna(v):
                return v
            digits_seen = 0
            out = []
            for ch in str(v):
                if ch.isdigit():
                    digits_seen += 1
                    out.append(ch if digits_seen <= 4 else '#')
                else:
                    out.append(ch)
            return ''.join(out)

        def _obf_fare(v):
            if v is None or pd.isna(v):
                return v
            int_part, dec_part = f"{float(v):.2f}".split('.')
            masked_int = int_part[:2] + '#' * max(0, len(int_part) - 2)
            return f"{masked_int}.{'#' * len(dec_part)}"

        df_verify_display = df_verify.copy()
        for col in df_verify_display.columns:
            if 'address' in col:
                df_verify_display[col] = df_verify_display[col].apply(_obf_address)
            elif col.endswith('_lat') or col.endswith('_lon'):
                df_verify_display[col] = df_verify_display[col].apply(_obf_coord)
            elif 'fare' in col:
                df_verify_display[col] = df_verify_display[col].apply(_obf_fare)
        display(df_verify_display)

except Exception as e:
    print(f"ERROR creating view in B: {e}")

--- Architecting the foundational 'v_trip_funnel_wide' view (Data Lake B - v1.3 Geo-Enriched) ---
SUCCESS: Foundational view 'v_trip_funnel_wide' (v1.3) created successfully in Data Lake B.

--- Displaying LAST 10 rows for verification (Geo-Enriched Sample) ---


,trip_id_legacy,t0_timestamp,t0_lat,t0_lon,t1_timestamp,t1_lat,t1_lon,t2_timestamp,t2_lat,t2_lon,t3_timestamp,t3_lat,t3_lon,t4_timestamp,t4_lat,t4_lon,upfront_fare,realized_fare
0,251001-07,NaN,NaN,NaN,2025-10-01 09:13:54,19.42######,-99.16######,2025-10-01 09:18:14,19.42######,-99.16######,2025-10-01 09:19:02,19.42######,-99.16#####,2025-10-01 09:52:14,19.42######,-99.21######,13#.##,12#.##
1,251001-06,2025-10-01 08:34:44,19.42######,-99.20######,2025-10-01 08:36:35,19.42######,-99.21######,NaN,NaN,NaN,2025-10-01 08:44:11,NaN,NaN,2025-10-01 09:13:29,19.42#####,-99.16######,16#.##,14#.##
2,251001-05,2025-10-01 07:53:16,19.38######,-99.26######,2025-10-01 07:57:49,19.39######,-99.25######,2025-10-01 08:04:21,19.38#####,-99.25######,2025-10-01 08:04:43,19.38######,-99.25#####,2025-10-01 08:34:35,19.42######,-99.20######,12#.##,10#.##
3,251001-04,NaN,NaN,NaN,2025-10-01 07:43:53,19.39######,-99.24######,2025-10-01 07:45:39,19.39######,-99.25######,2025-10-01 07:47:21,19.39######,-99.25######,2025-10-01 07:53:08,19.38######,-99.26######,13#.##,12#.##
4,251001-03,NaN,NaN,NaN,2025-10-01 07:00:04,19.43######,-99.20######,2025-10-01 07:10:17,19.44######,-99.20######,2025-10-01 07:11:27,19.44######,-99.20######,2025-10-01 07:43:01,19.38######,-99.25######,18#.##,15#.##
5,251001-02,2025-10-01 06:30:56,19.42#####,-99.20######,2025-10-01 06:33:52,19.41#####,-99.20######,2025-10-01 06:40:28,19.40#####,-99.19######,2025-10-01 06:41:38,19.40#####,-99.19######,2025-10-01 06:59:12,19.43######,-99.20######,11#.##,97.##
6,251001-01,2025-10-01 05:59:11,19.42######,-99.17######,2025-10-01 06:02:43,19.42######,-99.17######,2025-10-01 06:09:13,19.45######,-99.16######,2025-10-01 06:12:02,19.45######,-99.16######,2025-10-01 06:30:41,19.42######,-99.20######,15#.##,14#.##
7,250930-10,2025-09-30 15:40:18,19.43######,-99.21######,2025-09-30 15:44:44,19.43######,-99.21######,NaN,NaN,NaN,2025-09-30 15:49:01,NaN,NaN,2025-09-30 16:14:46,19.42######,-99.16######,19#.##,16#.##
8,250930-09,NaN,NaN,NaN,2025-09-30 15:12:14,19.36######,-99.26######,NaN,NaN,NaN,2025-09-30 15:14:42,19.36######,-99.26######,2025-09-30 15:39:54,19.43#####,-99.21######,10#.##,84.##
9,250930-08,2025-09-30 14:41:41,19.39######,-99.26######,2025-09-30 14:47:33,19.39######,-99.25######,2025-09-30 14:55:54,19.38######,-99.24#####,2025-09-30 14:57:20,19.38######,-99.24######,2025-09-30 15:10:58,19.36######,-99.27######,10#.##,10#.##


#### 2.9 — Linking task, step 2/2

Completes the linking logic from the previous two cells.

In [71]:
if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/workspaces/pienza'
    db_file_path = os.path.join(project_root, 'data/big_bang/pienza.db')

print("--- Architecting the final 'v_trip_final_kpis' analytical view (Data Lake B) ---")

view_sql = """
CREATE VIEW v_trip_final_kpis AS
WITH ImputedFlag AS (
    SELECT
        trip_id_legacy,
        MAX(is_imputed) AS is_imputed
    FROM
        trip_events
    GROUP BY
        trip_id_legacy
)
SELECT
    v.trip_id_legacy,
    DATE(v.t1_timestamp) AS trip_date,
    v.upfront_fare,
    v.realized_fare,
    i.is_imputed,

    CASE
        WHEN v.upfront_fare > 0 THEN v.realized_fare / v.upfront_fare
        ELSE NULL
    END AS spread_percentage,

    (julianday(v.t1_timestamp) - julianday(v.t0_timestamp)) * 86400.0 AS duration_lfr_sec,
    (julianday(v.t2_timestamp) - julianday(v.t1_timestamp)) * 86400.0 AS duration_dtp_sec,
    (julianday(v.t3_timestamp) - julianday(v.t2_timestamp)) * 86400.0 AS duration_wfp_sec,
    (julianday(v.t4_timestamp) - julianday(v.t3_timestamp)) * 86400.0 AS duration_on_ride_sec,

    CASE
        WHEN (julianday(v.t4_timestamp) - julianday(v.t3_timestamp)) > 0
        THEN v.upfront_fare / ((julianday(v.t4_timestamp) - julianday(v.t3_timestamp)) * 24.0)
        ELSE NULL
    END AS eph_upfront,

    CASE
        WHEN (julianday(v.t4_timestamp) - julianday(v.t3_timestamp)) > 0
        THEN v.realized_fare / ((julianday(v.t4_timestamp) - julianday(v.t3_timestamp)) * 24.0)
        ELSE NULL
    END AS eph_realized,

    v.t0_timestamp, v.t1_timestamp, v.t2_timestamp, v.t3_timestamp, v.t4_timestamp

FROM
    v_trip_funnel_wide v
LEFT JOIN
    ImputedFlag i ON v.trip_id_legacy = i.trip_id_legacy;
"""

try:
    with sqlite3.connect(db_file_path) as conn:
        conn.execute("DROP VIEW IF EXISTS v_trip_final_kpis;")
        conn.execute(view_sql)
    print("SUCCESS: Final KPI view 'v_trip_final_kpis' created successfully in Data Lake B.")

    print("\n--- Displaying LAST 10 rows for verification (KPI Sample) ---")
    with sqlite3.connect(db_file_path) as conn:
        verification_query = "SELECT * FROM v_trip_final_kpis ORDER BY t1_timestamp DESC LIMIT 10;"
        df_verify = pd.read_sql_query(verification_query, conn)

        def _obf_address(v):
            if v is None or str(v) in ('None', 'nan', ''):
                return v
            _mask_digits = lambda m: re.sub(r'\d', '#', m.group(0))
            return re.sub(r'\b(?!\d{5}\b)\d+[\w-]*\b', _mask_digits, str(v), count=1)

        def _obf_coord(v):
            if v is None or pd.isna(v):
                return v
            digits_seen = 0
            out = []
            for ch in str(v):
                if ch.isdigit():
                    digits_seen += 1
                    out.append(ch if digits_seen <= 4 else '#')
                else:
                    out.append(ch)
            return ''.join(out)

        def _obf_fare(v):
            if v is None or pd.isna(v):
                return v
            int_part, dec_part = f"{float(v):.2f}".split('.')
            masked_int = int_part[:2] + '#' * max(0, len(int_part) - 2)
            return f"{masked_int}.{'#' * len(dec_part)}"

        df_verify_display = df_verify.copy()
        for col in df_verify_display.columns:
            if 'address' in col:
                df_verify_display[col] = df_verify_display[col].apply(_obf_address)
            elif col.endswith('_lat') or col.endswith('_lon'):
                df_verify_display[col] = df_verify_display[col].apply(_obf_coord)
            elif 'fare' in col:
                df_verify_display[col] = df_verify_display[col].apply(_obf_fare)
        display(df_verify_display)

except Exception as e:
    print(f"ERROR creating view in B: {e}")

--- Architecting the final 'v_trip_final_kpis' analytical view (Data Lake B) ---
SUCCESS: Final KPI view 'v_trip_final_kpis' created successfully in Data Lake B.

--- Displaying LAST 10 rows for verification (KPI Sample) ---


,trip_id_legacy,trip_date,upfront_fare,realized_fare,is_imputed,spread_percentage,duration_lfr_sec,duration_dtp_sec,duration_wfp_sec,duration_on_ride_sec,eph_upfront,eph_realized,t0_timestamp,t1_timestamp,t2_timestamp,t3_timestamp,t4_timestamp
0,251001-07,2025-10-01,13#.##,12#.##,0,0.925907,NaN,259.999998,48.000021,1992.000018,244.644576,226.518070,NaN,2025-10-01 09:13:54,2025-10-01 09:18:14,2025-10-01 09:19:02,2025-10-01 09:52:14
1,251001-06,2025-10-01,16#.##,14#.##,1,0.894133,111.000001,NaN,NaN,1758.000000,339.276451,303.358362,2025-10-01 08:34:44,2025-10-01 08:36:35,NaN,2025-10-01 08:44:11,2025-10-01 09:13:29
2,251001-05,2025-10-01,12#.##,10#.##,0,0.887037,273.000008,392.000006,22.000001,1791.999976,242.216521,214.854914,2025-10-01 07:53:16,2025-10-01 07:57:49,2025-10-01 08:04:21,2025-10-01 08:04:43,2025-10-01 08:34:35
3,251001-04,2025-10-01,13#.##,12#.##,0,0.899153,NaN,106.000029,101.999970,347.000009,1384.703135,1245.060487,NaN,2025-10-01 07:43:53,2025-10-01 07:45:39,2025-10-01 07:47:21,2025-10-01 07:53:08
4,251001-03,2025-10-01,18#.##,15#.##,0,0.856727,NaN,613.000014,69.999982,1893.999986,346.257658,296.648365,NaN,2025-10-01 07:00:04,2025-10-01 07:10:17,2025-10-01 07:11:27,2025-10-01 07:43:01
5,251001-02,2025-10-01,11#.##,97.##,1,0.849246,176.000011,395.999984,70.000023,1053.999996,391.730552,332.675523,2025-10-01 06:30:56,2025-10-01 06:33:52,2025-10-01 06:40:28,2025-10-01 06:41:38,2025-10-01 06:59:12
6,251001-01,2025-10-01,15#.##,14#.##,0,0.941639,212.000017,390.000017,168.999968,1119.000006,486.756030,458.348523,2025-10-01 05:59:11,2025-10-01 06:02:43,2025-10-01 06:09:13,2025-10-01 06:12:02,2025-10-01 06:30:41
7,250930-10,2025-09-30,19#.##,16#.##,1,0.838819,266.000006,NaN,NaN,1545.000029,453.064069,380.038828,2025-09-30 15:40:18,2025-09-30 15:44:44,NaN,2025-09-30 15:49:01,2025-09-30 16:14:46
8,250930-09,2025-09-30,10#.##,84.##,0,0.825964,NaN,NaN,NaN,1512.000006,244.476189,201.928571,NaN,2025-09-30 15:12:14,NaN,2025-09-30 15:14:42,2025-09-30 15:39:54
9,250930-08,2025-09-30,10#.##,10#.##,0,0.942797,351.999982,501.000018,85.999976,818.000029,479.310496,451.892405,2025-09-30 14:41:41,2025-09-30 14:47:33,2025-09-30 14:55:54,2025-09-30 14:57:20,2025-09-30 15:10:58


#### 2.10 — Definitive money-table view

Creates the corrected view used for fare verification.

In [72]:
if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/workspaces/pienza'
    db_file_path = os.path.join(project_root, 'data/big_bang/pienza.db')

print("--- Architecting the definitive 'v_trip_final_kpis' view (Data Lake B - v2.2) ---")

view_sql = """
CREATE VIEW v_trip_final_kpis AS
WITH
ImputedFlag AS (
    SELECT
        trip_id_legacy,
        MAX(is_imputed) AS is_imputed
    FROM
        trip_events
    GROUP BY
        trip_id_legacy
),
Durations AS (
    SELECT
        trip_id_legacy,
        (julianday(t1_timestamp) - julianday(t0_timestamp)) * 86400.0 AS duration_lfr_sec,
        (julianday(COALESCE(t2_timestamp, t3_timestamp, t4_timestamp)) - julianday(t1_timestamp)) * 86400.0 AS duration_dtp_sec,
        (julianday(t3_timestamp) - julianday(t2_timestamp)) * 86400.0 AS duration_wfp_sec,
        (julianday(t4_timestamp) - julianday(t3_timestamp)) * 86400.0 AS duration_on_ride_sec,
        (julianday(MAX(
            IFNULL(t0_timestamp, '0000-01-01'), IFNULL(t1_timestamp, '0000-01-01'),
            IFNULL(t2_timestamp, '0000-01-01'), IFNULL(t3_timestamp, '0000-01-01'),
            IFNULL(t4_timestamp, '0000-01-01')
        )) -
         julianday(MIN(
            IFNULL(t0_timestamp, '9999-12-31'), IFNULL(t1_timestamp, '9999-12-31'),
            IFNULL(t2_timestamp, '9999-12-31'), IFNULL(t3_timestamp, '9999-12-31'),
            IFNULL(t4_timestamp, '9999-12-31')
        ))) * 86400.0 AS total_engagement_duration_sec
    FROM
        v_trip_funnel_wide
    GROUP BY
        trip_id_legacy
)
SELECT
    v.trip_id_legacy,
    DATE(v.t1_timestamp) AS trip_date,
    d.duration_lfr_sec,
    d.duration_dtp_sec,
    d.duration_wfp_sec,
    d.duration_on_ride_sec,
    (COALESCE(d.duration_lfr_sec, 0) +
     COALESCE(d.duration_dtp_sec, 0) +
     COALESCE(d.duration_wfp_sec, 0) +
     COALESCE(d.duration_on_ride_sec, 0)) AS total_duration_sec,
    v.upfront_fare,
    v.realized_fare,
    CASE WHEN v.upfront_fare > 0 THEN v.realized_fare / v.upfront_fare ELSE NULL END AS spread_percentage,
    CASE WHEN d.duration_on_ride_sec > 0 THEN v.realized_fare / (d.duration_on_ride_sec / 3600.0) ELSE NULL END AS eph_on_ride,
    CASE WHEN d.total_engagement_duration_sec > 0 THEN v.realized_fare / (d.total_engagement_duration_sec / 3600.0) ELSE NULL END AS eph_total_time,
    i.is_imputed
FROM
    v_trip_funnel_wide v
LEFT JOIN
    Durations d ON v.trip_id_legacy = d.trip_id_legacy
LEFT JOIN
    ImputedFlag i ON v.trip_id_legacy = i.trip_id_legacy;
"""

try:
    with sqlite3.connect(db_file_path) as conn:
        conn.execute("DROP VIEW IF EXISTS v_trip_final_kpis;")
        conn.execute(view_sql)
    print("SUCCESS: Final KPI view 'v_trip_final_kpis' (v2.2) created successfully in Data Lake B.")

    print("\n--- Displaying data from the last 10 trips for verification ---")
    with sqlite3.connect(db_file_path) as conn:
        verification_query = "SELECT * FROM v_trip_final_kpis ORDER BY trip_date DESC, trip_id_legacy DESC LIMIT 10;"
        df_verify = pd.read_sql_query(verification_query, conn)

        def _obf_address(v):
            if v is None or str(v) in ('None', 'nan', ''):
                return v
            _mask_digits = lambda m: re.sub(r'\d', '#', m.group(0))
            return re.sub(r'\b(?!\d{5}\b)\d+[\w-]*\b', _mask_digits, str(v), count=1)

        def _obf_coord(v):
            if v is None or pd.isna(v):
                return v
            digits_seen = 0
            out = []
            for ch in str(v):
                if ch.isdigit():
                    digits_seen += 1
                    out.append(ch if digits_seen <= 4 else '#')
                else:
                    out.append(ch)
            return ''.join(out)

        def _obf_fare(v):
            if v is None or pd.isna(v):
                return v
            int_part, dec_part = f"{float(v):.2f}".split('.')
            masked_int = int_part[:2] + '#' * max(0, len(int_part) - 2)
            return f"{masked_int}.{'#' * len(dec_part)}"

        df_verify_display = df_verify.copy()
        for col in df_verify_display.columns:
            if 'address' in col:
                df_verify_display[col] = df_verify_display[col].apply(_obf_address)
            elif col.endswith('_lat') or col.endswith('_lon'):
                df_verify_display[col] = df_verify_display[col].apply(_obf_coord)
            elif 'fare' in col:
                df_verify_display[col] = df_verify_display[col].apply(_obf_fare)
        display(df_verify_display)

except Exception as e:
    print(f"ERROR: View creation failed in B. Reason: {e}")

--- Architecting the definitive 'v_trip_final_kpis' view (Data Lake B - v2.2) ---
SUCCESS: Final KPI view 'v_trip_final_kpis' (v2.2) created successfully in Data Lake B.

--- Displaying data from the last 10 trips for verification ---


,trip_id_legacy,trip_date,duration_lfr_sec,duration_dtp_sec,duration_wfp_sec,duration_on_ride_sec,total_duration_sec,upfront_fare,realized_fare,spread_percentage,eph_on_ride,eph_total_time,is_imputed
0,251001-07,2025-10-01,NaN,259.999998,48.000021,1992.000018,2300.000037,13#.##,12#.##,0.925907,226.518070,196.184345,0
1,251001-06,2025-10-01,111.000001,455.999981,NaN,1758.000000,2324.999982,16#.##,14#.##,0.894133,303.358362,229.378066,1
2,251001-05,2025-10-01,273.000008,392.000006,22.000001,1791.999976,2478.999992,12#.##,10#.##,0.887037,214.854914,155.312627,0
3,251001-04,2025-10-01,NaN,106.000029,101.999970,347.000009,555.000007,13#.##,12#.##,0.899153,1245.060487,778.443233,0
4,251001-03,2025-10-01,NaN,613.000014,69.999982,1893.999986,2576.999983,18#.##,15#.##,0.856727,296.648365,218.025613,0
5,251001-02,2025-10-01,176.000011,395.999984,70.000023,1053.999996,1696.000014,11#.##,97.##,0.849246,332.675523,206.745281,1
6,251001-01,2025-10-01,212.000017,390.000017,168.999968,1119.000006,1890.000008,15#.##,14#.##,0.941639,458.348523,271.371427,0
7,250930-10,2025-09-30,266.000006,256.999974,NaN,1545.000029,2068.000008,19#.##,16#.##,0.838819,380.038828,283.926498,1
8,250930-09,2025-09-30,NaN,148.000002,NaN,1512.000006,1660.000008,10#.##,84.##,0.825964,201.928571,183.925300,0
9,250930-08,2025-09-30,351.999982,501.000018,85.999976,818.000029,1757.000005,10#.##,10#.##,0.942797,451.892405,210.385884,0


#### 2.11 — Final cascade: ETL for trip_events

The definitive `trip_events` ETL: stages GTS-4 data, then runs the three-tier linking cascade (exact match, fuzzy fare match, orphan-offer match) against `offers`.

In [73]:
print("\n--- PHASE 3: ENRICHING & LOADING trip_events (Data Lake B) ---")

warnings.simplefilter(action='ignore', category=FutureWarning)

print("Executing extraction from GCS Data Lake...")
try:
    # Source 1: 'trip_events' data from GCS bucket
    GCS_FILE_NAME = f"{DATE_PREFIX}_gts-4_trip_events.csv"
    blob = bucket.blob(GCS_FILE_NAME)
    content = blob.download_as_bytes()
    df_raw_events = pd.read_csv(io.BytesIO(content), dtype=str, na_filter=False)
    
    df_raw_events.columns = df_raw_events.columns.str.lower().str.replace(' ', '_')
    df_raw_events.reset_index(inplace=True)
    df_raw_events.rename(columns={'index': 'event_id'}, inplace=True)
    df_raw_events['event_id'] = df_raw_events['event_id'].astype(int) + 1
    print(f"Extracted and prepared {len(df_raw_events)} raw events from GCS.")

    # Source 2 & 3: 'offers' and 'event_types' tables from Pienza DB
    with sqlite3.connect(db_file_path) as conn:
        offers_query = "SELECT offer_id, session_fk, offer_timestamp, upfront_fare FROM offers WHERE offer_action_fk = 1;"
        df_offers = pd.read_sql_query(offers_query, conn)
        df_offers['offer_timestamp'] = pd.to_datetime(df_offers['offer_timestamp'], format='%Y:%m:%d %H:%M:%S', errors='coerce')
        df_event_types = pd.read_sql_query("SELECT * FROM event_types;", conn)
    print(f"Extracted {len(df_offers)} accepted offers and {len(df_event_types)} event types from database.")

except Exception as e:
    print(f"ERROR: GCS Extraction failed. Reason: {e}")
    raise SystemExit("Halting execution.")

print("\n--- [T] TRANSFORMING and enriching 'trip_events' data ---")

df_staged_events = pd.merge(df_raw_events, df_event_types, left_on='event_types_id_fk', right_on='description', how='left')
df_staged_events.drop(columns=['event_types_id_fk', 'description', 'event_code', 'event_name'], inplace=True, errors='ignore')
df_staged_events.rename(columns={'event_type_id': 'event_types_id_fk'}, inplace=True)
df_staged_events.replace({'': None, 'N/A': None}, inplace=True)
df_staged_events['event_timestamp'] = pd.to_datetime(df_staged_events['event_timestamp'], errors='coerce')
df_staged_events['upfront_fare'] = pd.to_numeric(df_staged_events['upfront_fare'].astype(str).str.replace(r'[MX$,]', '', regex=True), errors='coerce')
df_staged_events.dropna(subset=['event_id', 'event_timestamp'], inplace=True)
df_staged_events['event_id'] = df_staged_events['event_id'].astype(int)
print("Preliminary data cleaning and FK enrichment complete.")

df_t1_events = df_staged_events[df_staged_events['event_types_id_fk'] == 2].copy()
df_offers_timed = df_offers[df_offers['offer_timestamp'].notna()].copy()
df_offers_orphan = df_offers[df_offers['offer_timestamp'].isna()].copy()

df_t1_events['day'] = df_t1_events['event_timestamp'].dt.strftime('%Y-%m-%d')
df_offers_timed['day'] = df_offers_timed['offer_timestamp'].dt.strftime('%Y-%m-%d')

print("\nExecuting Tier 1 & 2: Matching against TIMED offers...")
perfect_matches = pd.merge(df_t1_events, df_offers_timed, on=['day', 'upfront_fare'], how='inner')
if not perfect_matches.empty: 
    perfect_matches['is_fuzzy_match'] = False
    perfect_matches['is_imputed_link'] = False
print(f"Found {len(perfect_matches)} perfect timed matches.")

perfectly_matched_ids = perfect_matches['event_id'].unique() if not perfect_matches.empty else []
events_for_fuzzy_match = df_t1_events[~df_t1_events['event_id'].isin(perfectly_matched_ids)].copy()
if not events_for_fuzzy_match.empty:
    events_for_fuzzy_match['rounded_fare'] = events_for_fuzzy_match['upfront_fare'].fillna(0).astype(int)
    df_offers_timed['rounded_fare'] = df_offers_timed['upfront_fare'].fillna(0).astype(int)
    fuzzy_matches = pd.merge(events_for_fuzzy_match, df_offers_timed, on=['day', 'rounded_fare'], how='inner')
    if not fuzzy_matches.empty: 
        fuzzy_matches['is_fuzzy_match'] = True
        fuzzy_matches['is_imputed_link'] = False
        print(f"Found {len(fuzzy_matches)} fuzzy timed matches.")
    else:
        fuzzy_matches = pd.DataFrame()
        print("Found 0 fuzzy timed matches.")
else:
    fuzzy_matches = pd.DataFrame()
    print("Found 0 fuzzy timed matches.")

timed_concat = pd.concat([perfect_matches, fuzzy_matches]) if not pd.concat([perfect_matches, fuzzy_matches]).empty else pd.DataFrame()
timed_matched_ids = timed_concat['event_id'].unique() if not timed_concat.empty else []
events_for_orphan_match = df_t1_events[~df_t1_events['event_id'].isin(timed_matched_ids)]

print(f"\nExecuting Tier 3: Matching {len(events_for_orphan_match)} remaining events against ORPHAN offers...")
orphan_matches = pd.merge(events_for_orphan_match, df_offers_orphan, on='upfront_fare', how='inner')
if not orphan_matches.empty:
    orphan_matches.drop_duplicates(subset='event_id', keep='first', inplace=True)
    orphan_matches['is_imputed_link'] = True
    orphan_matches['is_fuzzy_match'] = False
print(f"Found {len(orphan_matches)} imputed links (orphan matches).")

all_matches = pd.concat([perfect_matches, fuzzy_matches, orphan_matches], ignore_index=True)

# SURGICAL FIX: Ensure columns exist to prevent KeyError
for col in ['is_fuzzy_match', 'is_imputed_link']:
    if col not in all_matches.columns: all_matches[col] = False

final_matched_ids = all_matches['event_id'].unique() if not all_matches.empty else []
final_unmatched = df_t1_events[~df_t1_events['event_id'].isin(final_matched_ids)]

if not final_unmatched.empty:
    print(f"\nANOMALY REPORT: Found {len(final_unmatched)} T1 events that are TRUE anomalies.")

    def _obf_fare(v):
        if v is None or pd.isna(v):
            return v
        int_part, dec_part = f"{float(v):.2f}".split('.')
        masked_int = int_part[:2] + '#' * max(0, len(int_part) - 2)
        return f"{masked_int}.{'#' * len(dec_part)}"

    df_anomaly_display = final_unmatched[['event_id', 'trip_id_legacy', 'event_timestamp', 'upfront_fare']].copy()
    df_anomaly_display['upfront_fare'] = df_anomaly_display['upfront_fare'].apply(_obf_fare)
    display(df_anomaly_display)
else:
    print("\nAll T1 events were successfully matched.")

print("\n--- [L] Merging links and loading to database ---")
if not all_matches.empty:
    df_linked_final = pd.merge(df_staged_events, all_matches[['event_id', 'offer_id', 'session_fk', 'is_fuzzy_match', 'is_imputed_link']], on='event_id', how='left')
else:
    df_linked_final = df_staged_events.copy()
    df_linked_final['offer_id'], df_linked_final['session_fk'], df_linked_final['is_imputed_link'], df_linked_final['is_fuzzy_match'] = None, None, False, False

df_linked_final.rename(columns={'offer_id': 'offer_id_fk', 'session_fk': 'offers_session_fk'}, inplace=True)
df_linked_final['is_imputed_link'] = df_linked_final['is_imputed_link'].fillna(False).astype(bool)
df_linked_final['is_fuzzy_match'] = df_linked_final['is_fuzzy_match'].fillna(False).astype(bool)

final_columns = [
    'event_id', 'event_timestamp', 'event_lat', 'event_lon', 'event_address',
    'upfront_fare', 'realized_fare', 'source', 'is_imputed', 'comment',
    'trip_id_legacy', 'event_types_id_fk', 'offer_id_fk', 'offers_session_fk', 'is_fuzzy_match', 'is_imputed_link'
]
df_to_load = df_linked_final[[col for col in final_columns if col in df_linked_final.columns]]

try:
    with sqlite3.connect(db_file_path) as conn:
        df_to_load.to_sql('trip_events', conn, if_exists='replace', index=False)
    print(f"SUCCESS: Loaded {len(df_to_load)} records into Data Lake 'trip_events'.")
    print("\n--- PHASE 3 COMPLETE ---")
except Exception as e:
    print(f"ERROR: Failed to load 'trip_events' table. Reason: {e}")


--- PHASE 3: ENRICHING & LOADING trip_events (Data Lake B) ---
Executing extraction from GCS Data Lake...
Extracted and prepared 1031 raw events from GCS.
Extracted 346 accepted offers and 5 event types from database.

--- [T] TRANSFORMING and enriching 'trip_events' data ---
Preliminary data cleaning and FK enrichment complete.

Executing Tier 1 & 2: Matching against TIMED offers...
Found 0 perfect timed matches.
Found 0 fuzzy timed matches.

Executing Tier 3: Matching 259 remaining events against ORPHAN offers...
Found 259 imputed links (orphan matches).

All T1 events were successfully matched.

--- [L] Merging links and loading to database ---
SUCCESS: Loaded 1031 records into Data Lake 'trip_events'.

--- PHASE 3 COMPLETE ---


#### 2.12 — Golden propagation patch (TD-002)

Fixes a tracked bug where linked offer IDs weren't propagating correctly across grouped trip events.

In [74]:
print("--- Initiating Golden Propagation Patch (Data Lake B) ---")

if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/workspaces/pienza'
    db_file_path = os.path.join(project_root, 'data/big_bang/pienza.db')

if not os.path.exists(db_file_path):
    raise FileNotFoundError(f"CRITICAL ERROR: Database file not found at path: {db_file_path}")

print("Database path verified successfully.")

try:
    with sqlite3.connect(db_file_path) as conn:
        events_to_patch_df = pd.read_sql_query('SELECT * FROM trip_events;', conn)
    print(f"SUCCESS: Read {len(events_to_patch_df)} events to be patched.")
except Exception as e:
    print(f"ERROR: Could not read from trip_events table. Details: {e}")
    raise

print("Applying propagation logic across all linked attributes...")
events_to_patch_df = events_to_patch_df.sort_values(by=['trip_id_legacy', 'event_timestamp'])

link_columns = ['offer_id_fk', 'offers_session_fk', 'is_fuzzy_match', 'is_imputed_link']
for col in link_columns:
    if col in events_to_patch_df.columns:
        events_to_patch_df[col] = events_to_patch_df.groupby('trip_id_legacy')[col].transform(lambda x: x.ffill().bfill())

print("Propagation logic applied in memory.")

# In-line Unit Test
known_trip_id = 'GTS-4-20251001-073003-B'
if known_trip_id in events_to_patch_df['trip_id_legacy'].values:
    propagated_ids = events_to_patch_df[events_to_patch_df['trip_id_legacy'] == known_trip_id]['offer_id_fk']
    assert propagated_ids.notna().all(), f"Validation failed: Nulls still exist for trip {known_trip_id}"
    assert propagated_ids.nunique() == 1, f"Validation failed: Multiple different offer_ids found for trip {known_trip_id}"
    print(f"Validation successful for test case trip: {known_trip_id}")
else:
    print(f"Warning: Test case trip ID '{known_trip_id}' not found. Skipping validation.")

print("Writing patched data back to pienza.db...")
try:
    with sqlite3.connect(db_file_path) as conn:
        events_to_patch_df.to_sql('trip_events', conn, if_exists='replace', index=False)
    print("SUCCESS: Patched data successfully written to Data Lake database.")
except Exception as e:
    print(f"ERROR: Failed to write patched data. Details: {e}")

print("\n--- MISSION COMPLETE ---")
print("TD-002 Resolved. The 'trip_events' table has been successfully propagated in B.")

--- Initiating Golden Propagation Patch (Data Lake B) ---
Database path verified successfully.
SUCCESS: Read 1031 events to be patched.
Applying propagation logic across all linked attributes...


Propagation logic applied in memory.
Writing patched data back to pienza.db...
SUCCESS: Patched data successfully written to Data Lake database.

--- MISSION COMPLETE ---
TD-002 Resolved. The 'trip_events' table has been successfully propagated in B.


#### 2.13 — Post-ETL golden-link patch

Follow-up patch reinforcing the same link after the final cascade ran.

In [75]:
print("--- Starting the Post-ETL 'Golden Link' Patch for trip_events (Data Lake B) ---")

if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/workspaces/pienza'
    db_file_path = os.path.join(project_root, 'data/big_bang/pienza.db')

override_rules = {
    55: 'OF00318'
}

print(f"Found {len(override_rules)} override rule(s) to apply.")

try:
    with sqlite3.connect(db_file_path) as conn:
        cursor = conn.cursor()

        df_offers = pd.read_sql_query("SELECT offer_id, session_fk FROM offers;", conn)

        for event_id, correct_offer_id in override_rules.items():
            try:
                correct_session_fk = df_offers.loc[df_offers['offer_id'] == correct_offer_id, 'session_fk'].iloc[0]
            except IndexError:
                print(f"WARNING: Could not find correct_offer_id '{correct_offer_id}' in the offers table. Skipping override for event {event_id}.")
                continue

            update_sql = """
            UPDATE trip_events
            SET
                offer_id_fk = ?,
                offers_session_fk = ?
            WHERE
                event_id = ?;
            """

            cursor.execute(update_sql, (correct_offer_id, correct_session_fk, event_id))
            print(f"  - Override Applied: Event ID {event_id} is now linked to Offer ID {correct_offer_id} (Session: {correct_session_fk}).")

        conn.commit()

    print("\nSUCCESS: Patching complete.")

    print("\n--- Verifying the fix for Event ID 55 ---")
    with sqlite3.connect(db_file_path) as conn:
        verify_query = "SELECT event_id, offer_id_fk, offers_session_fk FROM trip_events WHERE event_id = 55;"
        df_verify = pd.read_sql_query(verify_query, conn)
        display(df_verify)

except Exception as e:
    print(f"ERROR during patching process in B: {e}")

--- Starting the Post-ETL 'Golden Link' Patch for trip_events (Data Lake B) ---
Found 1 override rule(s) to apply.
  - Override Applied: Event ID 55 is now linked to Offer ID OF00318 (Session: SID0006).

SUCCESS: Patching complete.

--- Verifying the fix for Event ID 55 ---


,event_id,offer_id_fk,offers_session_fk
0,55,OF00318,SID0006


#### 2.14 — Side-by-side reconciliation audit

Compares event-level and offer-level fares for every linked trip to confirm the linkage is exact.

In [76]:
print("--- Performing the final 'Side-by-Side' Reconciliation Audit (Data Lake B) ---")

if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/workspaces/pienza'
    db_file_path = os.path.join(project_root, 'data/big_bang/pienza.db')

query_sql = """
SELECT
    te.trip_id_legacy,
    te.offer_id_fk,
    te.upfront_fare AS event_upfront_fare,
    o.upfront_fare AS linked_offer_upfront_fare,
    (te.upfront_fare - o.upfront_fare) AS fare_discrepancy,
    te.is_imputed_link
FROM
    trip_events te
LEFT JOIN
    offers o ON te.offer_id_fk = o.offer_id
WHERE
    te.event_types_id_fk = 2
ORDER BY
    te.event_timestamp ASC;
"""

try:
    with sqlite3.connect(db_file_path) as conn:
        df_reconciliation = pd.read_sql_query(query_sql, conn)

    print("SUCCESS: Reconciliation query executed successfully.")
    print("Displaying a side-by-side comparison for all linked trips (fare values masked for display):")

    def _obf_fare(v):
        if v is None or pd.isna(v):
            return "-"
        int_part, dec_part = f"{float(v):.2f}".split('.')
        masked_int = int_part[:2] + '#' * max(0, len(int_part) - 2)
        return f"{masked_int}.{'#' * len(dec_part)}"

    pd.set_option('display.max_rows', None)

    df_display_reconciliation = df_reconciliation.copy()
    for col in ['event_upfront_fare', 'linked_offer_upfront_fare', 'fare_discrepancy']:
        df_display_reconciliation[col] = df_display_reconciliation[col].apply(_obf_fare)

    display(df_display_reconciliation)

    mismatch_count = df_reconciliation[df_reconciliation['fare_discrepancy'].abs() > 0.01].shape[0]

    print("\n--- FINAL VERDICT ---")
    if mismatch_count > 0:
        print(f"WARNING: Found {mismatch_count} linked records with a significant fare discrepancy. Manual review required.")
    else:
        print("SUCCESS: All linked records have a perfect fare match. The linkage is 100% validated.")

    pd.reset_option('display.max_rows')

except Exception as e:
    print(f"ERROR during reconciliation audit in Data Lake B: {e}")

--- Performing the final 'Side-by-Side' Reconciliation Audit (Data Lake B) ---
SUCCESS: Reconciliation query executed successfully.
Displaying a side-by-side comparison for all linked trips (fare values masked for display):


,trip_id_legacy,offer_id_fk,event_upfront_fare,linked_offer_upfront_fare,fare_discrepancy,is_imputed_link
0,250822-01,OF00003,13#.##,13#.##,0.##,1
1,250822-02,OF00012,16#.##,16#.##,0.##,1
2,250822-03,OF00032,64.##,64.##,0.##,1
3,250822-04,OF00034,14#.##,14#.##,0.##,1
4,250822-05,OF00035,60.##,60.##,0.##,1
5,250822-06,OF00101,10#.##,10#.##,0.##,1
6,250822-07,OF00113,17#.##,17#.##,0.##,1
7,250822-08,OF00130,17#.##,17#.##,0.##,1
8,250822-09,OF00167,89.##,89.##,0.##,1
9,250822-10,OF00174,20#.##,20#.##,0.##,1



--- FINAL VERDICT ---
SUCCESS: All linked records have a perfect fare match. The linkage is 100% validated.


## Phase 3 — lifetime_trips and activity_earnings

Builds the driver-level trip history (`lifetime_trips`) and per-trip bank earnings (`activity_earnings`) tables, with link-integrity validation against the schema established in Phase 2.

#### 3.1 — Link-integrity audit (v2.0)

Validates the `ocr_fk` link between `offers` and `raw_offers_ocr`.

In [77]:
if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/workspaces/pienza'
    db_file_path = os.path.join(project_root, 'data/big_bang/pienza.db')

print("--- Verifying the integrity of the ocr_fk link between 'offers' and 'raw_offers_ocr' (Data Lake B) ---")

query_sql = """
SELECT
    CASE
        WHEN ocr.ocr_id IS NULL THEN 'Orphaned (Link Failed)'
        ELSE 'Linked Successfully'
    END AS link_status,
    COUNT(o.offer_id) AS number_of_offers
FROM
    offers o
LEFT JOIN
    raw_offers_ocr ocr ON o.ocr_fk = ocr.ocr_id
GROUP BY
    link_status;
"""

try:
    with sqlite3.connect(f'file:{db_file_path}?mode=ro', uri=True) as conn:
        df_link_status = pd.read_sql_query(query_sql, conn)

    print("\nSUCCESS: Audit query executed.")
    print("Displaying link status count:")
    display(df_link_status)

    if 'Orphaned (Link Failed)' in df_link_status['link_status'].values:
        print("\nFINDING: One or more offers could not be linked back to their OCR source in B. Investigation required.")
    else:
        print("\nFINDING: 100% of offers are successfully linked to their raw OCR source in B. Data provenance is intact.")

except Exception as e:
    print(f"ERROR during audit in B: {e}")

--- Verifying the integrity of the ocr_fk link between 'offers' and 'raw_offers_ocr' (Data Lake B) ---

SUCCESS: Audit query executed.
Displaying link status count:


,link_status,number_of_offers
0,Orphaned (Link Failed),4765



FINDING: One or more offers could not be linked back to their OCR source in B. Investigation required.


#### 3.2 — Schema audit: finalizing the offers schema

Confirms the canonical column list for `offers`.

In [78]:
if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/workspaces/pienza'
    db_file_path = os.path.join(project_root, 'data/big_bang/pienza.db')

table_to_audit = 'offers'
print(f"\n--- Auditing the Canonical Schema of the '{table_to_audit}' table (Data Lake B) ---")

query_sql = f"PRAGMA table_info({table_to_audit});"

try:
    with sqlite3.connect(f'file:{db_file_path}?mode=ro', uri=True) as conn:
        df_schema = pd.read_sql_query(query_sql, conn)

    print("\nSUCCESS: Schema audit complete.")
    print(f"This is the definitive, canonical list of columns for the '{table_to_audit}' table:")
    display(df_schema)

except Exception as e:
    print(f"ERROR: Audit query failed in Data Lake B. Reason: {e}")


--- Auditing the Canonical Schema of the 'offers' table (Data Lake B) ---

SUCCESS: Schema audit complete.
This is the definitive, canonical list of columns for the 'offers' table:


,cid,name,type,notnull,dflt_value,pk
0,0,offer_id,TEXT,0,None,0
1,1,session_fk,TEXT,0,None,0
2,2,ocr_fk,TEXT,0,None,0
3,3,image_content_hash,TEXT,0,None,0
4,4,offer_timestamp,TEXT,0,None,0
5,5,upfront_fare,REAL,0,None,0
6,6,time_to_pickup_sec,REAL,0,None,0
7,7,dist_to_pickup_km,REAL,0,None,0
8,8,est_trip_time_sec,REAL,0,None,0
9,9,est_trip_dist_km,REAL,0,None,0


#### 3.3 — Link-integrity final validation (v2.1)

Final validation pass on the same link, after schema was confirmed.

In [79]:
print("\n--- Verifying the integrity of the ocr_fk link between 'offers' and 'raw_offers_ocr' (Data Lake B) ---")

if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/workspaces/pienza'
    db_file_path = os.path.join(project_root, 'data/big_bang/pienza.db')

query_sql = """
SELECT
    CASE
        WHEN ocr.ocr_id IS NULL THEN 'Orphaned (Link Failed)'
        ELSE 'Linked Successfully'
    END AS link_status,
    COUNT(o.offer_id) AS number_of_offers
FROM
    offers o
LEFT JOIN
    raw_offers_ocr ocr ON o.ocr_fk = ocr.ocr_id
GROUP BY
    link_status;
"""

try:
    with sqlite3.connect(db_file_path) as conn:
        df_link_status = pd.read_sql_query(query_sql, conn)

    print("\nSUCCESS: Audit query executed.")
    print("Displaying link status count:")
    display(df_link_status)

    if 'Orphaned (Link Failed)' in df_link_status['link_status'].values:
        print("\nFINDING: One or more offers could not be linked back to their OCR source in B. Investigation required.")
    else:
        print("\nFINDING: 100% of offers are successfully linked to their raw OCR source in B. Data provenance is intact.")

except Exception as e:
    print(f"ERROR during audit in Data Lake B: {e}")


--- Verifying the integrity of the ocr_fk link between 'offers' and 'raw_offers_ocr' (Data Lake B) ---

SUCCESS: Audit query executed.
Displaying link status count:


,link_status,number_of_offers
0,Orphaned (Link Failed),4765



FINDING: One or more offers could not be linked back to their OCR source in B. Investigation required.


#### 3.4 — ETL for activity_earnings

Builds the per-trip bank earnings table.

In [80]:
print("--- PHASE 5 STARTED: ETL for 'activity_earnings' table (Data Lake B) ---")

if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/workspaces/pienza'
    db_file_path = os.path.join(project_root, 'data/big_bang/pienza.db')

GCS_FILE_NAME = f"{DATE_PREFIX}_platform_data_activity_earnings.csv"
TARGET_TABLE = "activity_earnings"

print("\n--- [E] EXTRACTING raw data from GCS ---")
df_raw_earnings = None
try:
    blob = bucket.blob(GCS_FILE_NAME)
    content = blob.download_as_bytes()
    df_raw_earnings = pd.read_csv(io.BytesIO(content), header=None, skiprows=1, dtype=str)
    
    canonical_headers = [
        'activity_earnings_id', 'activity_earnings_timestamp', 'product_category',
        'net_earning', 'details_url'
    ]
    df_raw_earnings = df_raw_earnings.iloc[:, 0:len(canonical_headers)]
    df_raw_earnings.columns = canonical_headers
    print(f"SUCCESS: Extracted {len(df_raw_earnings)} rows from GCS using canonical headers.")

except Exception as e:
    print(f"ERROR during GCS extraction: {e}")

if df_raw_earnings is not None:
    print("\n--- [T] TRANSFORMING and cleaning 'activity_earnings' data ---")
    df_transformed = df_raw_earnings.copy()

    df_transformed.replace({'': None, 'N/A': None, 'nan': None}, inplace=True)
    df_transformed['net_earning'] = pd.to_numeric(df_transformed['net_earning'].astype(str).str.replace(r'[MX$,]', '', regex=True).str.strip(), errors='coerce')
    df_transformed['activity_earnings_id'] = df_transformed['activity_earnings_id'].astype(str).str.replace('AE', '').pipe(pd.to_numeric, errors='coerce')

    def intelligent_date_parser(date_string):
        if not isinstance(date_string, str) or date_string.strip() == '': return pd.NaT
        try: return pd.to_datetime(date_string)
        except:
            try:
                s_cleaned = date_string.replace('st', '').replace('nd', '').replace('rd', '').replace('th', '')
                return pd.to_datetime(s_cleaned, format='%A, %B %d, %Y\n%H:%M')
            except: return pd.NaT

    df_transformed['activity_earnings_timestamp'] = df_transformed['activity_earnings_timestamp'].apply(intelligent_date_parser)
    
    initial_rows = len(df_transformed)
    df_transformed.dropna(subset=['activity_earnings_timestamp', 'activity_earnings_id'], inplace=True)
    df_transformed['activity_earnings_id'] = df_transformed['activity_earnings_id'].astype(int)
    print(f"Dropped {initial_rows - len(df_transformed)} rows with missing critical data.")

    df_transformed.sort_values('activity_earnings_timestamp', inplace=True)
    df_transformed.reset_index(drop=True, inplace=True)
    
    df_to_load = df_transformed[['activity_earnings_id', 'activity_earnings_timestamp', 'product_category', 'net_earning', 'details_url']]

    print(f"\n--- Displaying diagnostics for {len(df_to_load)} records ---")
    if not df_to_load.empty:
        def _obf_fare(v):
            if v is None or pd.isna(v):
                return v
            int_part, dec_part = f"{float(v):.2f}".split('.')
            masked_int = int_part[:2] + '#' * max(0, len(int_part) - 2)
            return f"{masked_int}.{'#' * len(dec_part)}"

        df_to_load_display = df_to_load.copy()
        df_to_load_display['net_earning'] = df_to_load_display['net_earning'].apply(_obf_fare)

        print("\nFirst 20 records (True chronological start):")
        display(df_to_load_display.head(20))
        print("\nLast 20 records (True chronological end):")
        display(df_to_load_display.tail(20))

    print(f"\n--- [L] LOADING data into database ---")
    try:
        with sqlite3.connect(db_file_path) as conn:
            df_to_load.to_sql(TARGET_TABLE, conn, if_exists='replace', index=False)
        print(f"SUCCESS: Loaded {len(df_to_load)} records into '{TARGET_TABLE}'.")
    except Exception as e:
        print(f"ERROR during load operation: {e}")

--- PHASE 5 STARTED: ETL for 'activity_earnings' table (Data Lake B) ---

--- [E] EXTRACTING raw data from GCS ---
SUCCESS: Extracted 3446 rows from GCS using canonical headers.

--- [T] TRANSFORMING and cleaning 'activity_earnings' data ---
Dropped 0 rows with missing critical data.

--- Displaying diagnostics for 3446 records ---

First 20 records (True chronological start):


,activity_earnings_id,activity_earnings_timestamp,product_category,net_earning,details_url
0,1,2023-05-29 12:18:00,UberX,36.##,NaN
1,2,2023-05-29 12:39:00,UberX,41.##,NaN
2,3,2023-05-29 13:05:00,UberX,75.##,NaN
3,4,2023-05-29 13:47:00,UberX,69.##,NaN
4,5,2023-05-29 19:49:00,UberX,49.##,NaN
5,6,2023-05-29 20:21:00,UberX,34.##,NaN
6,7,2023-05-29 20:32:00,UberX,53.##,NaN
7,8,2023-06-05 11:38:00,UberX,82.##,NaN
8,9,2023-06-05 17:00:00,UberX,61.##,NaN
9,10,2023-06-05 17:31:00,UberX,56.##,NaN



Last 20 records (True chronological end):


,activity_earnings_id,activity_earnings_timestamp,product_category,net_earning,details_url
3426,3427,2025-09-29 17:11:00,Uber Priority,19#.##,View Details
3427,3428,2025-09-30 06:02:00,UberX,11#.##,View Details
3428,3429,2025-09-30 06:37:00,Uber Priority,13#.##,View Details
3429,3430,2025-09-30 07:20:00,Uber Priority,90.##,View Details
3430,3431,2025-09-30 07:47:00,UberX,0.##,View Details
3431,3432,2025-09-30 08:16:00,Comfort,12.##,View Details
3432,3433,2025-09-30 08:36:00,Uber Priority,12#.##,View Details
3433,3434,2025-09-30 13:41:00,Comfort,62.##,View Details
3434,3435,2025-09-30 14:04:00,UberX,89.##,View Details
3435,3436,2025-09-30 14:44:00,Comfort,88.##,View Details



--- [L] LOADING data into database ---
SUCCESS: Loaded 3446 records into 'activity_earnings'.


#### 3.5 — ETL for lifetime_trips

Builds the driver-level trip history table.

In [81]:
print("--- PHASE 6 STARTED: ETL for 'lifetime_trips' table (Data Lake B) ---")

if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/workspaces/pienza'
    db_file_path = os.path.join(project_root, 'data/big_bang/pienza.db')

GCS_FILE_NAME = f"{DATE_PREFIX}_platform_data_lifetime_trips.csv"
TARGET_TABLE = "lifetime_trips"

print(f"\n--- [E] EXTRACTING {GCS_FILE_NAME} from GCS ---")
df_raw_trips = None
try:
    blob = bucket.blob(GCS_FILE_NAME)
    content = blob.download_as_bytes()
    
    # Read without headers to maintain Resilient Header logic
    df_raw_trips = pd.read_csv(io.BytesIO(content), header=None, skiprows=1, dtype=str)
    
    canonical_headers = [
        'lifetime_trips_id', 'global_product_name', 'status', 'request_timestamp',
        'pickup_timestamp', 'dropoff_timestamp', 'trip_distance_miles',
        'trip_duration_seconds', 'base_fare', 'original_fare',
        'cancellation_fee', 'currency_code', 'vehicle_uuid', 'license_plate'
    ]
    
    df_raw_trips = df_raw_trips.iloc[:, 0:len(canonical_headers)]
    df_raw_trips.columns = canonical_headers
    print(f"SUCCESS: Extracted {len(df_raw_trips)} rows from GCS using canonical headers.")

except Exception as e:
    print(f"ERROR during GCS extraction: {e}")

if df_raw_trips is not None:
    print("\n--- [T] TRANSFORMING and cleaning 'lifetime_trips' data ---")
    df_transformed = df_raw_trips.copy()

    df_transformed.replace({'': None, 'N/A': None, 'nan': None}, inplace=True)
    
    df_transformed['lifetime_trips_id'] = pd.to_numeric(df_transformed['lifetime_trips_id'].astype(str).str.replace('LT', ''), errors='coerce')
    
    numeric_cols = ['trip_distance_miles', 'trip_duration_seconds', 'base_fare', 'original_fare', 'cancellation_fee']
    for col in numeric_cols:
        if col in df_transformed.columns:
            df_transformed[col] = pd.to_numeric(df_transformed[col], errors='coerce')
            
    timestamp_cols = ['request_timestamp', 'pickup_timestamp', 'dropoff_timestamp']
    for col in timestamp_cols:
        if col in df_transformed.columns:
            df_transformed[col] = pd.to_datetime(df_transformed[col], errors='coerce')

    df_transformed.dropna(subset=['lifetime_trips_id', 'request_timestamp'], inplace=True)
    df_transformed['lifetime_trips_id'] = df_transformed['lifetime_trips_id'].astype(int)

    final_columns = [
        'lifetime_trips_id', 'global_product_name', 'status',
        'request_timestamp', 'pickup_timestamp', 'dropoff_timestamp',
        'trip_distance_miles', 'trip_duration_seconds', 'base_fare',
        'original_fare', 'cancellation_fee', 'currency_code',
        'vehicle_uuid', 'license_plate'
    ]
    df_to_load = df_transformed[[col for col in final_columns if col in df_transformed.columns]].copy()
    
    for col in ['vehicle_uuid', 'license_plate']:
        if col not in df_to_load.columns:
            df_to_load[col] = None

    print(f"\n--- [L] LOADING data into database ---")
    try:
        with sqlite3.connect(db_file_path) as conn:
            df_to_load.to_sql(TARGET_TABLE, conn, if_exists='replace', index=False)
        print(f"SUCCESS: Loaded {len(df_to_load)} records into '{TARGET_TABLE}'.")
        print("\n--- PHASE 6 COMPLETE ---")
    except Exception as e:
        print(f"ERROR during load operation: {e}")

--- PHASE 6 STARTED: ETL for 'lifetime_trips' table (Data Lake B) ---

--- [E] EXTRACTING 260509_platform_data_lifetime_trips.csv from GCS ---


SUCCESS: Extracted 3446 rows from GCS using canonical headers.

--- [T] TRANSFORMING and cleaning 'lifetime_trips' data ---

--- [L] LOADING data into database ---
SUCCESS: Loaded 3446 records into 'lifetime_trips'.

--- PHASE 6 COMPLETE ---


#### 3.6 — Forge the golden link

Establishes the definitive foreign-key relationship between `lifetime_trips` and `activity_earnings`.

In [82]:
print("--- ️ TASK A STARTED: Forging the 'Golden Link' using Direct ID Match (Data Lake B) ---")

if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/workspaces/pienza'
    db_file_path = os.path.join(project_root, 'data/big_bang/pienza.db')

try:
    with sqlite3.connect(db_file_path) as conn:
        print(f"Extracting source tables from {os.path.basename(db_file_path)}...")
        df_earnings = pd.read_sql_query("SELECT * FROM activity_earnings;", conn)
        df_earnings['activity_earnings_timestamp'] = pd.to_datetime(df_earnings['activity_earnings_timestamp'])

        df_lifetime = pd.read_sql_query("SELECT lifetime_trips_id FROM lifetime_trips;", conn)
        df_lifetime['lifetime_trips_id'] = pd.to_numeric(df_lifetime['lifetime_trips_id'], errors='coerce') 
    
    print(f"Extracted {len(df_earnings)} earnings records and {len(df_lifetime)} lifetime trips.")

    print("\n--- [T] Performing Direct INNER JOIN on Primary Keys (1:1 Match) ---")

    df_linked = pd.merge(
        df_earnings,
        df_lifetime,
        left_on='activity_earnings_id',
        right_on='lifetime_trips_id',
        how='inner'
    )

    print("Direct ID Join complete. Only perfectly matched records proceed.")

    df_linked.rename(columns={'lifetime_trips_id': 'lifetime_trips_fk'}, inplace=True)

    final_columns = [
        'activity_earnings_id',
        'lifetime_trips_fk',
        'activity_earnings_timestamp',
        'product_category',
        'net_earning',
        'details_url'
    ]

    df_to_load = df_linked[[col for col in final_columns if col in df_linked.columns]].copy()
    print("Schema columns finalized for loading.")

    print(f"\n--- [L] LOADING enriched data back into 'activity_earnings' table ---")
    with sqlite3.connect(db_file_path) as conn:
        df_to_load.to_sql('activity_earnings', conn, if_exists='replace', index=False)
    
    print(f"SUCCESS: Loaded {len(df_to_load)} records into 'activity_earnings' with new FK.")
    print("\n--- TASK A COMPLETE ---")

except Exception as e:
    print(f"ERROR during ETL: {e}")

print("\n--- Verifying a sample of the newly linked data ---")
try:
    with sqlite3.connect(db_file_path) as conn:
        verify_query = "SELECT * FROM activity_earnings ORDER BY activity_earnings_id DESC LIMIT 5;"
        df_verify = pd.read_sql_query(verify_query, conn)

        def _obf_fare(v):
            if v is None or pd.isna(v):
                return v
            int_part, dec_part = f"{float(v):.2f}".split('.')
            masked_int = int_part[:2] + '#' * max(0, len(int_part) - 2)
            return f"{masked_int}.{'#' * len(dec_part)}"

        df_verify_display = df_verify.copy()
        if 'net_earning' in df_verify_display.columns:
            df_verify_display['net_earning'] = df_verify_display['net_earning'].apply(_obf_fare)
        display(df_verify_display)
except Exception as e:
    print(f"ERROR during verification: {e}")

--- ️ TASK A STARTED: Forging the 'Golden Link' using Direct ID Match (Data Lake B) ---
Extracting source tables from pienza.db...
Extracted 3446 earnings records and 3446 lifetime trips.

--- [T] Performing Direct INNER JOIN on Primary Keys (1:1 Match) ---
Direct ID Join complete. Only perfectly matched records proceed.
Schema columns finalized for loading.

--- [L] LOADING enriched data back into 'activity_earnings' table ---
SUCCESS: Loaded 3446 records into 'activity_earnings' with new FK.

--- TASK A COMPLETE ---

--- Verifying a sample of the newly linked data ---


,activity_earnings_id,lifetime_trips_fk,activity_earnings_timestamp,product_category,net_earning,details_url
0,3446,3446,2025-10-01 09:06:00,Comfort,14#.##,View Details
1,3445,3445,2025-10-01 08:35:00,Business Comfort,14#.##,View Details
2,3444,3444,2025-10-01 08:27:00,Comfort,0.##,View Details
3,3443,3443,2025-10-01 07:56:00,UberX,10#.##,View Details
4,3442,3442,2025-10-01 07:33:00,Black,11#.##,View Details


## Phase 4 — Manual data corrections

A sequence of targeted overrides for specific trips/offers where automated linking failed or produced an ambiguous result, discovered and fixed one at a time during development. Each patch reads the affected record, applies a manual override, writes it back, and verifies the fix persisted. Closes with a batch override covering the remaining known cases at once.

#### 4.1 — ID-set reconciliation for pickup_address

Confirms every non-empty `pickup_address` in the source made it into the database.

In [83]:
print("--- Performing 'ID Set' Reconciliation for 'pickup_address' (Data Lake B) ---")

if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/workspaces/pienza'
    db_file_path = os.path.join(project_root, 'data/big_bang/pienza.db')

GCS_FILE_NAME = f"{DATE_PREFIX}_raw_offers_diamond_offers.csv"

try:
    print(f"Extracting ID set from GCS: '{GCS_FILE_NAME}'...")
    blob = bucket.blob(GCS_FILE_NAME)
    content = blob.download_as_bytes()
    
    # na_filter=False ensures whitespace/NULL strings are treated as data, not NaN
    df_source = pd.read_csv(io.BytesIO(content), dtype=str, na_filter=False)
    df_source.columns = df_source.columns.str.lower().str.replace(' ', '_')

    # Filter for non-empty 'pickup_address' and get the set of 'offer_id's
    gcs_ids = set(df_source[df_source['pickup_address'] != '']['offer_id'])
    print(f"Found {len(gcs_ids)} offer_ids with non-empty 'pickup_address' in GCS.")

    print(f"Extracting ID set from {os.path.basename(db_file_path)}...")
    with sqlite3.connect(db_file_path) as conn:
        db_query = "SELECT offer_id FROM offers WHERE pickup_address IS NOT NULL AND pickup_address != '';"
        df_db = pd.read_sql_query(db_query, conn)

    db_ids = set(df_db['offer_id'])
    print(f"Found {len(db_ids)} offer_ids with non-NULL 'pickup_address' in the database.")

    print("\n--- Performing Set Difference Analysis ---")

    discrepancy_ids = gcs_ids - db_ids

    if not discrepancy_ids:
        print("\nSUCCESS: The sets are identical. Reconciliation is perfect.")
    else:
        print(f"\nANOMALY DETECTED: Found {len(discrepancy_ids)} 'offer_id'(s) present in the GCS count but missing from the database count.")
        print("This is the record with the whitespace-only 'pickup_address' value.")
        print("\nAnomaly (offer_id):")
        for offer_id in sorted(list(discrepancy_ids)):
            print(offer_id)

except Exception as e:
    print(f"ERROR during diagnostic: {e}")

--- Performing 'ID Set' Reconciliation for 'pickup_address' (Data Lake B) ---
Extracting ID set from GCS: '260509_raw_offers_diamond_offers.csv'...


Found 4757 offer_ids with non-empty 'pickup_address' in GCS.
Extracting ID set from pienza.db...
Found 4757 offer_ids with non-NULL 'pickup_address' in the database.

--- Performing Set Difference Analysis ---

SUCCESS: The sets are identical. Reconciliation is perfect.


#### 4.2 — Manual override: trip 250825-01

Targeted fix for a single trip whose offer link was ambiguous.

In [84]:
print("--- Initiating Manual Override Patch B55 (Data Lake B) ---")
try:
    db_path = '/workspaces/pienza/data/big_bang/pienza.db'
    if not os.path.exists(db_path):
        raise FileNotFoundError(f"Database file not found at path: {db_path}")
    db_engine = create_engine(f'sqlite:///{db_path}')
    print("Database engine created successfully.")
except Exception as e:
    print(f"CRITICAL ERROR: Database engine failed. Details: {e}")
    raise

TARGET_TRIP_ID = '250825-01'
CORRECT_OFFER_ID = 'OF00318'

print("\n⏳ Reading current state of 'trip_events' table...")
try:
    trip_events_df = pd.read_sql('SELECT * FROM trip_events', db_engine)
    print(f"Successfully read {len(trip_events_df)} records.")
    print("\n--- VERIFICATION (BEFORE PATCH) ---")
    display(trip_events_df[trip_events_df['trip_id_legacy'] == TARGET_TRIP_ID][['trip_id_legacy', 'event_types_id_fk', 'offer_id_fk']])
except Exception as e:
    print(f"ERROR: Could not read from trip_events table. Details: {e}")
    raise

print(f"\n⏳ Applying manual override to trip '{TARGET_TRIP_ID}' in memory...")
target_mask = trip_events_df['trip_id_legacy'] == TARGET_TRIP_ID
trip_events_df.loc[target_mask, 'offer_id_fk'] = np.nan
t1_mask = (trip_events_df['trip_id_legacy'] == TARGET_TRIP_ID) & (trip_events_df['event_types_id_fk'] == 2)
trip_events_df.loc[t1_mask, 'offer_id_fk'] = CORRECT_OFFER_ID
print(f"Override applied in memory to column 'offer_id_fk'.")

print("\n⏳ Re-running Golden Propagation in memory...")
trip_events_df = trip_events_df.sort_values(by=['trip_id_legacy', 'event_timestamp'])
trip_events_df['offer_id_fk'] = trip_events_df.groupby('trip_id_legacy')['offer_id_fk'].transform(lambda x: x.ffill().bfill())
print("Propagation complete in memory.")

print("\n⏳ Writing patched data back to pienza.db...")
try:
    trip_events_df.to_sql('trip_events', db_engine, if_exists='replace', index=False)
    db_engine.dispose()
    print("   - Data written and connection disposed. Pausing for 5 seconds...")
    time.sleep(5)
    print("Write operation complete.")
except Exception as e:
    print(f"ERROR: Failed to write patched data. Details: {e}")
    raise

print("\n--- VERIFICATION (AFTER PATCH) ---")
try:
    verify_engine = create_engine(f'sqlite:///{db_path}')
    verify_df = pd.read_sql_query(f"SELECT * FROM trip_events WHERE trip_id_legacy = '{TARGET_TRIP_ID}'", verify_engine)
    verify_engine.dispose()
    display(verify_df[['trip_id_legacy', 'event_types_id_fk', 'offer_id_fk']])
    assert (verify_df['offer_id_fk'] == CORRECT_OFFER_ID).all(), "Verification Failed: Mismatch after re-reading!"
    print("VERIFICATION SUCCESS: Changes persisted to pienza.db.")
except Exception as e:
    print(f"VERIFICATION FAILED: {e}")

print("\n--- MANUAL OVERRIDE COMPLETE ---")

--- Initiating Manual Override Patch B55 (Data Lake B) ---
Database engine created successfully.

⏳ Reading current state of 'trip_events' table...
Successfully read 1031 records.

--- VERIFICATION (BEFORE PATCH) ---


,trip_id_legacy,event_types_id_fk,offer_id_fk
54,250825-01,2,OF00318
55,250825-01,4,OF00034
56,250825-01,5,OF00034



⏳ Applying manual override to trip '250825-01' in memory...
Override applied in memory to column 'offer_id_fk'.

⏳ Re-running Golden Propagation in memory...
Propagation complete in memory.

⏳ Writing patched data back to pienza.db...
   - Data written and connection disposed. Pausing for 5 seconds...


Write operation complete.

--- VERIFICATION (AFTER PATCH) ---


,trip_id_legacy,event_types_id_fk,offer_id_fk
0,250825-01,2,OF00318
1,250825-01,4,OF00318
2,250825-01,5,OF00318


VERIFICATION SUCCESS: Changes persisted to pienza.db.

--- MANUAL OVERRIDE COMPLETE ---


#### 4.3 — Manual override: OF02649/OF02650

Targeted fix for a specific offer-pair mismatch.

In [85]:
print("--- Initiating Manual Override Patch B56 (Data Lake B) ---")
try:
    db_path = '/workspaces/pienza/data/big_bang/pienza.db'
    if not os.path.exists(db_path):
        raise FileNotFoundError(f"Database file not found at path: {db_path}")
    db_engine = create_engine(f'sqlite:///{db_path}')
    print("Database engine created successfully.")
except Exception as e:
    print(f"CRITICAL ERROR: Database engine failed. Details: {e}")
    raise

# Ground Truth from Architect Audit
TARGET_TRIP_ID = '250917-10'
CORRECT_OFFER_ID = 'OF02650'

print("\n⏳ Reading current state of 'trip_events' table...")
try:
    trip_events_df = pd.read_sql('SELECT * FROM trip_events', db_engine)
    print(f"Successfully read {len(trip_events_df)} records.")
    print("\n--- VERIFICATION (BEFORE PATCH) ---")
    display(trip_events_df[trip_events_df['trip_id_legacy'] == TARGET_TRIP_ID][['trip_id_legacy', 'event_types_id_fk', 'offer_id_fk']])
except Exception as e:
    print(f"ERROR: Could not read from trip_events table. Details: {e}")
    raise

print(f"\n⏳ Applying manual override to trip '{TARGET_TRIP_ID}' in memory...")
target_mask = trip_events_df['trip_id_legacy'] == TARGET_TRIP_ID

trip_events_df.loc[target_mask, 'offer_id_fk'] = np.nan

t1_mask = (trip_events_df['trip_id_legacy'] == TARGET_TRIP_ID) & (trip_events_df['event_types_id_fk'] == 2)
trip_events_df.loc[t1_mask, 'offer_id_fk'] = CORRECT_OFFER_ID
print(f"Rematched T1 event to correct offer '{CORRECT_OFFER_ID}'.")

print("\n⏳ Re-running Golden Propagation to ensure lifecycle consistency...")
trip_events_df = trip_events_df.sort_values(by=['trip_id_legacy', 'event_timestamp'])
trip_events_df['offer_id_fk'] = trip_events_df.groupby('trip_id_legacy')['offer_id_fk'].transform(lambda x: x.ffill().bfill())
print("Propagation complete in memory.")

print("\n⏳ Writing patched data back to pienza.db...")
try:
    trip_events_df.to_sql('trip_events', db_engine, if_exists='replace', index=False)
    db_engine.dispose()
    print("   - Data written. Pausing for 5 seconds for sync...")
    time.sleep(5)
    print("Write operation complete.")
except Exception as e:
    print(f"ERROR: Failed to write patched data. Details: {e}")
    raise

print("\n--- VERIFICATION (AFTER PATCH) ---")
try:
    verify_engine = create_engine(f'sqlite:///{db_path}')
    verify_df = pd.read_sql_query(f"SELECT * FROM trip_events WHERE trip_id_legacy = '{TARGET_TRIP_ID}'", verify_engine)
    verify_engine.dispose()
    display(verify_df[['trip_id_legacy', 'event_types_id_fk', 'offer_id_fk']])
    assert (verify_df['offer_id_fk'] == CORRECT_OFFER_ID).all(), "Verification Failed: Mismatch after re-reading!"
    print("VERIFICATION SUCCESS: Changes persisted to pienza.db.")
except Exception as e:
    print(f"VERIFICATION FAILED: {e}")

print("\n--- MANUAL OVERRIDE COMPLETE ---")

--- Initiating Manual Override Patch B56 (Data Lake B) ---
Database engine created successfully.

⏳ Reading current state of 'trip_events' table...
Successfully read 1031 records.

--- VERIFICATION (BEFORE PATCH) ---


,trip_id_legacy,event_types_id_fk,offer_id_fk
547,250917-10,1,OF02648
548,250917-10,2,OF02648
549,250917-10,3,OF02648
550,250917-10,4,OF02648
551,250917-10,5,OF02648



⏳ Applying manual override to trip '250917-10' in memory...
Rematched T1 event to correct offer 'OF02650'.

⏳ Re-running Golden Propagation to ensure lifecycle consistency...
Propagation complete in memory.

⏳ Writing patched data back to pienza.db...
   - Data written. Pausing for 5 seconds for sync...


Write operation complete.

--- VERIFICATION (AFTER PATCH) ---


,trip_id_legacy,event_types_id_fk,offer_id_fk
0,250917-10,1,OF02650
1,250917-10,2,OF02650
2,250917-10,3,OF02650
3,250917-10,4,OF02650
4,250917-10,5,OF02650


VERIFICATION SUCCESS: Changes persisted to pienza.db.

--- MANUAL OVERRIDE COMPLETE ---


#### 4.4 — Manual override: OF03759/OF03780

Targeted fix for a specific offer-pair mismatch.

In [86]:
print("--- Initiating Manual Override Patch B57 (Data Lake B) ---")
try:
    db_path = '/workspaces/pienza/data/big_bang/pienza.db'
    if not os.path.exists(db_path):
        raise FileNotFoundError(f"Database file not found at path: {db_path}")
    db_engine = create_engine(f'sqlite:///{db_path}')
    print("Database engine created successfully.")
except Exception as e:
    print(f"CRITICAL ERROR: Database engine failed. Details: {e}")
    raise

# Ground Truth from Architect Audit
TARGET_TRIP_ID = '250924-04'
CORRECT_OFFER_ID = 'OF03780'

print("\n⏳ Reading current state of 'trip_events' table...")
try:
    trip_events_df = pd.read_sql('SELECT * FROM trip_events', db_engine)
    print(f"Successfully read {len(trip_events_df)} records.")
    print("\n--- VERIFICATION (BEFORE PATCH) ---")
    display(trip_events_df[trip_events_df['trip_id_legacy'] == TARGET_TRIP_ID][['trip_id_legacy', 'event_types_id_fk', 'offer_id_fk']])
except Exception as e:
    print(f"ERROR: Could not read from trip_events table. Details: {e}")
    raise

print(f"\n⏳ Applying manual override to trip '{TARGET_TRIP_ID}' in memory...")
target_mask = trip_events_df['trip_id_legacy'] == TARGET_TRIP_ID

trip_events_df.loc[target_mask, 'offer_id_fk'] = np.nan

t1_mask = (trip_events_df['trip_id_legacy'] == TARGET_TRIP_ID) & (trip_events_df['event_types_id_fk'] == 2)
trip_events_df.loc[t1_mask, 'offer_id_fk'] = CORRECT_OFFER_ID
print(f"Rematched T1 event to correct offer '{CORRECT_OFFER_ID}'.")

print("\n⏳ Re-running Golden Propagation to ensure lifecycle consistency...")
trip_events_df = trip_events_df.sort_values(by=['trip_id_legacy', 'event_timestamp'])
trip_events_df['offer_id_fk'] = trip_events_df.groupby('trip_id_legacy')['offer_id_fk'].transform(lambda x: x.ffill().bfill())
print("Propagation complete in memory.")

print("\n⏳ Writing patched data back to pienza.db...")
try:
    trip_events_df.to_sql('trip_events', db_engine, if_exists='replace', index=False)
    db_engine.dispose()
    print("   - Data written. Pausing for 5 seconds for sync...")
    time.sleep(5)
    print("Write operation complete.")
except Exception as e:
    print(f"ERROR: Failed to write patched data. Details: {e}")
    raise

print("\n--- VERIFICATION (AFTER PATCH) ---")
try:
    verify_engine = create_engine(f'sqlite:///{db_path}')
    verify_df = pd.read_sql_query(f"SELECT * FROM trip_events WHERE trip_id_legacy = '{TARGET_TRIP_ID}'", verify_engine)
    verify_engine.dispose()
    display(verify_df[['trip_id_legacy', 'event_types_id_fk', 'offer_id_fk']])
    assert (verify_df['offer_id_fk'] == CORRECT_OFFER_ID).all(), "Verification Failed: Mismatch after re-reading!"
    print("VERIFICATION SUCCESS: Changes persisted to pienza.db.")
except Exception as e:
    print(f"VERIFICATION FAILED: {e}")

print("\n--- MANUAL OVERRIDE COMPLETE ---")

--- Initiating Manual Override Patch B57 (Data Lake B) ---
Database engine created successfully.

⏳ Reading current state of 'trip_events' table...
Successfully read 1031 records.

--- VERIFICATION (BEFORE PATCH) ---


,trip_id_legacy,event_types_id_fk,offer_id_fk
788,250924-04,2,OF03758
789,250924-04,3,OF03758
790,250924-04,4,OF03758
791,250924-04,5,OF03758



⏳ Applying manual override to trip '250924-04' in memory...
Rematched T1 event to correct offer 'OF03780'.

⏳ Re-running Golden Propagation to ensure lifecycle consistency...
Propagation complete in memory.

⏳ Writing patched data back to pienza.db...
   - Data written. Pausing for 5 seconds for sync...
Write operation complete.

--- VERIFICATION (AFTER PATCH) ---


,trip_id_legacy,event_types_id_fk,offer_id_fk
0,250924-04,2,OF03780
1,250924-04,3,OF03780
2,250924-04,4,OF03780
3,250924-04,5,OF03780


VERIFICATION SUCCESS: Changes persisted to pienza.db.

--- MANUAL OVERRIDE COMPLETE ---


#### 4.5 — Manual override: OF04729/OF04731

Targeted fix for a specific offer-pair mismatch.

In [87]:
print("--- Initiating Manual Override Patch B58 (Data Lake B) ---")
try:
    db_path = '/workspaces/pienza/data/big_bang/pienza.db'
    if not os.path.exists(db_path):
        raise FileNotFoundError(f"Database file not found at path: {db_path}")
    db_engine = create_engine(f'sqlite:///{db_path}')
    print("Database engine created successfully.")
except Exception as e:
    print(f"CRITICAL ERROR: Database engine failed. Details: {e}")
    raise

# Define the ground truth based on the Architect's audit
TARGET_TRIP_ID = '251001-02'
CORRECT_OFFER_ID = 'OF04733'

print("\n⏳ Reading current state of 'trip_events' table...")
try:
    trip_events_df = pd.read_sql('SELECT * FROM trip_events', db_engine)
    print(f"Successfully read {len(trip_events_df)} records.")
    print("\n--- VERIFICATION (BEFORE PATCH) ---")
    display(trip_events_df[trip_events_df['trip_id_legacy'] == TARGET_TRIP_ID][['trip_id_legacy', 'event_types_id_fk', 'offer_id_fk']])
except Exception as e:
    print(f"ERROR: Could not read from trip_events table. Details: {e}")
    raise

print(f"\n⏳ Applying manual override to trip '{TARGET_TRIP_ID}' in memory...")
target_mask = trip_events_df['trip_id_legacy'] == TARGET_TRIP_ID

trip_events_df.loc[target_mask, 'offer_id_fk'] = np.nan

t1_mask = (trip_events_df['trip_id_legacy'] == TARGET_TRIP_ID) & (trip_events_df['event_types_id_fk'] == 2)
trip_events_df.loc[t1_mask, 'offer_id_fk'] = CORRECT_OFFER_ID
print(f"Rematched T1 event to correct offer '{CORRECT_OFFER_ID}'.")

print("\n⏳ Re-running Golden Propagation to ensure lifecycle consistency...")
trip_events_df = trip_events_df.sort_values(by=['trip_id_legacy', 'event_timestamp'])
trip_events_df['offer_id_fk'] = trip_events_df.groupby('trip_id_legacy')['offer_id_fk'].transform(lambda x: x.ffill().bfill())
print("Propagation complete in memory.")

print("\n⏳ Writing patched data back to pienza.db...")
try:
    trip_events_df.to_sql('trip_events', db_engine, if_exists='replace', index=False)
    db_engine.dispose()
    print("   - Data written. Pausing for 5 seconds for sync...")
    time.sleep(5)
    print("Write operation complete.")
except Exception as e:
    print(f"ERROR: Failed to write patched data. Details: {e}")
    raise

print("\n--- VERIFICATION (AFTER PATCH) ---")
try:
    verify_engine = create_engine(f'sqlite:///{db_path}')
    verify_df = pd.read_sql_query(f"SELECT * FROM trip_events WHERE trip_id_legacy = '{TARGET_TRIP_ID}'", verify_engine)
    verify_engine.dispose()
    display(verify_df[['trip_id_legacy', 'event_types_id_fk', 'offer_id_fk']])
    assert (verify_df['offer_id_fk'] == CORRECT_OFFER_ID).all(), "Verification Failed: Mismatch after re-reading!"
    print("VERIFICATION SUCCESS: Changes persisted to pienza.db.")
except Exception as e:
    print(f"VERIFICATION FAILED: {e}")

print("\n--- MANUAL OVERRIDE COMPLETE ---")

--- Initiating Manual Override Patch B58 (Data Lake B) ---
Database engine created successfully.

⏳ Reading current state of 'trip_events' table...
Successfully read 1031 records.

--- VERIFICATION (BEFORE PATCH) ---


,trip_id_legacy,event_types_id_fk,offer_id_fk
1005,251001-02,1,OF04728
1006,251001-02,2,OF04728
1007,251001-02,3,OF04728
1008,251001-02,4,OF04728
1009,251001-02,5,OF04728



⏳ Applying manual override to trip '251001-02' in memory...
Rematched T1 event to correct offer 'OF04733'.

⏳ Re-running Golden Propagation to ensure lifecycle consistency...
Propagation complete in memory.

⏳ Writing patched data back to pienza.db...
   - Data written. Pausing for 5 seconds for sync...


Write operation complete.

--- VERIFICATION (AFTER PATCH) ---


,trip_id_legacy,event_types_id_fk,offer_id_fk
0,251001-02,1,OF04733
1,251001-02,2,OF04733
2,251001-02,3,OF04733
3,251001-02,4,OF04733
4,251001-02,5,OF04733


VERIFICATION SUCCESS: Changes persisted to pienza.db.

--- MANUAL OVERRIDE COMPLETE ---


#### 4.6 — Manual override: OF04649/OF04565

Targeted fix for a specific offer-pair mismatch.

In [88]:
print("--- Initiating CORRECTED Manual Override Patch B59 (Data Lake B) ---")
try:
    db_path = '/workspaces/pienza/data/big_bang/pienza.db'
    if not os.path.exists(db_path):
        raise FileNotFoundError(f"Database file not found at path: {db_path}")
    db_engine = create_engine(f'sqlite:///{db_path}')
    print("Database engine created successfully.")
except Exception as e:
    print(f"CRITICAL ERROR: Database engine failed. Details: {e}")
    raise

# Ground Truth from Architect Audit
TARGET_TRIP_ID = '250930-04'
CORRECT_OFFER_ID = 'OF04565'

print("\n⏳ Reading current state of 'trip_events' table...")
try:
    trip_events_df = pd.read_sql('SELECT * FROM trip_events', db_engine)
    print(f"Successfully read {len(trip_events_df)} records.")
    print("\n--- VERIFICATION (BEFORE PATCH) ---")
    display(trip_events_df[trip_events_df['trip_id_legacy'] == TARGET_TRIP_ID][['trip_id_legacy', 'event_types_id_fk', 'offer_id_fk']])
except Exception as e:
    print(f"ERROR: Could not read from trip_events table. Details: {e}")
    raise

print(f"\n⏳ Applying manual override to trip '{TARGET_TRIP_ID}' in memory...")
target_mask = trip_events_df['trip_id_legacy'] == TARGET_TRIP_ID

trip_events_df.loc[target_mask, 'offer_id_fk'] = np.nan

t1_mask = (trip_events_df['trip_id_legacy'] == TARGET_TRIP_ID) & (trip_events_df['event_types_id_fk'] == 2)
trip_events_df.loc[t1_mask, 'offer_id_fk'] = CORRECT_OFFER_ID
print(f"Rematched T1 event to correct offer '{CORRECT_OFFER_ID}'.")

print("\n⏳ Re-running Golden Propagation in memory...")
trip_events_df = trip_events_df.sort_values(by=['trip_id_legacy', 'event_timestamp'])
trip_events_df['offer_id_fk'] = trip_events_df.groupby('trip_id_legacy')['offer_id_fk'].transform(lambda x: x.ffill().bfill())
print("Propagation complete in memory.")

print("\n⏳ Writing patched data back to pienza.db...")
try:
    trip_events_df.to_sql('trip_events', db_engine, if_exists='replace', index=False)
    db_engine.dispose()
    print("   - Data written. Pausing for 5 seconds for sync...")
    time.sleep(5)
    print("Write operation complete.")
except Exception as e:
    print(f"ERROR: Failed to write patched data. Details: {e}")
    raise

print("\n--- VERIFICATION (AFTER PATCH) ---")
try:
    verify_engine = create_engine(f'sqlite:///{db_path}')
    verify_df = pd.read_sql_query(f"SELECT * FROM trip_events WHERE trip_id_legacy = '{TARGET_TRIP_ID}'", verify_engine)
    verify_engine.dispose()
    display(verify_df[['trip_id_legacy', 'event_types_id_fk', 'offer_id_fk']])
    assert (verify_df['offer_id_fk'] == CORRECT_OFFER_ID).all(), "Verification Failed: Mismatch after re-reading!"
    print("VERIFICATION SUCCESS: Changes persisted to pienza.db.")
except Exception as e:
    print(f"VERIFICATION FAILED: {e}")

print("\n--- MANUAL OVERRIDE COMPLETE ---")

--- Initiating CORRECTED Manual Override Patch B59 (Data Lake B) ---
Database engine created successfully.

⏳ Reading current state of 'trip_events' table...
Successfully read 1031 records.

--- VERIFICATION (BEFORE PATCH) ---


,trip_id_legacy,event_types_id_fk,offer_id_fk
972,250930-04,1,OF04564
973,250930-04,2,OF04564



⏳ Applying manual override to trip '250930-04' in memory...
Rematched T1 event to correct offer 'OF04565'.

⏳ Re-running Golden Propagation in memory...
Propagation complete in memory.

⏳ Writing patched data back to pienza.db...
   - Data written. Pausing for 5 seconds for sync...
Write operation complete.

--- VERIFICATION (AFTER PATCH) ---


,trip_id_legacy,event_types_id_fk,offer_id_fk
0,250930-04,1,OF04565
1,250930-04,2,OF04565


VERIFICATION SUCCESS: Changes persisted to pienza.db.

--- MANUAL OVERRIDE COMPLETE ---


#### 4.7 — Manual override: OF03904/OF04649

Targeted fix for a specific offer-pair mismatch.

In [89]:
print("--- Initiating Manual Override Patch B60 (Data Lake B) ---")
try:
    db_path = '/workspaces/pienza/data/big_bang/pienza.db'
    if not os.path.exists(db_path):
        raise FileNotFoundError(f"Database file not found at path: {db_path}")
    db_engine = create_engine(f'sqlite:///{db_path}')
    print("Database engine created successfully.")
except Exception as e:
    print(f"CRITICAL ERROR: Database engine failed. Details: {e}")
    raise

# Ground Truth from Architect Audit
TARGET_TRIP_ID = '250930-09'
CORRECT_OFFER_ID = 'OF04649'

print("\n⏳ Reading current state of 'trip_events' table...")
try:
    trip_events_df = pd.read_sql('SELECT * FROM trip_events', db_engine)
    print(f"Successfully read {len(trip_events_df)} records.")
    print("\n--- VERIFICATION (BEFORE PATCH) ---")
    display(trip_events_df[trip_events_df['trip_id_legacy'] == TARGET_TRIP_ID][['trip_id_legacy', 'event_types_id_fk', 'offer_id_fk']])
except Exception as e:
    print(f"ERROR: Could not read from trip_events table. Details: {e}")
    raise

print(f"\n⏳ Applying manual override to trip '{TARGET_TRIP_ID}' in memory...")
target_mask = trip_events_df['trip_id_legacy'] == TARGET_TRIP_ID

trip_events_df.loc[target_mask, 'offer_id_fk'] = np.nan

t1_mask = (trip_events_df['trip_id_legacy'] == TARGET_TRIP_ID) & (trip_events_df['event_types_id_fk'] == 2)
trip_events_df.loc[t1_mask, 'offer_id_fk'] = CORRECT_OFFER_ID
print(f"Rematched T1 event to correct offer '{CORRECT_OFFER_ID}'.")

print("\n⏳ Re-running Golden Propagation in memory...")
trip_events_df = trip_events_df.sort_values(by=['trip_id_legacy', 'event_timestamp'])
trip_events_df['offer_id_fk'] = trip_events_df.groupby('trip_id_legacy')['offer_id_fk'].transform(lambda x: x.ffill().bfill())
print("Propagation complete in memory.")

print("\n⏳ Writing patched data back to pienza.db...")
try:
    trip_events_df.to_sql('trip_events', db_engine, if_exists='replace', index=False)
    db_engine.dispose()
    print("   - Data written. Pausing for 5 seconds for sync...")
    time.sleep(5)
    print("Write operation complete.")
except Exception as e:
    print(f"ERROR: Failed to write patched data. Details: {e}")
    raise

print("\n--- VERIFICATION (AFTER PATCH) ---")
try:
    verify_engine = create_engine(f'sqlite:///{db_path}')
    verify_df = pd.read_sql_query(f"SELECT * FROM trip_events WHERE trip_id_legacy = '{TARGET_TRIP_ID}'", verify_engine)
    verify_engine.dispose()
    display(verify_df[['trip_id_legacy', 'event_types_id_fk', 'offer_id_fk']])
    assert (verify_df['offer_id_fk'] == CORRECT_OFFER_ID).all(), "Verification Failed: Mismatch after re-reading!"
    print("VERIFICATION SUCCESS: Changes persisted to pienza.db.")
except Exception as e:
    print(f"VERIFICATION FAILED: {e}")

print("\n--- MANUAL OVERRIDE COMPLETE ---")

--- Initiating Manual Override Patch B60 (Data Lake B) ---
Database engine created successfully.

⏳ Reading current state of 'trip_events' table...
Successfully read 1031 records.

--- VERIFICATION (BEFORE PATCH) ---


,trip_id_legacy,event_types_id_fk,offer_id_fk
993,250930-09,2,OF03903
994,250930-09,4,OF03903
995,250930-09,5,OF03903



⏳ Applying manual override to trip '250930-09' in memory...
Rematched T1 event to correct offer 'OF04649'.

⏳ Re-running Golden Propagation in memory...
Propagation complete in memory.

⏳ Writing patched data back to pienza.db...
   - Data written. Pausing for 5 seconds for sync...
Write operation complete.

--- VERIFICATION (AFTER PATCH) ---


,trip_id_legacy,event_types_id_fk,offer_id_fk
0,250930-09,2,OF04649
1,250930-09,4,OF04649
2,250930-09,5,OF04649


VERIFICATION SUCCESS: Changes persisted to pienza.db.

--- MANUAL OVERRIDE COMPLETE ---


#### 4.8 — Batch override for remaining cases

Applies the remaining known corrections in a single batch pass.

In [90]:
print("--- Initiating Final Patch: Batch Override (Data Lake B) ---")
try:
    db_path = '/workspaces/pienza/data/big_bang/pienza.db'
    if not os.path.exists(db_path):
        raise FileNotFoundError(f"Database file not found at path: {db_path}")
    db_engine = create_engine(f'sqlite:///{db_path}')
    print("Database engine created successfully.")
except Exception as e:
    print(f"CRITICAL ERROR: Database engine failed. Details: {e}")
    raise

# The definitive list of corrections
GOLDEN_EDICT = [
    ('250911-10', 'OF02170', 'OF02169'),
    ('250911-09', 'OF02160', 'OF02159'),
    ('250904-05', 'OF01667', 'OF01666'),
    ('250902-02', 'OF01290', 'OF01289'),
    ('250901-10', 'OF01280', 'OF01279')
]
target_trip_ids = [item[0] for item in GOLDEN_EDICT]

print("\n⏳ Reading current state of 'trip_events' table...")
try:
    trip_events_df = pd.read_sql('SELECT * FROM trip_events', db_engine)
    print(f"Successfully read {len(trip_events_df)} records.")
    print("\n--- VERIFICATION (BEFORE PATCH) ---")
    display(trip_events_df[trip_events_df['trip_id_legacy'].isin(target_trip_ids)][['trip_id_legacy', 'event_types_id_fk', 'offer_id_fk']])
except Exception as e:
    print(f"ERROR: Could not read from trip_events. Details: {e}")
    raise

print(f"\n⏳ Applying {len(GOLDEN_EDICT)} manual overrides in memory...")
for trip_id, incorrect_id, correct_id in GOLDEN_EDICT:
    target_mask = trip_events_df['trip_id_legacy'] == trip_id
    trip_events_df.loc[target_mask, 'offer_id_fk'] = np.nan
    t1_mask = (trip_events_df['trip_id_legacy'] == trip_id) & (trip_events_df['event_types_id_fk'] == 2)
    trip_events_df.loc[t1_mask, 'offer_id_fk'] = correct_id
print("All overrides applied in memory.")

print("\n⏳ Re-running Golden Propagation for all patched trips...")
trip_events_df = trip_events_df.sort_values(by=['trip_id_legacy', 'event_timestamp'])
trip_events_df['offer_id_fk'] = trip_events_df.groupby('trip_id_legacy')['offer_id_fk'].transform(lambda x: x.ffill().bfill())
print("Propagation complete in memory.")

print("\n⏳ Writing final patched data back to pienza.db...")
try:
    trip_events_df.to_sql('trip_events', db_engine, if_exists='replace', index=False)
    db_engine.dispose()
    print("   - Data written. Pausing for 5 seconds for sync...")
    time.sleep(5)
    print("Write operation complete.")
except Exception as e:
    print(f"ERROR: Failed to write patched data. Details: {e}")
    raise

print("\n--- VERIFICATION (AFTER PATCH) ---")
try:
    verify_engine = create_engine(f'sqlite:///{db_path}')
    trip_tuple = tuple(target_trip_ids) if len(target_trip_ids) > 1 else f"('{target_trip_ids[0]}')"
    verify_df = pd.read_sql_query(f"SELECT * FROM trip_events WHERE trip_id_legacy IN {trip_tuple}", verify_engine)
    verify_engine.dispose()

    display(verify_df[['trip_id_legacy', 'event_types_id_fk', 'offer_id_fk']].sort_values(by='trip_id_legacy'))

    for trip_id, _, correct_id in GOLDEN_EDICT:
        assert (verify_df[verify_df['trip_id_legacy'] == trip_id]['offer_id_fk'] == correct_id).all(), f"Verification Failed for {trip_id}!"
    print("VERIFICATION SUCCESS: All changes persisted to pienza.db.")
except Exception as e:
    print(f"VERIFICATION FAILED: {e}")

print("\n--- BATCH MANUAL OVERRIDE COMPLETE ---")

--- Initiating Final Patch: Batch Override (Data Lake B) ---
Database engine created successfully.

⏳ Reading current state of 'trip_events' table...
Successfully read 1031 records.

--- VERIFICATION (BEFORE PATCH) ---


,trip_id_legacy,event_types_id_fk,offer_id_fk
252,250901-10,2,OF01279
253,250901-10,3,OF01279
254,250901-10,4,OF01279
255,250901-10,5,OF01279
261,250902-02,2,OF01289
262,250902-02,3,OF01289
263,250902-02,4,OF01289
264,250902-02,5,OF01289
346,250904-05,1,OF01666
347,250904-05,2,OF01666



⏳ Applying 5 manual overrides in memory...
All overrides applied in memory.

⏳ Re-running Golden Propagation for all patched trips...
Propagation complete in memory.

⏳ Writing final patched data back to pienza.db...
   - Data written. Pausing for 5 seconds for sync...
Write operation complete.

--- VERIFICATION (AFTER PATCH) ---


,trip_id_legacy,event_types_id_fk,offer_id_fk
0,250901-10,2,OF01279
1,250901-10,3,OF01279
2,250901-10,4,OF01279
3,250901-10,5,OF01279
4,250902-02,2,OF01289
5,250902-02,3,OF01289
6,250902-02,4,OF01289
7,250902-02,5,OF01289
8,250904-05,1,OF01666
9,250904-05,2,OF01666


VERIFICATION SUCCESS: All changes persisted to pienza.db.

--- BATCH MANUAL OVERRIDE COMPLETE ---


## Phase 5 — Engineered features and consolidated views

Ingests the external "Consomaster" reconciliation sheet, forges the `engineered_features` table (with a full type-hygiene audit), applies two linking patches (OCR deterministic linker, "twin link" protocol), then builds four consolidated analytical views tracing data lineage from raw OCR through to bank earnings.

#### 5.1 — Consomaster bridge

Ingests the external reconciliation spreadsheet used to build engineered features.

In [91]:
print("--- Initiating Consomaster Bridge Protocol B62 (Data Lake B) ---")

if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/workspaces/pienza'
    db_path = os.path.join(project_root, 'data/big_bang/pienza.db')
else:
    db_path = db_file_path

db_engine = create_engine(f'sqlite:///{db_path}')

# Canonical filename updated per Architect instruction
CONSO_GCS_NAME = '260509_consomaster_sheet1.csv'

print(f"⏳ Ingesting Consomaster Bridge from GCS: {CONSO_GCS_NAME}...")
try:
    blob = bucket.blob(CONSO_GCS_NAME)
    content = blob.download_as_bytes()
    # na_filter=False keeps strings clean for the ID normalization
    conso_df = pd.read_csv(io.BytesIO(content), dtype=str, na_filter=False)

    bridge_df = conso_df[
        (conso_df['trip_id_legacy'] != '') & 
        (conso_df['lifetime_trips_id'] != '')
    ][['trip_id_legacy', 'lifetime_trips_id']].copy()

    # NORMALIZE IDs (Strip 'LT' and leading zeros)
    bridge_df['lifetime_trips_id'] = bridge_df['lifetime_trips_id'].str.replace('LT', '').astype(float).astype(int).astype(str)

    print(f"Ingested bridge. Found {len(bridge_df)} valid link pairs.")
except Exception as e:
    print(f"ERROR: Failed to ingest/normalize GCS file. Details: {e}")
    raise

print("\n⏳ Fetching 'offer_id_fk' source data from 'trip_events'...")
try:
    source_links_df = pd.read_sql(
        "SELECT DISTINCT trip_id_legacy, offer_id_fk FROM trip_events WHERE offer_id_fk IS NOT NULL", 
        db_engine
    )
    source_links_df['trip_id_legacy'] = source_links_df['trip_id_legacy'].astype(str)
    print(f"Loaded {len(source_links_df)} source links from trip_events.")
except Exception as e:
    print(f"ERROR: Failed to read from trip_events. Details: {e}")
    raise

print("\n⏳ Constructing the Golden Link map...")
merged_map = pd.merge(bridge_df, source_links_df, on='trip_id_legacy', how='inner')

lifetime_to_offer_map = pd.Series(
    merged_map.offer_id_fk.values, 
    index=merged_map.lifetime_trips_id
).to_dict()

print(f"Bridge constructed. Mapped {len(lifetime_to_offer_map)} lifetime trips to offers.")

print(f"\n⏳ Updating 'lifetime_trips' table in {os.path.basename(db_path)}...")
try:
    lifetime_trips_df = pd.read_sql("SELECT * FROM lifetime_trips", db_engine)
    lifetime_trips_df['offer_id_fk'] = lifetime_trips_df['lifetime_trips_id'].astype(str).map(lifetime_to_offer_map)

    updated_count = lifetime_trips_df['offer_id_fk'].notna().sum()
    print(f"   - Rows updated with Offer IDs: {updated_count}")

    if updated_count > 0:
        lifetime_trips_df.to_sql('lifetime_trips', db_engine, if_exists='replace', index=False)
        db_engine.dispose() 
        print("   - Data written. Connection closed.")
        time.sleep(5)
        print("SUCCESS: 'lifetime_trips' table updated.")
    else:
        print("WARNING: No rows were updated. Check ID normalization parity.")
except Exception as e:
    print(f"ERROR: Failed to update database. Details: {e}")
    raise

print("\n⏳ Performing Read-Back Verification...")
try:
    verify_engine = create_engine(f'sqlite:///{db_path}')
    verify_count = pd.read_sql("SELECT COUNT(*) FROM lifetime_trips WHERE offer_id_fk IS NOT NULL", verify_engine).iloc[0,0]
    verify_engine.dispose()
    print(f"Verified on Disk: {verify_count} rows now have an offer_id_fk.")
    if verify_count > 0:
        print("VERIFICATION PASSED.")
    else:
        print("VERIFICATION FAILED: Link count is zero.")
except Exception as e:
    print(f"ERROR during verification: {e}")

print("\n--- BRIDGE PROTOCOL COMPLETE ---")

--- Initiating Consomaster Bridge Protocol B62 (Data Lake B) ---
⏳ Ingesting Consomaster Bridge from GCS: 260509_consomaster_sheet1.csv...
Ingested bridge. Found 254 valid link pairs.

⏳ Fetching 'offer_id_fk' source data from 'trip_events'...
Loaded 259 source links from trip_events.

⏳ Constructing the Golden Link map...
Bridge constructed. Mapped 254 lifetime trips to offers.

⏳ Updating 'lifetime_trips' table in pienza.db...
   - Rows updated with Offer IDs: 254
   - Data written. Connection closed.
SUCCESS: 'lifetime_trips' table updated.

⏳ Performing Read-Back Verification...
Verified on Disk: 254 rows now have an offer_id_fk.
VERIFICATION PASSED.

--- BRIDGE PROTOCOL COMPLETE ---


#### 5.2 — Analytical layer embedding

Prepares the view layer that the engineered-features step reads from.

In [92]:
print("--- Initiating View Embedding Protocol (Data Lake B) ---")

if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/workspaces/pienza'
    db_path = os.path.join(project_root, 'data/big_bang/pienza.db')
else:
    db_path = db_file_path

db_engine = create_engine(f'sqlite:///{db_path}')

# Define the SQL commands to Drop (cleanup) and Create (build).
view_commands = [
    "DROP VIEW IF EXISTS v_mission_dossier;",
    "DROP VIEW IF EXISTS v_trip_final_kpis;",
    "DROP VIEW IF EXISTS v_trip_funnel_wide;",
    "DROP VIEW IF EXISTS v_reconciled_offer;",

    # 2. BUILD BASE: v_trip_funnel_wide (Pivots events into columns)
    """
    CREATE VIEW v_trip_funnel_wide AS
    SELECT
        trip_id_legacy,
        MAX(offer_id_fk) AS offer_id_fk,
        MAX(CASE WHEN event_types_id_fk = 1 THEN event_timestamp END) AS t0_timestamp,
        MAX(CASE WHEN event_types_id_fk = 2 THEN event_timestamp END) AS t1_timestamp,
        MAX(CASE WHEN event_types_id_fk = 3 THEN event_timestamp END) AS t2_timestamp,
        MAX(CASE WHEN event_types_id_fk = 4 THEN event_timestamp END) AS t3_timestamp,
        MAX(CASE WHEN event_types_id_fk = 5 THEN event_timestamp END) AS t4_timestamp,
        MAX(CASE WHEN event_types_id_fk = 2 THEN upfront_fare END) AS upfront_fare,
        MAX(CASE WHEN event_types_id_fk = 5 THEN realized_fare END) AS realized_fare
    FROM trip_events
    GROUP BY trip_id_legacy;
    """,

    # 3. BUILD LOGIC: v_trip_final_kpis (Calculates durations and spreads)
    """
    CREATE VIEW v_trip_final_kpis AS
    SELECT
        v.trip_id_legacy,
        DATE(v.t1_timestamp) AS trip_date,
        (julianday(v.t2_timestamp) - julianday(v.t1_timestamp)) * 86400.0 AS duration_to_pickup_sec,
        (julianday(v.t3_timestamp) - julianday(v.t2_timestamp)) * 86400.0 AS duration_waiting_sec,
        (julianday(v.t4_timestamp) - julianday(v.t3_timestamp)) * 86400.0 AS duration_trip_sec,
        (julianday(v.t4_timestamp) - julianday(v.t1_timestamp)) * 86400.0 AS total_engagement_duration_sec,
        v.upfront_fare,
        v.realized_fare,
        CASE WHEN v.upfront_fare > 0 THEN v.realized_fare / v.upfront_fare ELSE NULL END AS spread_percentage,
        CASE WHEN (julianday(v.t4_timestamp) - julianday(v.t3_timestamp)) > 0 THEN v.realized_fare / (((julianday(v.t4_timestamp) - julianday(v.t3_timestamp)) * 86400.0) / 3600.0) ELSE NULL END AS eph_on_ride,
        CASE WHEN (julianday(v.t4_timestamp) - julianday(v.t1_timestamp)) > 0 THEN v.realized_fare / (((julianday(v.t4_timestamp) - julianday(v.t1_timestamp)) * 86400.0) / 3600.0) ELSE NULL END AS eph_total_time
    FROM v_trip_funnel_wide v;
    """,

    # 4. BUILD PRODUCT: v_mission_dossier (The Golden Link View)
    """
    CREATE VIEW v_mission_dossier AS
    WITH OfferLink AS (
        SELECT
            trip_id_legacy,
            MAX(offer_id_fk) AS offer_id
        FROM
            trip_events
        GROUP BY
            trip_id_legacy
    )
    SELECT
        ol.offer_id,
        kpi.*
    FROM
        v_trip_final_kpis AS kpi
    LEFT JOIN
        OfferLink AS ol ON kpi.trip_id_legacy = ol.trip_id_legacy;
    """
]

# EXECUTION LOOP
print(f"⏳ Embedding views into {os.path.basename(db_path)}...")
try:
    with db_engine.connect() as connection:
        for cmd in view_commands:
            connection.execute(text(cmd))
        connection.commit()
    print("SUCCESS: All Analytical Views permanently embedded in pienza.db.")

except Exception as e:
    print(f"ERROR: Failed to embed views. Details: {e}")
    raise

print("--- ANALYTICAL LAYER READY ---")

--- Initiating View Embedding Protocol (Data Lake B) ---
⏳ Embedding views into pienza.db...
SUCCESS: All Analytical Views permanently embedded in pienza.db.
--- ANALYTICAL LAYER READY ---


#### 5.3 — Forge engineered_features

Builds the `engineered_features` table, with a full type-hygiene audit.

In [93]:
print("--- Initiating engineered_features materialization (Data Lake B) ---")

if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/workspaces/pienza'
    db_path = os.path.join(project_root, 'data/big_bang/pienza.db')
else:
    db_path = db_file_path

db_engine = create_engine(f'sqlite:///{db_path}')

CONSO_GCS_NAME = '260509_engineered_features_sheet2.csv'

print(f"⏳ Ingesting features from GCS: {CONSO_GCS_NAME}...")
try:
    blob = bucket.blob(CONSO_GCS_NAME)
    content = blob.download_as_bytes()
    features_df = pd.read_csv(io.BytesIO(content), dtype=str, na_filter=False)
    features_df.replace('', np.nan, inplace=True)
    print(f"Loaded {len(features_df)} rows of engineered features.")
except Exception as e:
    print(f"ERROR: Failed to ingest GCS file. Details: {e}")
    raise

print("\n⏳ Fetching canonical 'offer_id' keys from pienza.db...")
try:
    db_keys_df = pd.read_sql("SELECT offer_id FROM offers", db_engine)
    features_df['match_key'] = features_df['feature_id'].astype(str).str.extract(r'(\d+)').astype(int)
    db_keys_df['match_key'] = db_keys_df['offer_id'].astype(str).str.extract(r'(\d+)').astype(int)

    merged_df = pd.merge(features_df, db_keys_df[['offer_id', 'match_key']], on='match_key', how='left')
    merged_df['offer_id_fk'] = merged_df['offer_id']
    final_features_df = merged_df.drop(columns=['match_key', 'offer_id'])
except Exception as e:
    print(f"ERROR during ID matching. Details: {e}")
    raise

print("\n⏳ Enforcing data type hygiene (protecting text and booleans)...")

# A. Clean Currency
currency_cols = [col for col in final_features_df.columns if any(k in col for k in ['earnings', 'eph', 'fare'])]
for col in currency_cols:
    if final_features_df[col].dtype == 'object':
        final_features_df[col] = final_features_df[col].astype(str).str.replace(r'[$,]', '', regex=True)

# B. Protect Text Columns
text_keywords = ['_label', '_block', '_type', '_ambiguity', 'feature_id', 'offer_id_fk', 'day_shift', 'day_type', 'day_of_week']

# C. Convert Numerics with Protection
for col in final_features_df.columns:
    is_protected_text = any(keyword in col for keyword in text_keywords)
    is_boolean = 'is_' in col
    if not is_protected_text and not is_boolean:
        final_features_df[col] = pd.to_numeric(final_features_df[col], errors='coerce')

# D. Handle Booleans (1/0 for SQLite)
bool_cols = [col for col in final_features_df.columns if 'is_' in col]
for col in bool_cols:
    final_features_df[col] = final_features_df[col].astype(str).str.upper().map({
        'TRUE': 1, 'FALSE': 0, '1': 1, '0': 0, 'NAN': None, 'NONE': None
    })

print(f"\n⏳ Writing 'engineered_features' to {os.path.basename(db_path)}...")
try:
    final_features_df.to_sql('engineered_features', db_engine, if_exists='replace', index=False)
    
    verify_df = pd.read_sql("SELECT * FROM engineered_features", db_engine)
    cols_to_check = [c for c in ['day_of_week', 'is_total_cycle_downgrade_EDA', 'eph_realized_ML'] if c in verify_df.columns]
    display(verify_df[cols_to_check].tail(5))
    print(f"SUCCESS: Materialized {len(verify_df)} rows with {len(verify_df.columns)} features.")
except Exception as e:
    print(f"ERROR: Final write failed. Details: {e}")
    raise

print("\n--- FEATURE STORE COMPLETE ---")

--- Initiating engineered_features materialization (Data Lake B) ---
⏳ Ingesting features from GCS: 260509_engineered_features_sheet2.csv...
Loaded 4765 rows of engineered features.

⏳ Fetching canonical 'offer_id' keys from pienza.db...

⏳ Enforcing data type hygiene (protecting text and booleans)...

⏳ Writing 'engineered_features' to pienza.db...


,day_of_week,is_total_cycle_downgrade_EDA,eph_realized_ML
4760,Wednesday,1.0,None
4761,Wednesday,0.0,None
4762,Wednesday,1.0,None
4763,Wednesday,0.0,None
4764,Wednesday,1.0,None


SUCCESS: Materialized 4765 rows with 45 features.

--- FEATURE STORE COMPLETE ---


#### 5.4 — OCR deterministic linker patch (TD-006)

Fixes a tracked bug in how OCR records were linked to offers.

In [94]:
print("--- Initiating OCR Deterministic Linker Protocol (Data Lake B) ---")

if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/workspaces/pienza'
    db_path = os.path.join(project_root, 'data/big_bang/pienza.db')
else:
    db_path = db_file_path

db_engine = create_engine(f'sqlite:///{db_path}')

print(f"⏳ Loading tables from {os.path.basename(db_path)}...")
try:
    offers_df = pd.read_sql("SELECT * FROM offers", db_engine)
    ocr_df = pd.read_sql("SELECT ocr_id FROM raw_offers_ocr", db_engine)
    print(f"Loaded {len(offers_df)} Offers and {len(ocr_df)} OCR records.")
except Exception as e:
    print(f"ERROR: Failed to load tables. Details: {e}")
    raise

print("\n⏳ Executing Integer Extraction and mapping...")
offers_df['match_key'] = offers_df['offer_id'].astype(str).str.extract(r'(\d+)').astype(int)
ocr_df['match_key'] = ocr_df['ocr_id'].astype(str).str.extract(r'(\d+)').astype(int)

ocr_key_map = pd.Series(ocr_df.ocr_id.values, index=ocr_df.match_key).to_dict()

print("⏳ Applying deterministic links to 'offers.ocr_fk'...")
offers_df['ocr_fk'] = offers_df['match_key'].map(ocr_key_map)

linked_count = offers_df['ocr_fk'].notna().sum()
print(f"\nLinkage Report:")
print(f"   - Total Offers: {len(offers_df)}")
print(f"   - Linked to OCR: {linked_count}")
print(f"   - Gaps Detected: {len(offers_df) - linked_count}")

offers_df.drop(columns=['match_key'], inplace=True)

if linked_count > 0:
    try:
        offers_df.to_sql('offers', db_engine, if_exists='replace', index=False)
        print(f"SUCCESS: 'offers' updated in {os.path.basename(db_path)}.")
        display(offers_df[['offer_id', 'ocr_fk']].head(5))
    except Exception as e:
        print(f"ERROR: Database write failed. Details: {e}")
else:
    print("CRITICAL WARNING: No links were established.")

print("\n--- LINKER PROTOCOL COMPLETE ---")

--- Initiating OCR Deterministic Linker Protocol (Data Lake B) ---
⏳ Loading tables from pienza.db...
Loaded 4765 Offers and 4765 OCR records.

⏳ Executing Integer Extraction and mapping...
⏳ Applying deterministic links to 'offers.ocr_fk'...

Linkage Report:
   - Total Offers: 4765
   - Linked to OCR: 4765
   - Gaps Detected: 0
SUCCESS: 'offers' updated in pienza.db.


,offer_id,ocr_fk
0,OF00001,OCR00001
1,OF00002,OCR00002
2,OF00003,OCR00003
3,OF00004,OCR00004
4,OF00005,OCR00005



--- LINKER PROTOCOL COMPLETE ---


#### 5.5 — Twin-link protocol patch

Cleans up a duplicate-linking edge case found after the previous patch.

In [95]:
print("--- Initiating Earnings Reconciliation Protocol (Data Lake B) ---")

if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/workspaces/pienza'
    db_path = os.path.join(project_root, 'data/big_bang/pienza.db')
else:
    db_path = db_file_path

db_engine = create_engine(f'sqlite:///{db_path}')

try:
    print(f"⏳ Loading tables from {os.path.basename(db_path)}...")
    ae_df = pd.read_sql("SELECT * FROM activity_earnings", db_engine)
    lt_df = pd.read_sql("SELECT lifetime_trips_id, offer_id_fk FROM lifetime_trips", db_engine)

    ae_df = ae_df.drop(columns=['lifetime_trips_fk', 'offer_id_fk'], errors='ignore')

    print("⏳ Extracting numeric match keys...")
    ae_df['match_key'] = ae_df['activity_earnings_id'].astype(str).str.extract(r'(\d+)').astype(int)
    lt_df['match_key'] = lt_df['lifetime_trips_id'].astype(str).str.extract(r'(\d+)').astype(int)

    print("⏳ Linking tables based on numeric ID...")
    merged_df = pd.merge(
        ae_df,
        lt_df[['lifetime_trips_id', 'offer_id_fk', 'match_key']],
        on='match_key',
        how='left'
    )

    merged_df['lifetime_trips_fk'] = merged_df['lifetime_trips_id']

    final_ae_df = merged_df.drop(columns=['match_key', 'lifetime_trips_id'], errors='ignore')

    linked_lt = final_ae_df['lifetime_trips_fk'].notna().sum()
    linked_offer = final_ae_df['offer_id_fk'].notna().sum()

    print(f"\nLinkage Report:")
    print(f"   - Rows in Activity Earnings: {len(final_ae_df)}")
    print(f"   - Linked to Lifetime Trips:  {linked_lt}")
    print(f"   - Linked to Offers:          {linked_offer}")

    if linked_lt > 0:
        print(f"⏳ Saving updated 'activity_earnings' to {os.path.basename(db_path)}...")
        final_ae_df.to_sql('activity_earnings', db_engine, if_exists='replace', index=False)
        print("SUCCESS: 'activity_earnings' fully reconciled.")
        display(final_ae_df[['activity_earnings_id', 'lifetime_trips_fk', 'offer_id_fk']].head())
    else:
        print("WARNING: No links were created. Check ID formats.")

except Exception as e:
    print(f"ERROR: Failed to reconcile earnings. Details: {e}")

print("\n--- PROTOCOL COMPLETE ---")

--- Initiating Earnings Reconciliation Protocol (Data Lake B) ---
⏳ Loading tables from pienza.db...
⏳ Extracting numeric match keys...
⏳ Linking tables based on numeric ID...

Linkage Report:
   - Rows in Activity Earnings: 3446
   - Linked to Lifetime Trips:  3446
   - Linked to Offers:          254
⏳ Saving updated 'activity_earnings' to pienza.db...
SUCCESS: 'activity_earnings' fully reconciled.


,activity_earnings_id,lifetime_trips_fk,offer_id_fk
0,1,1,NaN
1,2,2,NaN
2,3,3,NaN
3,4,4,NaN
4,5,5,NaN



--- PROTOCOL COMPLETE ---


#### 5.6 — FK audit view

Builds a view auditing foreign-key integrity across the joined tables end to end.

In [96]:
print("--- FORGING BROCHE 1: THE FK MEGA JOIN (Data Lake B) ---")

if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/workspaces/pienza'
    db_path = os.path.join(project_root, 'data/big_bang/pienza.db')
else:
    db_path = db_file_path

db_engine = create_engine(f'sqlite:///{db_path}')

drop_sql = "DROP VIEW IF EXISTS v_broche_fks;"

create_sql = """
CREATE VIEW v_broche_fks AS
SELECT
    r.ocr_id                  AS raw_ocr_id,
    o.offer_id                AS hub_offer_id,
    o.session_fk              AS hub_session_fk,
    o.ocr_fk                  AS hub_ocr_fk,
    ef.feature_id             AS feat_id,
    ef.offer_id_fk            AS feat_offer_id_fk,
    te.event_id               AS event_id,
    te.offer_id_fk            AS event_offer_id_fk,
    lt.lifetime_trips_id      AS lt_id,
    lt.offer_id_fk            AS lt_offer_id_fk,
    ae.activity_earnings_id   AS ae_id,
    ae.offer_id_fk            AS ae_offer_id_fk,
    ae.lifetime_trips_fk      AS ae_lt_fk
FROM
    offers o
    LEFT JOIN raw_offers_ocr r    ON o.ocr_fk = r.ocr_id
    LEFT JOIN engineered_features ef  ON o.offer_id = ef.offer_id_fk
    LEFT JOIN trip_events te          ON o.offer_id = te.offer_id_fk
    LEFT JOIN lifetime_trips lt       ON o.offer_id = lt.offer_id_fk
    LEFT JOIN activity_earnings ae    ON lt.lifetime_trips_id = ae.lifetime_trips_fk;
"""

try:
    with db_engine.connect() as conn:
        conn.execute(text(drop_sql))
        conn.execute(text(create_sql))
        conn.commit()
        print(f"VIEW 'v_broche_fks' CREATED IN {os.path.basename(db_path)}.")

    print("\n--- AUDIT: INSPECTING THE DNA ---")
    audit_df = pd.read_sql("""
        SELECT * FROM v_broche_fks
        WHERE lt_id IS NOT NULL
        AND ae_id IS NOT NULL
        LIMIT 5
    """, db_engine)

    if not audit_df.empty:
        display(audit_df)
        print("AUDIT PASSED: Full data lineage visible from OCR to Earnings.")

    else:
        print("️ WARNING: No fully connected rows found. Check join integrity.")

except Exception as e:
    print(f"ERROR creating view: {e}")

--- FORGING BROCHE 1: THE FK MEGA JOIN (Data Lake B) ---
VIEW 'v_broche_fks' CREATED IN pienza.db.

--- AUDIT: INSPECTING THE DNA ---


,raw_ocr_id,hub_offer_id,hub_session_fk,hub_ocr_fk,feat_id,feat_offer_id_fk,event_id,event_offer_id_fk,lt_id,lt_offer_id_fk,ae_id,ae_offer_id_fk,ae_lt_fk
0,OCR04761,OF04761,SID0062,OCR04761,EF04761,OF04761,1028,OF04761,3446,OF04761,3446,OF04761,3446
1,OCR04761,OF04761,SID0062,OCR04761,EF04761,OF04761,1029,OF04761,3446,OF04761,3446,OF04761,3446
2,OCR04761,OF04761,SID0062,OCR04761,EF04761,OF04761,1030,OF04761,3446,OF04761,3446,OF04761,3446
3,OCR04761,OF04761,SID0062,OCR04761,EF04761,OF04761,1031,OF04761,3446,OF04761,3446,OF04761,3446
4,OCR04760,OF04760,SID0062,OCR04760,EF04760,OF04760,1024,OF04760,3445,OF04760,3445,OF04760,3445


AUDIT PASSED: Full data lineage visible from OCR to Earnings.


#### 5.7 — Human-readable master view

Builds `v_offers_human`, resolving every foreign key to its human-readable label.

In [97]:
print("--- FORGING BROCHE 2: THE HUMAN READABLE VIEW (Data Lake B) ---")

if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/workspaces/pienza'
    db_path = os.path.join(project_root, 'data/big_bang/pienza.db')
else:
    db_path = db_file_path

db_engine = create_engine(f'sqlite:///{db_path}')

drop_sql = "DROP VIEW IF EXISTS v_offers_human;"

create_sql = """
CREATE VIEW v_offers_human AS
SELECT
    -- 1. IDENTITY & TIME
    o.offer_id,
    o.session_fk,
    o.offer_timestamp,

    -- 2. CORE METRICS
    o.upfront_fare,
    o.time_to_pickup_sec,
    o.dist_to_pickup_km,
    o.est_trip_time_sec,
    o.est_trip_dist_km,

    -- 3. HUMAN READABLE LABELS
    pc.category_name                      AS str_product,
    oa.offer_action_description           AS str_action,
    rp.reason_primary_description         AS str_reason,
    ds.driver_state_at_request_description AS str_driver_state,
    pos.post_offer_status_description     AS str_post_status,
    out.outcome_description               AS str_outcome,
    iq.interpolation_quality_description  AS str_interp_quality,
    rs.record_status_description          AS str_record_status,

    -- 4. GEOSPATIAL CONTEXT
    o.pickup_address,
    o.dropoff_address,
    o.pickup_lat, o.pickup_lon,
    o.dropoff_lat, o.dropoff_lon,

    -- 5. FLAGS & NOTES
    o.is_surge, o.surge_amount,
    o.is_turbo_plus, o.turbo_plus_amount,
    o.is_reservation, o.reservation_amount,
    o.special_note_raw,

    -- 6. THE BRAIN (Engineered Features)
    ef.pickup_ambiguity,
    ef.dropoff_ambiguity,
    ef.traffic_index_base_120,
    ef.time_since_last_offer,
    ef.offer_density_60sec,
    ef.cycle_avg_dtp_km,
    ef.cycle_rolling_avg_spread,
    ef.total_accumulated_deadhead_sec,
    ef.cycle_cumulative_net_earnings,
    ef.eph_direct,
    ef.eph_operational,
    ef.is_operational_downgrade,
    ef.eph_realized_ML,
    ef.eph_complete_ML,
    ef.is_spread_downgrade_ML,
    ef.is_total_cycle_downgrade_ML,
    ef.home_vector_alignment_score,
    ef.day_of_week,
    ef.time_of_day_block,
    ef.day_type

FROM
    offers o
    LEFT JOIN product_category pc        ON o.product_category_fk = pc.product_category_id
    LEFT JOIN offer_action oa            ON o.offer_action_fk = oa.offer_action_id
    LEFT JOIN reason_primary rp          ON o.reason_primary_fk = rp.reason_primary_id
    LEFT JOIN driver_state_at_request ds ON o.driver_state_at_request_fk = ds.driver_state_at_request_id
    LEFT JOIN post_offer_status pos      ON o.post_offer_status_fk = pos.post_offer_status_id
    LEFT JOIN outcome out                ON o.outcome_fk = out.outcome_id
    LEFT JOIN interpolation_quality iq   ON o.interpolation_quality_fk = iq.interpolation_quality_id
    LEFT JOIN record_status rs           ON o.record_status_fk = rs.record_status_id
    LEFT JOIN engineered_features ef     ON o.offer_id = ef.offer_id_fk;
"""

try:
    with db_engine.connect() as conn:
        conn.execute(text(drop_sql))
        conn.execute(text(create_sql))
        conn.commit()
        print(f"VIEW 'v_offers_human' CREATED IN {os.path.basename(db_path)}.")

    print("\n--- AUDIT: READING THE ROSETTA STONE ---")
    audit_cols = "offer_id, str_action, eph_direct, is_operational_downgrade, home_vector_alignment_score"
    audit_df = pd.read_sql(f"SELECT {audit_cols} FROM v_offers_human LIMIT 3", db_engine)

    if not audit_df.empty:
        display(audit_df)
        print("AUDIT PASSED: The schema is aligned with the ERD.")

    else:
        print("️ WARNING: View returned no rows.")

except Exception as e:
    print(f"ERROR creating view: {e}")

--- FORGING BROCHE 2: THE HUMAN READABLE VIEW (Data Lake B) ---
VIEW 'v_offers_human' CREATED IN pienza.db.

--- AUDIT: READING THE ROSETTA STONE ---


,offer_id,str_action,eph_direct,is_operational_downgrade,home_vector_alignment_score
0,OF00001,reject,284.986047,0.0,None
1,OF00002,reject,260.790000,0.0,None
2,OF00003,accepted,NaN,NaN,None


AUDIT PASSED: The schema is aligned with the ERD.


#### 5.8 — Lifecycle audit view

Builds `v_lifecycle_audit`, tracing each offer from raw OCR through to bank earnings.

In [98]:
print("--- FORGING BROCHE 3: THE OMNISCIENT LIFECYCLE VIEW (Data Lake B) ---")

if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/workspaces/pienza'
    db_path = os.path.join(project_root, 'data/big_bang/pienza.db')
else:
    db_path = db_file_path

db_engine = create_engine(f'sqlite:///{db_path}')

drop_sql = "DROP VIEW IF EXISTS v_lifecycle_audit;"

create_sql = """
CREATE VIEW v_lifecycle_audit AS
WITH TripEventsPivot AS (
    SELECT
        offer_id_fk,
        trip_id_legacy,
        MAX(CASE WHEN event_types_id_fk = 2 THEN event_timestamp END) as t1_timestamp,
        MAX(CASE WHEN event_types_id_fk = 4 THEN event_timestamp END) as t3_timestamp,
        MAX(CASE WHEN event_types_id_fk = 5 THEN event_timestamp END) as t4_timestamp,
        MAX(upfront_fare) as te_upfront_fare,
        MAX(realized_fare) as te_realized_fare
    FROM trip_events
    GROUP BY offer_id_fk
),
HistoryStats AS (
    SELECT
        lt.offer_id_fk,
        lt.lifetime_trips_id,
        lt.original_fare,
        ae.net_earning,
        SUM(lt.original_fare) OVER (ORDER BY lt.request_timestamp ROWS UNBOUNDED PRECEDING) as cum_uber_earnings,
        SUM(ae.net_earning) OVER (ORDER BY lt.request_timestamp ROWS UNBOUNDED PRECEDING) as cum_net_earnings,
        AVG(ae.net_earning / NULLIF(lt.original_fare, 0)) OVER (ORDER BY lt.request_timestamp ROWS UNBOUNDED PRECEDING) as rolling_avg_net_take_rate
    FROM lifetime_trips lt
    LEFT JOIN activity_earnings ae ON lt.lifetime_trips_id = ae.lifetime_trips_fk
)
SELECT
    o.offer_id,
    o.session_fk,
    te.trip_id_legacy,
    r.time_taken                 AS ocr_raw_time,
    o.offer_timestamp            AS clean_timestamp,
    ef.day_of_week,
    ef.time_of_day_block,
    r.ride_type                  AS ocr_product,
    pc.category_name             AS internal_product,
    ae.product_category          AS bank_product,
    lt.global_product_name       AS official_product,
    r.pickup_address             AS ocr_pickup,
    o.pickup_address             AS clean_pickup,
    ef.pickup_ambiguity,
    r.dropoff_address            AS ocr_dropoff,
    o.dropoff_address            AS clean_dropoff,
    ef.dropoff_ambiguity,
    lt.original_fare             AS uber_original_fare,
    r.upfront_fare               AS ocr_upfront,
    o.upfront_fare               AS clean_upfront,
    te.te_upfront_fare           AS events_upfront,
    te.te_realized_fare          AS events_realized,
    ae.net_earning               AS bank_net_earning,
    te.t1_timestamp               AS gts_t1_accepted,
    lt.request_timestamp         AS uber_request,
    ROUND((julianday(te.t1_timestamp) - julianday(lt.request_timestamp)) * 86400) AS delta_accept_sec,
    te.t3_timestamp               AS gts_t3_started,
    lt.pickup_timestamp          AS uber_pickup,
    ROUND((julianday(te.t3_timestamp) - julianday(lt.pickup_timestamp)) * 86400) AS delta_start_sec,
    te.t4_timestamp               AS gts_t4_completed,
    lt.dropoff_timestamp         AS uber_dropoff,
    ROUND((julianday(te.t4_timestamp) - julianday(lt.dropoff_timestamp)) * 86400) AS delta_end_sec,
    hist.cum_uber_earnings,
    hist.cum_net_earnings,
    hist.rolling_avg_net_take_rate
FROM
    offers o
    LEFT JOIN raw_offers_ocr r        ON o.ocr_fk = r.ocr_id
    LEFT JOIN engineered_features ef  ON o.offer_id = ef.offer_id_fk
    LEFT JOIN product_category pc     ON o.product_category_fk = pc.product_category_id
    LEFT JOIN TripEventsPivot te      ON o.offer_id = te.offer_id_fk
    LEFT JOIN lifetime_trips lt       ON o.offer_id = lt.offer_id_fk
    LEFT JOIN activity_earnings ae    ON lt.lifetime_trips_id = ae.lifetime_trips_fk
    LEFT JOIN HistoryStats hist       ON o.offer_id = hist.offer_id_fk;
"""

try:
    with db_engine.connect() as conn:
        conn.execute(text(drop_sql))
        conn.execute(text(create_sql))
        conn.commit()
        print(f"VIEW 'v_lifecycle_audit' CREATED IN {os.path.basename(db_path)}.")

    print("\n--- AUDIT: THE OMNISCIENT VIEW ---")
    cols = "offer_id, clean_timestamp, uber_original_fare, bank_net_earning, delta_end_sec, cum_net_earnings, rolling_avg_net_take_rate"
    audit_df = pd.read_sql(f"SELECT {cols} FROM v_lifecycle_audit WHERE bank_net_earning IS NOT NULL ORDER BY clean_timestamp DESC LIMIT 5", db_engine)

    if not audit_df.empty:
        def _obf_fare(v):
            if v is None or pd.isna(v):
                return "-"
            int_part, dec_part = f"{float(v):.2f}".split('.')
            masked_int = int_part[:2] + '#' * max(0, len(int_part) - 2)
            return f"{masked_int}.{'#' * len(dec_part)}"

        df_display_audit = audit_df.copy()
        for col in ['uber_original_fare', 'bank_net_earning', 'cum_net_earnings']:
            df_display_audit[col] = df_display_audit[col].apply(_obf_fare)
        display(df_display_audit)
        print("AUDIT PASSED: History, Time, and Money are aligned.")

except Exception as e:
    print(f"ERROR creating view: {e}")

--- FORGING BROCHE 3: THE OMNISCIENT LIFECYCLE VIEW (Data Lake B) ---
VIEW 'v_lifecycle_audit' CREATED IN pienza.db.

--- AUDIT: THE OMNISCIENT VIEW ---


,offer_id,clean_timestamp,uber_original_fare,bank_net_earning,delta_end_sec,cum_net_earnings,rolling_avg_net_take_rate
0,OF04761,2025-10-01 09:06:38,15#.##,14#.##,11.0,32####.##,0.649466
1,OF04760,2025-10-01 08:35:48,20#.##,14#.##,60.0,32####.##,0.649380
2,OF04751,2025-10-01 07:56:38,16#.##,10#.##,34.0,32####.##,0.649353
3,OF04739,2025-10-01 07:34:29,17#.##,11#.##,22.0,32####.##,0.649363
4,OF04738,2025-10-01 06:57:35,27#.##,15#.##,8.0,32####.##,0.649359


AUDIT PASSED: History, Time, and Money are aligned.


#### 5.9 — Accepted-missions view

Builds `v_lifecycle_audit_accepted`, filtered to accepted offers only.

In [99]:
print("--- FORGING BROCHE 4: THE WINNER'S CIRCLE (Data Lake B) ---")

if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/workspaces/pienza'
    db_path = os.path.join(project_root, 'data/big_bang/pienza.db')
else:
    db_path = db_file_path

db_engine = create_engine(f'sqlite:///{db_path}')

drop_sql = "DROP VIEW IF EXISTS v_lifecycle_audit_accepted;"

create_sql = """
CREATE VIEW v_lifecycle_audit_accepted AS
SELECT *
FROM v_lifecycle_audit
WHERE trip_id_legacy IS NOT NULL 
ORDER BY clean_timestamp DESC;
"""

try:
    with db_engine.connect() as conn:
        conn.execute(text(drop_sql))
        conn.execute(text(create_sql))
        conn.commit()
        print(f"VIEW 'v_lifecycle_audit_accepted' CREATED IN {os.path.basename(db_path)}.")

    print("\n--- AUDIT: THE PURE GOLD ---")
    count_df = pd.read_sql("SELECT COUNT(*) as count FROM v_lifecycle_audit_accepted", db_engine)
    print(f"   - Total Accepted Missions in View: {count_df.iloc[0]['count']}")

    cols = "trip_id_legacy, clean_timestamp, uber_original_fare, bank_net_earning, cum_net_earnings"
    sample_df = pd.read_sql(f"SELECT {cols} FROM v_lifecycle_audit_accepted LIMIT 5", db_engine)

    if not sample_df.empty:
        def _obf_fare(v):
            if v is None or pd.isna(v):
                return v
            int_part, dec_part = f"{float(v):.2f}".split('.')
            masked_int = int_part[:2] + '#' * max(0, len(int_part) - 2)
            return f"{masked_int}.{'#' * len(dec_part)}"

        sample_df_display = sample_df.copy()
        for col in ('uber_original_fare', 'bank_net_earning', 'cum_net_earnings'):
            if col in sample_df_display.columns:
                sample_df_display[col] = sample_df_display[col].apply(_obf_fare)
        display(sample_df_display)
        print("AUDIT PASSED: Focused view created.")

except Exception as e:
    print(f"ERROR creating view: {e}")

--- FORGING BROCHE 4: THE WINNER'S CIRCLE (Data Lake B) ---
VIEW 'v_lifecycle_audit_accepted' CREATED IN pienza.db.

--- AUDIT: THE PURE GOLD ---


   - Total Accepted Missions in View: 259


,trip_id_legacy,clean_timestamp,uber_original_fare,bank_net_earning,cum_net_earnings
0,251001-07,2025-10-01 09:06:38,15#.##,14#.##,32####.##
1,251001-06,2025-10-01 08:35:48,20#.##,14#.##,32####.##
2,251001-05,2025-10-01 07:56:38,16#.##,10#.##,32####.##
3,251001-04,2025-10-01 07:34:29,17#.##,11#.##,32####.##
4,251001-03,2025-10-01 06:57:35,27#.##,15#.##,32####.##


AUDIT PASSED: Focused view created.


ohh lordy troubles so hard....

#### 5.10 — Off-by-one hotfix (TD-007)

Fixes a tracked indexing bug found after the four views above were built.

In [100]:
print("--- Initiating 'Off-by-One' Hotfix Patch (Data Lake B) ---")

try:
    if 'db_file_path' not in locals() and 'db_file_path' not in globals():
        project_root = '/workspaces/pienza'
        db_path = os.path.join(project_root, 'data/big_bang/pienza.db')
    else:
        db_path = db_file_path

    db_engine = create_engine(f'sqlite:///{db_path}')
    print(f"Database engine connection established to {os.path.basename(db_path)}.")
except Exception as e:
    print(f"CRITICAL ERROR: Could not create database engine. Details: {e}")
    raise

# The Architect's "Correction Edict"
CORRECTION_EDICT = {
    'OF02650': 'OF02649',
    'OF03780': 'OF03779',
    'OF04649': 'OF04648',
    'OF04733': 'OF04732'
}

TARGET_TRIP_ID_LEGACYS = ['250917-10', '250924-04', '250930-09', '251001-02']

print("\n⏳ Reading current state of 'trip_events' table...")
try:
    trip_events_df = pd.read_sql('SELECT * FROM trip_events', db_engine)
    print(f"Successfully read {len(trip_events_df)} records.")
except Exception as e:
    print(f"ERROR: Could not read from trip_events. Details: {e}")
    raise

print(f"\n⏳ Applying {len(CORRECTION_EDICT)} manual corrections in memory...")
for incorrect_offer, correct_offer in CORRECTION_EDICT.items():
    mask = (trip_events_df['offer_id_fk'] == incorrect_offer)
    if trip_events_df.loc[mask].shape[0] > 0:
        trip_events_df.loc[mask, 'offer_id_fk'] = correct_offer
        print(f"  - Corrected '{incorrect_offer}' to '{correct_offer}'.")

print("\n⏳ Re-running Golden Propagation for session consistency...")
try:
    df_offers_lookup = pd.read_sql("SELECT offer_id, session_fk FROM offers", db_engine)
    offers_session_map = pd.Series(df_offers_lookup.session_fk.values, index=df_offers_lookup.offer_id).to_dict()
    
    trip_events_df['offers_session_fk'] = trip_events_df['offer_id_fk'].map(offers_session_map)
    trip_events_df = trip_events_df.sort_values(by=['trip_id_legacy', 'event_timestamp'])
    trip_events_df['offer_id_fk'] = trip_events_df.groupby('trip_id_legacy')['offer_id_fk'].transform(lambda x: x.ffill().bfill())
    trip_events_df['offers_session_fk'] = trip_events_df.groupby('trip_id_legacy')['offers_session_fk'].transform(lambda x: x.ffill().bfill())
    print("Golden Propagation complete.")
except Exception as e:
    print(f"ERROR during propagation: {e}")
    raise

print("\n⏳ Writing patched data back to pienza.db...")
try:
    trip_events_df.to_sql('trip_events', db_engine, if_exists='replace', index=False)
    db_engine.dispose()
    time.sleep(2)
    print("Write operation complete.")
except Exception as e:
    print(f"ERROR: Failed to write to database. Details: {e}")
    raise

print("\n--- VERIFICATION (AFTER PATCH) ---")
try:
    verify_engine = create_engine(f'sqlite:///{db_path}')
    verify_df = pd.read_sql_query(f"SELECT trip_id_legacy, offer_id_fk, offers_session_fk FROM trip_events WHERE trip_id_legacy IN {tuple(TARGET_TRIP_ID_LEGACYS)}", verify_engine)
    display(verify_df.sort_values(by='trip_id_legacy'))
    verify_engine.dispose()
    print("VERIFICATION SUCCESS: All target trips correctly aligned.")
except Exception as e:
    print(f"VERIFICATION FAILED: {e}")

print("\n--- MANUAL OVERRIDE COMPLETE ---")

--- Initiating 'Off-by-One' Hotfix Patch (Data Lake B) ---
Database engine connection established to pienza.db.

⏳ Reading current state of 'trip_events' table...


Successfully read 1031 records.

⏳ Applying 4 manual corrections in memory...
  - Corrected 'OF02650' to 'OF02649'.
  - Corrected 'OF03780' to 'OF03779'.
  - Corrected 'OF04649' to 'OF04648'.
  - Corrected 'OF04733' to 'OF04732'.

⏳ Re-running Golden Propagation for session consistency...
Golden Propagation complete.

⏳ Writing patched data back to pienza.db...
Write operation complete.

--- VERIFICATION (AFTER PATCH) ---


,trip_id_legacy,offer_id_fk,offers_session_fk
0,250917-10,OF02649,SID0036
1,250917-10,OF02649,SID0036
2,250917-10,OF02649,SID0036
3,250917-10,OF02649,SID0036
4,250917-10,OF02649,SID0036
5,250924-04,OF03779,SID0049
6,250924-04,OF03779,SID0049
7,250924-04,OF03779,SID0049
8,250924-04,OF03779,SID0049
9,250930-09,OF04648,SID0061


VERIFICATION SUCCESS: All target trips correctly aligned.

--- MANUAL OVERRIDE COMPLETE ---


## Phase 6 — Silver palette and final ML views

Materializes the `silver_palette` table (geo + volatility features), replaces its foreign keys atomically for relational hygiene, populates it, then builds the two dynamic `v_ML_Supervised` views (with and without heuristic flags) that the rest of the project's modeling notebooks read from.

#### 6.1 — Materialize the silver_palette schema

Creates the `silver_palette` table (geo + volatility features).

In [101]:
print("--- Initiating Silver Palette Materialization (Data Lake B) ---")

if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/workspaces/pienza'
    db_path = os.path.join(project_root, 'data/big_bang/pienza.db')
else:
    db_path = db_file_path

db_engine = create_engine(f'sqlite:///{db_path}')

DDL_SQL_V3 = """
CREATE TABLE silver_palette (
    -- Clave Primaria / Anchor
    offer_id TEXT NOT NULL PRIMARY KEY,

    -- DROP-OFF CLUSTERING RESULTS
    dropoff_polygon_id INTEGER,
    dropoff_polygon_name TEXT,
    dropoff_h3_hex_id TEXT,
    dropoff_hdbscan_id INTEGER,
    dropoff_hdbscan_name TEXT,

    -- TRAFFIC INDEX FEATURES
    realized_traffic_index REAL,
    historical_rolling_avg_traffic_index REAL,
    traffic_volatility_index_ML REAL,
    traffic_volatility_index_EDA REAL,

    -- PICKUP CLUSTERING (FUTURE PLACEHOLDERS)
    pickup_polygon_id INTEGER,
    pickup_polygon_name TEXT,
    pickup_h3_hex_id TEXT,
    pickup_hdbscan_id INTEGER,
    pickup_hdbscan_name TEXT
);
"""

print(f"\n⏳ Creating 'silver_palette' in {os.path.basename(db_path)}...")

try:
    with db_engine.connect() as conn:
        conn.execute(text("DROP TABLE IF EXISTS silver_palette;"))
        conn.execute(text(DDL_SQL_V3))
        conn.commit()
    print("SUCCESS: Table 'silver_palette' created.")

    print("\n--- Verifying Schema Structure ---")
    with sqlite3.connect(db_path) as conn:
        schema_check = pd.read_sql("PRAGMA table_info(silver_palette);", conn)
        display(schema_check)

except Exception as e:
    print(f"CRITICAL ERROR: Could not create the Silver Palette table. Details: {e}")
    raise

print("\n--- PHASE 2 PREP: SILVER PALETTE READY ---")

--- Initiating Silver Palette Materialization (Data Lake B) ---

⏳ Creating 'silver_palette' in pienza.db...
SUCCESS: Table 'silver_palette' created.

--- Verifying Schema Structure ---


,cid,name,type,notnull,dflt_value,pk
0,0,offer_id,TEXT,1,None,1
1,1,dropoff_polygon_id,INTEGER,0,None,0
2,2,dropoff_polygon_name,TEXT,0,None,0
3,3,dropoff_h3_hex_id,TEXT,0,None,0
4,4,dropoff_hdbscan_id,INTEGER,0,None,0
5,5,dropoff_hdbscan_name,TEXT,0,None,0
6,6,realized_traffic_index,REAL,0,None,0
7,7,historical_rolling_avg_traffic_index,REAL,0,None,0
8,8,traffic_volatility_index_ML,REAL,0,None,0
9,9,traffic_volatility_index_EDA,REAL,0,None,0



--- PHASE 2 PREP: SILVER PALETTE READY ---


#### 6.2 — Atomic FK replacement

Replaces foreign keys on `silver_palette` atomically for relational hygiene.

In [102]:
print("--- Initiating Atomic Replacement Protocol (Data Lake B) ---")

if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/workspaces/pienza'
    db_path = os.path.join(project_root, 'data/big_bang/pienza.db')
else:
    db_path = db_file_path

db_engine = create_engine(f'sqlite:///{db_path}')

SQL_CLEAN_ZOMBIE = "DROP TABLE IF EXISTS silver_palette_new;"

SQL_CREATE_TEMP = """
CREATE TABLE silver_palette_new (
    offer_id TEXT NOT NULL PRIMARY KEY,
    dropoff_polygon_id INTEGER,
    dropoff_polygon_name TEXT,
    dropoff_h3_hex_id TEXT,
    dropoff_hdbscan_id INTEGER,
    dropoff_hdbscan_name TEXT,
    realized_traffic_index REAL,
    historical_rolling_avg_traffic_index REAL,
    traffic_volatility_index_ML REAL,
    traffic_volatility_index_EDA REAL,
    pickup_polygon_id INTEGER,
    pickup_polygon_name TEXT,
    pickup_h3_hex_id TEXT,
    pickup_hdbscan_id INTEGER,
    pickup_hdbscan_name TEXT,
    FOREIGN KEY (offer_id) REFERENCES offers(offer_id)
);
"""

SQL_COPY_DATA = "INSERT INTO silver_palette_new SELECT * FROM silver_palette;"
SQL_DROP_OLD = "DROP TABLE IF EXISTS silver_palette;"
SQL_RENAME_NEW = "ALTER TABLE silver_palette_new RENAME TO silver_palette;"

print(f"\n⏳ Rebuilding 'silver_palette' in {os.path.basename(db_path)}...")

try:
    with db_engine.connect() as conn:
        conn.execute(text(SQL_CLEAN_ZOMBIE))
        
        conn.execute(text(SQL_CREATE_TEMP))
        print("   - Temp table created with Foreign Key.")

        try:
            conn.execute(text(SQL_COPY_DATA))
            print("   - Data migrated from previous iteration.")
        except Exception as e:
            print(f"   - [Info] No existing data to migrate: {e}")

        conn.execute(text(SQL_DROP_OLD))
        conn.execute(text(SQL_RENAME_NEW))
        conn.commit()
        
    print("SUCCESS: Atomic replacement complete. FK is now established.")

    print("\n--- Verifying Schema Integrity ---")
    with db_engine.connect() as conn:
        schema_check = pd.read_sql("PRAGMA table_info(silver_palette);", conn)
        display(schema_check)

except Exception as e:
    print(f"CRITICAL ERROR: Atomic swap failed. Details: {e}")
    raise

--- Initiating Atomic Replacement Protocol (Data Lake B) ---

⏳ Rebuilding 'silver_palette' in pienza.db...
   - Temp table created with Foreign Key.
   - Data migrated from previous iteration.
SUCCESS: Atomic replacement complete. FK is now established.

--- Verifying Schema Integrity ---


,cid,name,type,notnull,dflt_value,pk
0,0,offer_id,TEXT,1,None,1
1,1,dropoff_polygon_id,INTEGER,0,None,0
2,2,dropoff_polygon_name,TEXT,0,None,0
3,3,dropoff_h3_hex_id,TEXT,0,None,0
4,4,dropoff_hdbscan_id,INTEGER,0,None,0
5,5,dropoff_hdbscan_name,TEXT,0,None,0
6,6,realized_traffic_index,REAL,0,None,0
7,7,historical_rolling_avg_traffic_index,REAL,0,None,0
8,8,traffic_volatility_index_ML,REAL,0,None,0
9,9,traffic_volatility_index_EDA,REAL,0,None,0


#### 6.3 — Populate silver_palette

Runs the ETL that populates the table created above.

In [103]:
try:
    if 'db_file_path' not in locals() and 'db_file_path' not in globals():
        project_root = '/workspaces/pienza'
        db_path = os.path.join(project_root, 'data/big_bang/pienza.db')
    else:
        db_path = db_file_path
    
    db_engine = create_engine(f'sqlite:///{db_path}')
    print("Environment check passed.")
except Exception as e:
    print(f"SETUP ERROR: {e}")
    assert False, "Setup failed."

# Canonical GCS Filenames
SRC_CLUSTERING_GCS = '260509_master_clustering_sheet1.csv'
SRC_TRAFFIC_GCS = '260509_engineered_features_silver_palette.csv'

TARGET_TABLE = "silver_palette"

print(f"\n--- PHASE 3: ETL FOR '{TARGET_TABLE}' (Data Lake B) ---")

try:
    # 1. EXTRACT: Clustering Data (CORRECTED & DEFENSIVE)
    print(f"1️⃣ Extracting Clustering Data from '{SRC_CLUSTERING_GCS}'...")
    blob_c = bucket.blob(SRC_CLUSTERING_GCS)
    df_cluster = pd.read_csv(io.BytesIO(blob_c.download_as_bytes()), dtype=str, na_filter=False)

    # Basic name normalization
    df_cluster.columns = df_cluster.columns.str.lower().str.strip().str.replace(' ', '_')

    clustering_allowed_cols = [
        'offer_id',
        'dropoff_polygon_id',
        'dropoff_polygon_name',
        'dropoff_h3_hex_id',
        'dropoff_hdbscan_id',
        'dropoff_hdbscan_name'
    ]

    cols_to_keep = [c for c in clustering_allowed_cols if c in df_cluster.columns]
    df_cluster = df_cluster[cols_to_keep]

    print(f"   -> Columns kept: {list(df_cluster.columns)}")
    print(f"   -> Loaded {len(df_cluster)} clustering records.")

    # 2. EXTRACT: Traffic Data
    print(f"2️⃣ Extracting Traffic Data from '{SRC_TRAFFIC_GCS}'...")
    blob_t = bucket.blob(SRC_TRAFFIC_GCS)
    df_traffic = pd.read_csv(io.BytesIO(blob_t.download_as_bytes()), dtype=str, na_filter=False)

    # Column normalization
    df_traffic.columns = df_traffic.columns.str.lower().str.strip().str.replace(' ', '_')

    # Defensive column selection
    traffic_cols = [
        'offer_id',
        'realized_traffic_index',
        'historical_rolling_avg_traffic_index',
        'traffic_volatility_index_ml',
        'traffic_volatility_index_eda'
    ]
    existing_traffic_cols = [c for c in traffic_cols if c in df_traffic.columns]
    df_traffic = df_traffic[existing_traffic_cols]

    print(f"   -> Loaded {len(df_traffic)} traffic records.")

    # 3. TRANSFORM: Merge Sources
    print("3️⃣ Merging sources on 'offer_id'...")

    df_merged = pd.merge(df_cluster, df_traffic, on='offer_id', how='outer')

    # Clean up empty strings to None (NULL)
    df_merged.replace('', None, inplace=True)

    print(f"   -> Merged Dataset: {len(df_merged)} rows.")

    # 4. LOAD: Populate Database
    print(f"4️⃣ Loading into SQLite table '{TARGET_TABLE}'...")

    with db_engine.connect() as conn:
        df_merged.to_sql(TARGET_TABLE, conn, if_exists='replace', index=False)

    print(f"SUCCESS: Phase 3 Complete. '{TARGET_TABLE}' is populated.")

except Exception as e:
    print(f"ETL FAILED: {e}")
    assert False, "Phase 3 Execution Halted."

print("\n--- DATA SAMPLE ---")
with db_engine.connect() as conn:
    sample = pd.read_sql(f"SELECT * FROM {TARGET_TABLE} LIMIT 5", conn)
    display(sample)

Environment check passed.

--- PHASE 3: ETL FOR 'silver_palette' (Data Lake B) ---
1️⃣ Extracting Clustering Data from '260509_master_clustering_sheet1.csv'...


   -> Columns kept: ['offer_id', 'dropoff_polygon_id', 'dropoff_polygon_name', 'dropoff_h3_hex_id', 'dropoff_hdbscan_id', 'dropoff_hdbscan_name']
   -> Loaded 4765 clustering records.
2️⃣ Extracting Traffic Data from '260509_engineered_features_silver_palette.csv'...
   -> Loaded 4765 traffic records.
3️⃣ Merging sources on 'offer_id'...
   -> Merged Dataset: 4765 rows.
4️⃣ Loading into SQLite table 'silver_palette'...
SUCCESS: Phase 3 Complete. 'silver_palette' is populated.

--- DATA SAMPLE ---


,offer_id,dropoff_polygon_id,dropoff_polygon_name,dropoff_h3_hex_id,dropoff_hdbscan_id,dropoff_hdbscan_name,realized_traffic_index,historical_rolling_avg_traffic_index,traffic_volatility_index_ml,traffic_volatility_index_eda
0,OF00001,-1,unassigned,89499516bafffff,-1,unassigned,None,None,None,None
1,OF00002,-1,unassigned,894995b1dd7ffff,-1,unassigned,None,None,None,None
2,OF00003,-1,unassigned,NaN,-2,missing_coordinates,None,None,None,None
3,OF00004,15,lomas_fc_cuernavaca,894995bac03ffff,41,el_semaforo_de_palmas,None,None,None,None
4,OF00005,-1,unassigned,894995bab23ffff,-1,unassigned,None,None,None,None


#### 6.4 — Consolidated view: v_ML_Supervised

Builds the primary view most downstream modeling notebooks read from.

In [104]:
print("\n--- PHASE 4: DYNAMIC VIEW CREATION 'v_ML_Supervised' (Data Lake B) ---")

if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/workspaces/pienza'
    db_path = os.path.join(project_root, 'data/big_bang/pienza.db')
else:
    db_path = db_file_path

db_engine = create_engine(f'sqlite:///{db_path}')

try:
    with db_engine.connect() as conn:
        cols_o = pd.read_sql("PRAGMA table_info(offers)", conn)['name'].tolist()
        cols_ef = pd.read_sql("PRAGMA table_info(engineered_features)", conn)['name'].tolist()
        cols_sp = pd.read_sql("PRAGMA table_info(silver_palette)", conn)['name'].tolist()

        select_parts = [f"o.{c}" for c in cols_o]
        
        ef_exclude = ['offer_id_fk']
        select_parts += [f"ef.{c}" for c in cols_ef if c not in ef_exclude]

        sp_exclude = ['offer_id']
        select_parts += [f"sp.{c}" for c in cols_sp if c not in sp_exclude]

        select_clause = ",\n    ".join(select_parts)

        VIEW_SQL_DYNAMIC = f"""
        CREATE VIEW v_ML_Supervised AS
        SELECT
            {select_clause}
        FROM
            offers o
        LEFT JOIN engineered_features ef ON o.offer_id = ef.offer_id_fk
        LEFT JOIN silver_palette sp ON o.offer_id = sp.offer_id;
        """

        conn.execute(text("DROP VIEW IF EXISTS v_ML_Supervised;"))
        conn.execute(text(VIEW_SQL_DYNAMIC))
        conn.commit()

    print(f"SUCCESS: Master View created in {os.path.basename(db_path)}.")

    print("\n--- Auditing Final Schema: Column List ---")
    with db_engine.connect() as conn:
        view_info = pd.read_sql("PRAGMA table_info(v_ML_Supervised)", conn)
        all_columns = view_info['name'].tolist()
        print(f"Total Columns: {len(all_columns)}")
        print("-" * 30)
        for i, col in enumerate(all_columns):
            print(f"{i}: {col}")

except Exception as e:
    print(f"ERROR: {e}")
    assert False, "Phase 4 Failed."


--- PHASE 4: DYNAMIC VIEW CREATION 'v_ML_Supervised' (Data Lake B) ---
SUCCESS: Master View created in pienza.db.

--- Auditing Final Schema: Column List ---
Total Columns: 103
------------------------------
0: offer_id
1: session_fk
2: ocr_fk
3: image_content_hash
4: offer_timestamp
5: upfront_fare
6: time_to_pickup_sec
7: dist_to_pickup_km
8: est_trip_time_sec
9: est_trip_dist_km
10: pickup_address
11: dropoff_address
12: pickup_lat
13: pickup_lon
14: dropoff_lat
15: dropoff_lon
16: is_surge
17: surge_amount
18: is_turbo_plus
19: turbo_plus_amount
20: is_reservation
21: reservation_amount
22: is_priority
23: priority_amount
24: is_exclusive
25: is_vip
26: is_identity_verified
27: is_long_trip
28: is_multiple_destinations
29: is_teens
30: rider_star_rating
31: rider_trip_count
32: time_in_session_sec
33: session_progress_ratio
34: inferred_agent_lat
35: inferred_agent_lon
36: inferred_agent_bearing
37: inferred_agent_speed_mps
38: is_imputed
39: special_note_raw
40: comment_1
41: com

#### 6.5 — Consolidated view with heuristic flags

Variant of the view above that also includes heuristic-flag labels.

In [105]:
print("\n--- PHASE 4 (RE-REVISADA): DYNAMIC VIEW WITH HEURISTIC FLAGS (Data Lake B) ---")

if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/workspaces/pienza'
    db_path = os.path.join(project_root, 'data/big_bang/pienza.db')
else:
    db_path = db_file_path

db_engine = create_engine(f'sqlite:///{db_path}')

try:
    with db_engine.connect() as conn:
        cols_o = pd.read_sql("PRAGMA table_info(offers)", conn)['name'].tolist()
        cols_ef = pd.read_sql("PRAGMA table_info(engineered_features)", conn)['name'].tolist()
        cols_sp = pd.read_sql("PRAGMA table_info(silver_palette)", conn)['name'].tolist()

        select_parts = [f"o.{c}" for c in cols_o]

        ef_exclude = ['offer_id_fk']
        select_parts += [f"ef.{c}" for c in cols_ef if c not in ef_exclude]

        sp_exclude = ['offer_id']
        select_parts += [f"sp.{c}" for c in cols_sp if c not in sp_exclude]

        # Pull the heuristic flag from the many-to-many table
        select_parts.append("hf.heuristic_flag_description AS heuristic_flag_context")

        select_clause = ",\n    ".join(select_parts)

        VIEW_SQL_DYNAMIC = f"""
        CREATE VIEW v_ML_Supervised AS
        SELECT
            {select_clause}
        FROM
            offers o
        LEFT JOIN engineered_features ef ON o.offer_id = ef.offer_id_fk
        LEFT JOIN silver_palette sp        ON o.offer_id = sp.offer_id
        LEFT JOIN heuristic_flag_offers hfo ON o.offer_id = hfo.offers_offer_id
        LEFT JOIN heuristic_flag hf          ON hfo.heuristic_flag_heuristic_flag_id = hf.heuristic_flag_id;
        """

        conn.execute(text("DROP VIEW IF EXISTS v_ML_Supervised;"))
        conn.execute(text(VIEW_SQL_DYNAMIC))
        conn.commit()

    print(f"SUCCESS: 'v_ML_Supervised' in {os.path.basename(db_path)} now includes heuristic labels.")

except Exception as e:
    print(f"ERROR: {e}")
    assert False, "Phase 4 Reconstruction Failed."


--- PHASE 4 (RE-REVISADA): DYNAMIC VIEW WITH HEURISTIC FLAGS (Data Lake B) ---
SUCCESS: 'v_ML_Supervised' in pienza.db now includes heuristic labels.


#### 6.6 — Closing checkpoint and flush

Final database checkpoint before closing the connection.

In [106]:
print("--- Initiating Final Shutdown Protocol (Data Lake B) ---")

print("Checkpointing SQLite WAL for 'pienza.db'...")
try:
    with db_engine.connect() as conn:
        # 'TRUNCATE' ensures the .db-wal file is merged and the .db file is primary
        conn.execute(text("PRAGMA wal_checkpoint(TRUNCATE);"))
        conn.execute(text("VACUUM;")) 
        conn.commit()
    print("SQLite Checkpoint & Vacuum Complete.")
except Exception as e:
    print(f"️ Warning during checkpoint: {e}")

db_engine.dispose()
print("DB Connection Closed.")

print("Forcing OS Flush to Storage...")
db_path_str = str(db_engine.url).replace('sqlite:///', '') 
try:
    with open(db_path_str, 'r+') as f:
        os.fsync(f.fileno())
    print("OS Buffer Flushed.")
except Exception as e:
    print(f"️ Could not fsync: {e}")

print("⏳ Waiting 10s for File Sync (GCS/Workspace)...")
time.sleep(10)
print("ETL PIPELINE FINALIZED SUCCESSFULLY.")

--- Initiating Final Shutdown Protocol (Data Lake B) ---
Checkpointing SQLite WAL for 'pienza.db'...
SQLite Checkpoint & Vacuum Complete.
DB Connection Closed.
Forcing OS Flush to Storage...
OS Buffer Flushed.
⏳ Waiting 10s for File Sync (GCS/Workspace)...
ETL PIPELINE FINALIZED SUCCESSFULLY.


---

## Appendix — table journey

Given this notebook's size, the appendix traces table-level lineage (not individual DataFrames) — the object that matters most here is which SQLite table or view gets built from which source, and in what order. Several tables are patched more than once later in the pipeline (e.g. `trip_events`, `lifetime_trips`, `activity_earnings`); those are listed again at the waypoint where the patch happens.

| Table / view | Built from | Waypoint |
|---|---|---|
| `raw_offers_ocr` | GCS: `raw_offers_raw_requests_messy.csv` | 1.3-1.4 |
| `offers` (+ lookup dimension tables) | GCS: `raw_offers_diamond_offers.csv`, linked to `raw_offers_ocr` | 1.5-1.6 |
| `v_reconciled_offer` | `offers` (+ lookup tables) | 1.7 |
| `event_types` | GCS: `gts-4_trip_events.csv` | 2.1-2.2 |
| `trip_events` | GCS: `gts-4_trip_events.csv`, `event_types`, `offers` | 2.3, 2.11-2.13 |
| `activity_earnings` | GCS: `platform_data_activity_earnings.csv` | 3.4 |
| `lifetime_trips` | GCS: `platform_data_lifetime_trips.csv` | 3.5 |
| `activity_earnings` (FK patch) | `activity_earnings`, `lifetime_trips` (direct ID join) | 3.6 |
| `trip_events` (manual overrides) | `trip_events` (in-place corrections) | 4.2-4.8 |
| `lifetime_trips` (bridge patch) | GCS: `consomaster_sheet1.csv`, `trip_events` | 5.1 |
| `v_trip_funnel_wide`, `v_trip_final_kpis`, `v_mission_dossier` | `trip_events` | 5.2 |
| `engineered_features` | GCS: `engineered_features_sheet2.csv`, `offers` | 5.3 |
| `offers` (OCR-link patch) | `offers`, `raw_offers_ocr` | 5.4 |
| `activity_earnings` (relink patch) | `activity_earnings`, `lifetime_trips` | 5.5 |
| `v_broche_fks` (FK audit view) | `offers`, `raw_offers_ocr`, `engineered_features`, `trip_events`, `lifetime_trips`, `activity_earnings` | 5.6 |
| `v_offers_human` | `offers`, `engineered_features` (+ lookup tables) | 5.7 |
| `v_lifecycle_audit` | `offers`, `raw_offers_ocr`, `engineered_features`, `trip_events`, `lifetime_trips`, `activity_earnings` | 5.8 |
| `v_lifecycle_audit_accepted` | `v_lifecycle_audit` | 5.9 |
| `trip_events` (off-by-one hotfix) | `trip_events` | 5.10 |
| `silver_palette` | GCS: `master_clustering_sheet1.csv`, GCS: `engineered_features_silver_palette.csv` | 6.1, 6.3 |
| `v_ML_Supervised` | `offers`, `engineered_features`, `silver_palette` | 6.4 |
| `v_ML_Supervised` (+ heuristic-flag variant) | `offers`, `engineered_features`, `silver_palette` | 6.5 |